In [1]:
def convert_to_grelex(eqs):
    """
    把多项式列表转换到 grelex (deglex) 排序下
    返回 (new_eqs, new_ring)
    """
    R_old = eqs[0].parent()
    F = R_old.base_ring()
    var_names = R_old.variable_names()

    # 新建 deglex 排序的多项式环
    R_new = PolynomialRing(F, var_names, order='deglex')

    # 构造变量替换映射
    subs_dict = {R_old.gen(i): R_new.gen(i) for i in range(R_old.ngens())}

    new_eqs = [R_new(f.subs(subs_dict)) for f in eqs]

    return new_eqs, R_new

# def monomials_by_degree(eqs):
#     """
#     返回:
#     { degree : set( monomials ) }
#     """
#     R = eqs[0].parent()
#     result = {}
#     for f in eqs:
#         for mon, coeff in f.dict().items():
#             # mon 是指数tuple
#             deg = sum(mon)

#             if deg not in result:
#                 result[deg] = set()

#             result[deg].add(mon)
#     return result

def monomials_by_degree(eqs):
    """
    返回:
    { degree : set( exponent_tuples ) }
    exponent_tuples 全部是 Python tuple
    """
    result = {}
    for f in eqs:
        for mon, coeff in f.dict().items():
            mon = tuple(mon)          # ✅ 强制统一类型
            deg = sum(mon)
            result.setdefault(deg, set()).add(mon)
    return result

from itertools import combinations_with_replacement

def all_monomials_of_degree(k, d):
    """
    返回所有 degree=d 的指数tuple
    """
    from sage.all import IntegerVectors
    return [tuple(v) for v in IntegerVectors(d, k)]

def analyze_monomial_support(eqs):
    """
    输出每个degree：
        出现了哪些单项式
        缺少哪些单项式
    """
    R = eqs[0].parent()
    k = R.ngens()

    support = monomials_by_degree(eqs)

    max_deg = max(support.keys())

    for d in sorted(support.keys()):
        print("\n========================")
        print(f"Degree {d}")
        print("========================")

        present = support[d]
        all_possible = set(all_monomials_of_degree(k, d))
        missing = all_possible - present

        print(f"Total possible: {len(all_possible)}")
        print(f"Present: {len(present)}")
        print(f"Missing: {len(missing)}")

        print("\nPresent monomials (exponent vectors):")
        for m in sorted(present):
            print(m)

        print("\nMissing monomials:")
        for m in sorted(missing):
            print(m)

def make_ring(p, nvars=4):
    S = PolynomialRing(GF(p), 't')
    # t = vector(S, S.gen())
    t = S.gen()

    R = PolynomialRing(S, nvars, 'x', order='degrevlex')
    X = vector(R, R.gens())

    return R, X, t

R, X, t = make_ring(1009, 4)
print(R)
print(X)
print(t)

from sage.combinat.integer_vector import IntegerVectors

def monomials_of_degree(R, X, d):
    mons = []
    n = len(X)

    for exps in IntegerVectors(d, n):
        m = R(1)
        for xi, ei in zip(X, exps):
            m *= xi**ei
        mons.append(m)

    return mons
print(monomials_of_degree(R, X, 2))

def monomials_leq_degree(R, X, D):
    mons = []

    for d in range(D+1):
        mons += monomials_of_degree(R, X, d)

    return mons
mons = monomials_leq_degree(R, X, 2)
print(mons)

Multivariate Polynomial Ring in x0, x1, x2, x3 over Univariate Polynomial Ring in t over Finite Field of size 1009
(x0, x1, x2, x3)
t
[x0^2, x0*x1, x0*x2, x0*x3, x1^2, x1*x2, x1*x3, x2^2, x2*x3, x3^2]
[1, x0, x1, x2, x3, x0^2, x0*x1, x0*x2, x0*x3, x1^2, x1*x2, x1*x3, x2^2, x2*x3, x3^2]


In [2]:
def span_equations_from_compressed_matrix(Mcrit):
    """
    对压缩后的矩阵 Mcrit：
    - 前 n-1 行是线性无关前缀基
    - 最后一行是目标行
    构造“最后一行属于前缀行张成空间”的方程组。
    """

    R = Mcrit.base_ring()
    K = R.base_ring()

    r = Mcrit.nrows()
    m = r - 1
    ncols = Mcrit.ncols()

    lam_names = [f"lam{i}" for i in range(m)]
    S = PolynomialRing(K, list(R.variable_names()) + lam_names, order='lex')

    # 嵌入到新环
    sub = {R(v): S(v) for v in R.variable_names()}

    prefix = []
    for i in range(m):
        prefix.append([S(Mcrit[i,j].subs(sub)) for j in range(ncols)])

    last = [S(Mcrit[m,j].subs(sub)) for j in range(ncols)]
    lam = [S(name) for name in lam_names]

    eqs_span = []
    for j in range(ncols):
        expr = last[j] - sum(lam[i] * prefix[i][j] for i in range(m))
        if expr != 0:
            eqs_span.append(expr)

    return S, eqs_span, lam


def random_search_span_system(S, eqs_span, fixed_subs, free_vars, trials=20000, p=101, seed=1234):
    set_random_seed(seed)

    sols = []
    free_vars_S = [S(str(v)) for v in free_vars]

    for _ in range(trials):
        subs = dict(fixed_subs)
        for v in free_vars_S:
            subs[v] = GF(p).random_element()

        ok = True
        for f in eqs_span:
            if f.subs(subs) != 0:
                ok = False
                break

        if ok:
            sols.append(subs)
            print("found solution =", subs)
            break

    return sols


def run_random_slice_univariate_support_only(M, keep_var='t0', trials=100, seed=1234):
    if seed is not None:
        set_random_seed(seed)

    R0 = M.base_ring()
    base = R0.base_ring()

    # 只收集 M 中真正出现的原变量
    used_var_set = set()
    for i in range(M.nrows()):
        for j in range(M.ncols()):
            f = M[i, j]
            if f != 0:
                used_var_set.update(f.variables())

    used_vars = sorted(used_var_set, key=str)

    print("M.nrows() =", M.nrows())
    print("M.ncols() =", M.ncols())
    print("used original variables =", used_vars)
    print("number of used original variables =", len(used_vars))

    # 新环：只包含真正出现的原变量 + 新 c 变量
    c_names = [f'c{i}' for i in range(M.nrows())]
    var_names = [str(v) for v in used_vars] + c_names
    R = PolynomialRing(base, var_names, order='degrevlex')

    # 构造从旧环到新环的替换映射
    sub_to_R = {v: R(str(v)) for v in used_vars}

    # 把 M 搬到新环，但只按出现变量替换
    M_new = matrix(
        R,
        M.nrows(),
        M.ncols(),
        [
            R(M[i, j].subs(sub_to_R)) if M[i, j] != 0 else R.zero()
            for i in range(M.nrows())
            for j in range(M.ncols())
        ]
    )

    vars_all = list(R.gens())
    c_vars = vars_all[len(used_vars):]

    keep = R(keep_var)
    if keep not in vars_all:
        raise ValueError(f"keep_var={keep_var} 不在实际使用变量里。可选变量: {[str(v) for v in vars_all]}")

    # 构造 c*M 的列方程
    eqs = []
    for j in range(M_new.ncols()):
        eq = sum(c_vars[i] * M_new[i, j] for i in range(M_new.nrows()))
        if eq != 0:
            eqs.append(eq)

    print("方程数量:", len(eqs))
    print("总变量数(实际使用变量 + c):", len(vars_all))
    print("变量列表:", vars_all)
    print("保留变量:", keep)
    print("=" * 50)

    def randval():
        if hasattr(base, "is_finite") and base.is_finite():
            L = list(base)
            return L[ZZ.random_element(len(L))]
        return ZZ.random_element(-5, 6)

    for tr in range(trials):
        # 固定 c0 = 1，排除平凡零解
        subs = {c_vars[0]: base(1)}

        for v in vars_all:
            if v != keep and v != c_vars[0]:
                subs[v] = randval()

        sliced = [f.subs(subs) for f in eqs]
        sliced = [f for f in sliced if f != 0]

        if len(sliced) == 0:
            print(f"[trial {tr}] 全部为 0 -> {keep} 自由")
            continue

        # 检查是否真成了单变量
        if any(any(w != keep for w in f.variables()) for f in sliced):
            print(f"[trial {tr}] 代入后不是单变量，跳过")
            continue

        S = PolynomialRing(base, str(keep))
        polys = [S(f) for f in sliced if S(f) != 0]

        if len(polys) == 0:
            print(f"[trial {tr}] 转单变量环后全 0 -> {keep} 自由")
            continue

        g = polys[0]
        for f in polys[1:]:
            g = gcd(g, f)

        print(f"[trial {tr}] gcd = {g}")
        if g.degree() <= 0:
            print("   -> 无解")
        else:
            try:
                roots = g.roots(multiplicities=False)
            except Exception:
                roots = "无法直接求"
            print("   -> 有解, roots =", roots)

def reduce_macaulay_columns_symbolic(M, verbose=True, return_data=False):
    """
    对多项式矩阵 M 的列做“符号线性去冗余”。

    方法：
      - 收集所有条目中出现过的单项式
      - 把每一列展开成一个普通系数向量
      - 在底层系数域上做列消元
      - 保留线性无关列

    输入：
      M : entries 在多项式环中的矩阵
      verbose : 是否打印信息
      return_data : 是否额外返回中间数据

    输出：
      M_reduced
      keep_cols
      （若 return_data=True，还返回 coeff_mat, monomials）
    """

    R = M.base_ring()
    K = R.base_ring()

    nrows = M.nrows()
    ncols = M.ncols()

    if verbose:
        print("========================================")
        print("原矩阵大小:", nrows, "x", ncols)
        print("base ring =", R)
        print("coefficient field/ring =", K)

    # 1. 收集所有出现过的单项式
    monomial_set = set()
    for i in range(nrows):
        for j in range(ncols):
            f = M[i, j]
            if f != 0:
                # f.monomials() 返回出现过的单项式
                monomial_set.update(f.monomials())

    monomials = sorted(monomial_set, key=str)
    nm = len(monomials)
    mon_index = {m: k for k, m in enumerate(monomials)}

    if verbose:
        print("出现过的不同单项式数:", nm)

    # 2. 构造普通系数矩阵：
    #    行索引 = (原矩阵行号, 单项式)
    #    列索引 = 原矩阵列号
    #
    #    shape = (nrows * nm) x ncols
    coeff_mat = Matrix(K, nrows * nm, ncols)

    for j in range(ncols):
        for i in range(nrows):
            f = M[i, j]
            if f == 0:
                continue
            d = f.dict()   # exponent tuple -> coefficient
            for exp, coeff in d.items():
                mono = R.monomial(*exp)
                row_idx = i * nm + mon_index[mono]
                coeff_mat[row_idx, j] = coeff

    if verbose:
        print("展开后的系数矩阵大小:", coeff_mat.nrows(), "x", coeff_mat.ncols())

    # 3. 找列主元（保留线性无关列）
    #
    # Sage 的 echelon_form/transformation 对列不总是最方便，
    # 所以转置后找行主元：coeff_mat 的列独立 <=> coeff_mat^T 的行独立
    E = coeff_mat.transpose().echelon_form()
    pivot_rows = E.pivots()

    # 但上面这个 pivot_rows 不是原列索引，需要改用右核来更稳？
    # 更直接的办法：对 coeff_mat 做列空间基提取
    #
    # Sage 里更稳妥：逐列增量加入，看 rank 是否上升

    keep_cols = []
    current = Matrix(K, coeff_mat.nrows(), 0)

    current_rank = 0
    for j in range(ncols):
        test = current.augment(coeff_mat.column(j))
        r = test.rank()
        if r > current_rank:
            keep_cols.append(j)
            current = test
            current_rank = r

    M_reduced = M.matrix_from_columns(keep_cols)

    if verbose:
        print("保留列数:", len(keep_cols))
        print("删除列数:", ncols - len(keep_cols))
        print("保留列索引:", keep_cols)
        print("缩减后矩阵大小:", M_reduced.nrows(), "x", M_reduced.ncols())
        print("========================================")

    if return_data:
        return M_reduced, keep_cols, coeff_mat, monomials
    else:
        return M_reduced, keep_cols

In [3]:
import random
from itertools import product

# =========================================================
# 基础工具
# =========================================================

def _normalize_point_dict_for_ring(pt, ring_or_gens):
    """
    统一成 { "t0": val0, "t1": val1, ... } 形式
    支持:
      - dict: {"t0":..., "t1":...}
      - list/tuple: [..]  (按变量顺序)
    """
    gens = ring_or_gens.gens() if hasattr(ring_or_gens, "gens") else tuple(ring_or_gens)

    if isinstance(pt, dict):
        out = {}
        for g in gens:
            name = str(g)
            if name in pt:
                out[name] = pt[name]
        return out

    if isinstance(pt, (list, tuple)):
        if len(pt) != len(gens):
            raise ValueError(f"需要 {len(gens)} 个参数值，但给了 {len(pt)} 个")
        return {str(gens[i]): pt[i] for i in range(len(gens))}

    raise TypeError("pt 必须是 dict / list / tuple")


def _point_dict_to_subs(M, pt):
    """
    把点 dict/list 变成适合 substitute 的 {t_i: val} 字典
    """
    S = M.base_ring()
    gens = list(S.gens())
    pt_dict = _normalize_point_dict_for_ring(pt, gens)
    F = S.base_ring()

    subs = {}
    for g in gens:
        name = str(g)
        if name not in pt_dict:
            raise ValueError(f"点里缺少变量 {name}")
        subs[g] = F(pt_dict[name])
    return subs


def specialize_matrix_over_base_field(M, pt):
    """
    把关于 t 的矩阵 M 代值为底域上的数值矩阵
    """
    S = M.base_ring()
    F = S.base_ring()
    subs = _point_dict_to_subs(M, pt)
    return matrix(F, M.substitute(subs))


def check_last_row_in_span_numeric(M, pt, verbose=False):
    """
    数值检查：最后一行是否在前面行张成空间里
    """
    M0 = specialize_matrix_over_base_field(M, pt)
    r_full = M0.rank()
    r_pref = M0[:-1].rank()
    ok = (r_full == r_pref)

    if verbose:
        print("point =", _normalize_point_dict_for_ring(pt, M.base_ring()))
        print("rank(full)   =", r_full)
        print("rank(prefix) =", r_pref)
        print("last row in span? ", ok)

    return {
        "matrix": M0,
        "rank_full": r_full,
        "rank_prefix": r_pref,
        "ok": ok,
    }


def _random_point_for_matrix(M, fixed_assignments=None, free_vars=None, value_pool=None):
    """
    生成一个随机点:
      - fixed_assignments: dict, 固定某些 t
      - free_vars: 只随机这些变量；若为 None 则随机所有未固定变量
      - value_pool: 可迭代对象；若 None 则从底域随机取
    """
    S = M.base_ring()
    F = S.base_ring()
    gens = list(S.gens())
    fixed_assignments = {} if fixed_assignments is None else dict(fixed_assignments)

    fixed_assignments = {str(k): F(v) for k, v in fixed_assignments.items()}

    if free_vars is None:
        free_names = [str(t) for t in gens if str(t) not in fixed_assignments]
    else:
        free_names = [str(t) for t in free_vars if str(t) not in fixed_assignments]

    pt = {}
    for g in gens:
        name = str(g)
        if name in fixed_assignments:
            pt[name] = F(fixed_assignments[name])
        elif name in free_names:
            if value_pool is None:
                pt[name] = F.random_element()
            else:
                vals = list(value_pool)
                pt[name] = F(random.choice(vals))
        else:
            # 不在 fixed，也不在 free_vars 里的，默认随机
            if value_pool is None:
                pt[name] = F.random_element()
            else:
                vals = list(value_pool)
                pt[name] = F(random.choice(vals))

    return pt


def _enumerate_points_for_matrix(M, fixed_assignments=None, free_vars=None, value_pool=None, max_points=None):
    """
    枚举点：
      - 如果 free_vars 给出，且 value_pool 可迭代，则枚举它们的笛卡尔积
      - 其余变量按 fixed_assignments 固定；没有指定的变量默认填 0
    用于低维切片小范围扫描
    """
    S = M.base_ring()
    F = S.base_ring()
    gens = list(S.gens())
    fixed_assignments = {} if fixed_assignments is None else dict(fixed_assignments)
    fixed_assignments = {str(k): F(v) for k, v in fixed_assignments.items()}

    if free_vars is None:
        raise ValueError("枚举模式下请显式给出 free_vars")

    if value_pool is None:
        raise ValueError("枚举模式下请给出 value_pool")

    free_names = [str(t) for t in free_vars]
    vals = [F(v) for v in value_pool]

    count = 0
    for tup in product(vals, repeat=len(free_names)):
        pt = {}
        for g in gens:
            name = str(g)
            if name in fixed_assignments:
                pt[name] = fixed_assignments[name]
            elif name in free_names:
                idx = free_names.index(name)
                pt[name] = tup[idx]
            else:
                pt[name] = F.zero()

        yield pt
        count += 1
        if max_points is not None and count >= max_points:
            return


# =========================================================
# 子式复杂度与选取
# =========================================================

def entry_t_degree(a):
    """
    a 是系数环 S=F[t...] 里的元素，返回 t-总次数
    零元记为 -1
    """
    if a == 0:
        return -1
    try:
        return a.total_degree()
    except Exception:
        try:
            mons = a.monomials()
            if len(mons) == 0:
                return 0
            return max(sum(m.degree(v) for v in a.parent().gens()) for m in mons)
        except Exception:
            return 0


def column_complexity_score(M, j, row_indices=None):
    """
    一列的复杂度打分：次数越低越好，非零越少越好
    """
    if row_indices is None:
        row_indices = range(M.nrows())

    deg_sum = 0
    nnz = 0
    for i in row_indices:
        a = M[i, j]
        if a != 0:
            nnz += 1
            d = entry_t_degree(a)
            deg_sum += max(d, 0)

    # 先看次数，再看非零数
    return (deg_sum, nnz, j)


def minor_complexity_score(M, col_set):
    """
    一个方阵子式的粗略复杂度：条目次数和 + 非零数
    """
    deg_sum = 0
    nnz = 0
    n = len(col_set)
    row_indices = list(range(n))
    for i in row_indices:
        for j in col_set:
            a = M[i, j]
            if a != 0:
                nnz += 1
                d = entry_t_degree(a)
                deg_sum += max(d, 0)
    return (deg_sum, nnz, tuple(col_set))


def find_prefix_pivot_columns_numeric(M, trials=30, value_pool=None, fixed_assignments=None, seed=None, verbose=True):
    """
    在随机特化点上找 prefix(前 nrows-1 行) 的一个 15 列 pivot chart
    返回长度为 nrows-1 的列集 J，使 prefix[:,J] 常常可逆
    """
    if seed is not None:
        random.seed(seed)

    nrows = M.nrows()
    ncols = M.ncols()
    pref_rows = nrows - 1

    if pref_rows <= 0:
        raise ValueError("矩阵行数至少要 >= 2")

    S = M.base_ring()
    F = S.base_ring()

    for trial in range(trials):
        pt = _random_point_for_matrix(
            M,
            fixed_assignments=fixed_assignments,
            free_vars=None,
            value_pool=value_pool
        )
        M0 = specialize_matrix_over_base_field(M, pt)
        P0 = M0[:pref_rows, :]

        r = P0.rank()
        if r < pref_rows:
            continue

        try:
            piv = list(P0.pivots())
        except Exception:
            E = P0.echelon_form()
            piv = list(E.pivots())

        if len(piv) >= pref_rows:
            J = piv[:pref_rows]
            if verbose:
                print(f"[find_prefix_pivot_columns_numeric] found at trial {trial}")
                print("pivot columns J =", J)
            return J

    raise RuntimeError("没有在随机点上找到满秩 prefix 的 pivot 列集；可增加 trials 或换 fixed_assignments")


def build_test_minors_from_pivot_chart(M, pivot_cols=None, num_test=6, candidate_pool=20, seed=None, verbose=True):
    """
    构造若干个 16x16 测试子式：
      - 行固定为全部 16 行
      - 列为 pivot_cols(15列) + 1个额外列
    这样比乱抽 16x16 更贴近“最后一行是否落入 prefix span”
    """
    if seed is not None:
        random.seed(seed)

    nrows = M.nrows()
    ncols = M.ncols()
    pref_rows = nrows - 1

    if pivot_cols is None:
        pivot_cols = find_prefix_pivot_columns_numeric(M, verbose=verbose)

    pivot_cols = list(pivot_cols)
    if len(pivot_cols) != pref_rows:
        raise ValueError(f"pivot_cols 长度应为 {pref_rows}，现在是 {len(pivot_cols)}")

    rest = [j for j in range(ncols) if j not in pivot_cols]

    # 优先挑复杂度较低的额外列
    scored = []
    for j in rest:
        # 主要关注全部16行上的复杂度；也可改成只看最后一行+少量行
        score = column_complexity_score(M, j, row_indices=range(nrows))
        scored.append(score)

    scored.sort()
    extras = [x[2] for x in scored[:candidate_pool]]

    # 再从候选里取前 num_test 个
    test_minors = []
    for k in extras[:num_test]:
        cols = list(pivot_cols) + [k]
        test_minors.append(cols)

    if verbose:
        print("[build_test_minors_from_pivot_chart]")
        print("pivot_cols =", pivot_cols)
        print("chosen extra columns =", [cols[-1] for cols in test_minors])
        print("num test minors =", len(test_minors))
        for idx, cols in enumerate(test_minors):
            print(f"  minor {idx}: extra={cols[-1]}, score={minor_complexity_score(M, cols)[:2]}")

    return {
        "pivot_cols": pivot_cols,
        "test_minors": test_minors,
    }


# =========================================================
# 子式 det 检查
# =========================================================

def minor_det_numeric(M, col_set, pt):
    """
    数值点上计算全部行、指定列的方阵 det
    """
    M0 = specialize_matrix_over_base_field(M, pt)
    A = M0.matrix_from_columns(col_set)
    return A.det()


def evaluate_test_minors_numeric(M, test_minors, pt, stop_at_first_nonzero=True):
    """
    在点 pt 上检查一组测试子式 det
    返回:
      {
        "all_zero": bool,
        "det_values": [...],
        "first_bad_index": int / None
      }
    """
    det_values = []
    first_bad = None

    for idx, cols in enumerate(test_minors):
        d = minor_det_numeric(M, cols, pt)
        det_values.append(d)
        if d != 0 and first_bad is None:
            first_bad = idx
            if stop_at_first_nonzero:
                return {
                    "all_zero": False,
                    "det_values": det_values,
                    "first_bad_index": first_bad,
                }

    return {
        "all_zero": (first_bad is None),
        "det_values": det_values,
        "first_bad_index": first_bad,
    }


# =========================================================
# 主搜索函数
# =========================================================

def search_points_by_minor_filter(
    M,
    test_minors=None,
    pivot_cols=None,
    num_test=6,
    candidate_pool=20,
    fixed_assignments=None,
    free_vars=None,
    value_pool=None,
    mode="random",          # "random" / "enumerate"
    max_trials=2000,
    stop_after_first_good=True,
    seed=None,
    verbose=True,
):
    """
    主函数：
      1) 若未给 test_minors，则自动构造
      2) 先检查测试子式 det 是否全 0
      3) 通过后再检查 rank(full)==rank(prefix)

    返回 summary 字典
    """
    if seed is not None:
        random.seed(seed)

    if test_minors is None:
        built = build_test_minors_from_pivot_chart(
            M,
            pivot_cols=pivot_cols,
            num_test=num_test,
            candidate_pool=candidate_pool,
            seed=seed,
            verbose=verbose
        )
        pivot_cols = built["pivot_cols"]
        test_minors = built["test_minors"]
    else:
        pivot_cols = pivot_cols

    good_points = []
    minor_pass_points = []

    stats = {
        "tested_points": 0,
        "minor_pass": 0,
        "rank_pass": 0,
        "first_good_point": None,
    }

    if mode == "enumerate":
        point_iter = _enumerate_points_for_matrix(
            M,
            fixed_assignments=fixed_assignments,
            free_vars=free_vars,
            value_pool=value_pool,
            max_points=max_trials
        )
    elif mode == "random":
        def _rand_iter():
            for _ in range(max_trials):
                yield _random_point_for_matrix(
                    M,
                    fixed_assignments=fixed_assignments,
                    free_vars=free_vars,
                    value_pool=value_pool
                )
        point_iter = _rand_iter()
    else:
        raise ValueError("mode 只能是 'random' 或 'enumerate'")

    for it, pt in enumerate(point_iter):
        stats["tested_points"] += 1

        minor_info = evaluate_test_minors_numeric(
            M,
            test_minors,
            pt,
            stop_at_first_nonzero=True
        )

        if not minor_info["all_zero"]:
            if verbose and (it + 1) % 100 == 0:
                print(f"[trial {it+1}] minor filter not passed")
            continue

        stats["minor_pass"] += 1
        minor_pass_points.append(dict(pt))

        rank_info = check_last_row_in_span_numeric(M, pt, verbose=False)

        if rank_info["ok"]:
            stats["rank_pass"] += 1
            good_points.append({
                "point": dict(pt),
                "rank_full": rank_info["rank_full"],
                "rank_prefix": rank_info["rank_prefix"],
                "minor_values": minor_info["det_values"],
            })

            if stats["first_good_point"] is None:
                stats["first_good_point"] = dict(pt)

            if verbose:
                print(f"[trial {it+1}] FOUND GOOD POINT")
                print("point =", pt)
                print("rank(full) =", rank_info["rank_full"])
                print("rank(prefix) =", rank_info["rank_prefix"])
                print("minor dets =", minor_info["det_values"])

            if stop_after_first_good:
                break
        else:
            if verbose:
                print(f"[trial {it+1}] minors all zero, but rank check failed")
                print("point =", pt)
                print("rank(full) =", rank_info["rank_full"])
                print("rank(prefix) =", rank_info["rank_prefix"])

    summary = {
        "pivot_cols": pivot_cols,
        "test_minors": test_minors,
        "stats": stats,
        "minor_pass_points": minor_pass_points,
        "good_points": good_points,
        "first_good_point": stats["first_good_point"],
    }

    if verbose:
        print("\n========== search summary ==========")
        print("tested_points =", stats["tested_points"])
        print("minor_pass    =", stats["minor_pass"])
        print("rank_pass     =", stats["rank_pass"])
        print("first_good_point =", stats["first_good_point"])

    return summary


import random
import time

def _normalize_fixed_assignments_for_matrix(M, fixed_assignments=None):
    """
    fixed_assignments 可写成:
      {"t0": 1, "t1": 2}
    或
      {T[0]: 1, T[1]: 2}
    统一转成 {str(var): field_element}
    """
    S = M.base_ring()
    F = S.base_ring()
    fixed_assignments = {} if fixed_assignments is None else dict(fixed_assignments)

    out = {}
    for k, v in fixed_assignments.items():
        out[str(k)] = F(v)
    return out


def random_point_for_matrix(M, fixed_assignments=None, free_vars=None, value_pool=None):
    """
    生成一个随机点:
      - fixed_assignments: 固定部分变量
      - free_vars: 若给出，则只把这些变量视为“要随机选的重点变量”；
                  其他未固定变量也会随机，只是写法上保留这个接口方便以后改
      - value_pool: 例如 range(50)，则从这些值里随机取
    """
    S = M.base_ring()
    F = S.base_ring()
    gens = list(S.gens())

    fixed = _normalize_fixed_assignments_for_matrix(M, fixed_assignments)

    if free_vars is None:
        free_names = set(str(g) for g in gens if str(g) not in fixed)
    else:
        free_names = set(str(x) for x in free_vars if str(x) not in fixed)

    pt = {}
    for g in gens:
        name = str(g)
        if name in fixed:
            pt[name] = fixed[name]
        else:
            if value_pool is None:
                pt[name] = F.random_element()
            else:
                vals = list(value_pool)
                pt[name] = F(random.choice(vals))
    return pt


def specialize_matrix_numeric(M, pt):
    """
    把符号矩阵 M 代入点 pt，得到底域上的数值矩阵
    """
    S = M.base_ring()
    F = S.base_ring()
    gens = list(S.gens())

    subs = {}
    for g in gens:
        name = str(g)
        if name not in pt:
            raise ValueError(f"点里缺少变量 {name}")
        subs[g] = F(pt[name])

    return matrix(F, M.substitute(subs))


def rank_gap_at_point(M, pt, prefix_rows=None, return_matrix=False):
    """
    默认检查“最后一行是否被前面所有行 span 到”
    即比较:
      prefix = M0[:-1, :]
      full   = M0

    如果想改成别的 prefix_rows，可以传入一个整数，表示取前 prefix_rows 行作 prefix
    """
    M0 = specialize_matrix_numeric(M, pt)

    if prefix_rows is None:
        prefix_rows = M0.nrows() - 1

    P0 = M0[:prefix_rows, :]
    r_pref = P0.rank()
    r_full = M0.rank()
    gap = r_full - r_pref

    out = {
        "rank_prefix": r_pref,
        "rank_full": r_full,
        "gap": gap,
        "ok": (gap == 0),
    }
    if return_matrix:
        out["matrix"] = M0
    return out


def random_search_rank_drop(
    M,
    max_trials=1000,
    fixed_assignments=None,
    free_vars=None,
    value_pool=None,
    prefix_rows=None,
    stop_after_first_good=True,
    seed=0,
    progress_every=100,
    verbose=True,
    keep_good_matrices=False,
):
    """
    直接随机搜点:
      - M: 例如 Ms[0]
      - max_trials: 随机次数
      - fixed_assignments: 固定某些 t
      - free_vars: 保留接口，不传即可
      - value_pool: 例如 range(50)；不传则在整个底域随机
      - prefix_rows: 默认 nrows-1，即检查“最后一行是否被前面 span 到”
      - stop_after_first_good: 找到一个 gap=0 就停
      - keep_good_matrices: 是否把命中的数值矩阵也存下来

    返回统计 summary
    """
    if seed is not None:
        random.seed(int(seed))

    stats = {
        "tested_points": 0,
        "gap_hist": {},          # 统计 gap 分布
        "best_gap": None,        # 最小 gap
        "first_good_point": None,
        "num_good_points": 0,
    }

    good_points = []

    t0 = time.time()

    for trial in range(1, max_trials + 1):
        pt = random_point_for_matrix(
            M,
            fixed_assignments=fixed_assignments,
            free_vars=free_vars,
            value_pool=value_pool
        )

        info = rank_gap_at_point(M, pt, prefix_rows=prefix_rows, return_matrix=keep_good_matrices)
        gap = info["gap"]

        stats["tested_points"] += 1
        stats["gap_hist"][gap] = stats["gap_hist"].get(gap, 0) + 1

        if stats["best_gap"] is None or gap < stats["best_gap"]:
            stats["best_gap"] = gap

        if verbose and progress_every is not None and trial % progress_every == 0:
            elapsed = time.time() - t0
            print(f"[trial {trial}] elapsed={elapsed:.2f}s, best_gap={stats['best_gap']}, gap_hist={stats['gap_hist']}")

        if info["ok"]:
            stats["num_good_points"] += 1
            if stats["first_good_point"] is None:
                stats["first_good_point"] = dict(pt)

            rec = {
                "point": dict(pt),
                "rank_prefix": info["rank_prefix"],
                "rank_full": info["rank_full"],
                "gap": info["gap"],
            }
            if keep_good_matrices:
                rec["matrix"] = info["matrix"]

            good_points.append(rec)

            if verbose:
                elapsed = time.time() - t0
                print("\nFOUND GOOD POINT")
                print("trial        =", trial)
                print("elapsed      =", f"{elapsed:.2f}s")
                print("rank_prefix  =", info["rank_prefix"])
                print("rank_full    =", info["rank_full"])
                print("gap          =", info["gap"])
                print("point        =", pt)

            if stop_after_first_good:
                break

    elapsed = time.time() - t0

    summary = {
        "stats": stats,
        "good_points": good_points,
        "elapsed": elapsed,
    }

    if verbose:
        print("\n========== random search summary ==========")
        print("tested_points     =", stats["tested_points"])
        print("gap_hist          =", stats["gap_hist"])
        print("best_gap          =", stats["best_gap"])
        print("num_good_points   =", stats["num_good_points"])
        print("first_good_point  =", stats["first_good_point"])
        print("elapsed           =", f"{elapsed:.2f}s")

    return summary

def random_univariate_slice_from_macaulay(
    M,
    keep_var=None,
    trials=50,
    value_pool=None,
    seed=None,
    verbose=True,
):
    """
    输入:
        M         : Macaulay矩阵 (entries 在某个多项式环里)
        keep_var  : 保留为唯一未知量的变量。可以传 ring 的 generator，
                    也可以传变量名字符串；若为 None，默认保留最后一个变量
        trials    : 随机切片次数
        value_pool: 若底层不是有限域，可指定一个随机值池，如 list(range(10))
        seed      : 随机种子
        verbose   : 是否打印过程

    返回:
        一个列表 results，每个元素对应一次 trial 的结果字典。
    """

    if seed is not None:
        set_random_seed(seed)

    # 原多项式环
    R0 = M.base_ring()
    base = R0.base_ring()
    orig_gens = list(R0.gens())
    nrows = M.nrows()
    ncols = M.ncols()

    # ---- 构造扩张多项式环：原变量 + c变量 ----
    c_names = [f'c{i}' for i in range(nrows)]
    all_names = [str(x) for x in orig_gens] + c_names
    R = PolynomialRing(base, all_names, order='degrevlex')

    # 迁移矩阵到新环
    M_R = M.change_ring(R)

    # 原变量与新环变量的对应
    new_orig = list(R.gens()[:len(orig_gens)])
    c_vars = list(R.gens()[len(orig_gens):])

    # 选保留变量
    if keep_var is None:
        keep = R.gens()[-1]
    elif isinstance(keep_var, str):
        keep = R(keep_var)
    else:
        keep_name = str(keep_var)
        keep = R(keep_name)

    all_vars = list(R.gens())
    if keep not in all_vars:
        raise ValueError(f"keep_var={keep} 不在变量列表中")

    # ---- 构造 c*M 的列方程 ----
    # eq_j = sum_i c_i * M[i,j]
    eqs = []
    for j in range(ncols):
        eq = sum(c_vars[i] * M_R[i, j] for i in range(nrows))
        if eq != 0:
            eqs.append(eq)

    if verbose:
        print("====================================")
        print("构造出的列方程数量:", len(eqs))
        print("总变量数:", len(all_vars))
        print("变量列表:", all_vars)
        print("保留变量:", keep)
        print("====================================")

    # ---- 随机赋值函数 ----
    def random_value():
        # 有限域：直接随机域元素
        if hasattr(base, "is_finite") and base.is_finite():
            elems = list(base)
            return elems[ZZ.random_element(0, len(elems))]
        # 其他情况：从 value_pool 里抽
        if value_pool is None:
            return ZZ.random_element(-5, 6)
        return value_pool[ZZ.random_element(0, len(value_pool))]

    # ---- 单变量系统求解 ----
    results = []

    for tr in range(trials):
        subs = {}
        for v in all_vars:
            if v != keep:
                subs[v] = random_value()

        sliced = [f.subs(subs) for f in eqs]
        sliced = [f for f in sliced if f != 0]

        # 检查是否真的都变成 keep 的单变量多项式
        bad = []
        for f in sliced:
            vars_f = f.variables()
            if any(v != keep for v in vars_f):
                bad.append(f)

        if bad:
            result = {
                'trial': tr,
                'status': 'not_univariate_after_subs',
                'subs': subs,
                'bad_equations': bad[:5],
            }
            results.append(result)
            if verbose:
                print(f"[trial {tr}] 代入后仍非单变量，跳过")
            continue

        # 全部恒零
        if len(sliced) == 0:
            result = {
                'trial': tr,
                'status': 'all_zero',
                'subs': subs,
                'has_root': True,   # 此时对 keep 没有限制
                'roots': 'free variable',
            }
            results.append(result)
            if verbose:
                print(f"[trial {tr}] 所有方程都变成 0，{keep} 自由")
            continue

        # 放到单变量环里做 gcd
        S = PolynomialRing(base, str(keep))
        u = S.gen()

        polys = []
        for f in sliced:
            fS = S(f)
            if fS != 0:
                polys.append(fS)

        if len(polys) == 0:
            result = {
                'trial': tr,
                'status': 'all_zero_after_cast',
                'subs': subs,
                'has_root': True,
                'roots': 'free variable',
            }
            results.append(result)
            if verbose:
                print(f"[trial {tr}] 转到单变量环后全 0，{keep} 自由")
            continue

        g = polys[0]
        for f in polys[1:]:
            g = gcd(g, f)

        if verbose:
            print(f"\n[trial {tr}]")
            print("固定变量赋值:")
            print(subs)
            print("剩余单变量多项式个数:", len(polys))
            print("gcd =", g)

        # gcd=1 表示公共根为空
        if g.degree() <= 0:
            result = {
                'trial': tr,
                'status': 'no_common_root',
                'subs': subs,
                'has_root': False,
                'gcd': g,
                'polys': polys[:10],
            }
            results.append(result)
            if verbose:
                print("=> 无公共根")
            continue

        # 找根
        roots = []
        try:
            # 有限域/可分解情形
            fac = g.factor()
            for h, e in fac:
                if h.degree() == 1:
                    # h = a*u + b
                    a = h[u.degree()] if False else None  # 占位，不使用
            roots = g.roots(multiplicities=False)
        except Exception:
            fac = None
            try:
                roots = g.roots()
            except Exception:
                roots = "unable to compute directly"

        result = {
            'trial': tr,
            'status': 'common_root_exists',
            'subs': subs,
            'has_root': True,
            'gcd': g,
            'factorization': fac,
            'roots': roots,
            'polys': polys[:10],
        }
        results.append(result)

        if verbose:
            print("=> 可能有公共根")
            print("roots =", roots)

    return results

def solve_ct_system_gb(
    M,
    nonzero_vars=('t1', 't3', 't5'),
    normalize_c_index=0,
    fixed_vars=None,
    max_slice_trials=50,
    seed=1234,
    verbose=True,
):
    """
    直接构造 c,t 联合系统并求 Gröbner basis。

    输入:
        M                : Macaulay矩阵，例如 Ms[0]
        nonzero_vars     : 必须非零的参数名列表/元组，如 ('t1','t3','t5')
        normalize_c_index: 固定哪个 c_i = 1 来排除平凡解
        fixed_vars       : 预先固定的变量，字典，如 {'t1': 1} 或 {T[1]: 1}
        max_slice_trials : 若理想非0维，随机切片找点的最大次数
        seed             : 随机种子
        verbose          : 是否打印过程

    返回:
        dict，包含：
            ring
            equations
            ideal
            groebner_basis
            dimension
            solution / solutions / sliced_solution
    """

    if fixed_vars is None:
        fixed_vars = {}

    if seed is not None:
        set_random_seed(seed)

    R0 = M.base_ring()
    K = R0.base_ring()

    # ---- 只收集 M 中实际出现的原变量 ----
    used_var_set = set()
    for i in range(M.nrows()):
        for j in range(M.ncols()):
            f = M[i, j]
            if f != 0:
                used_var_set.update(f.variables())

    used_vars = sorted(used_var_set, key=str)
    used_names = [str(v) for v in used_vars]

    if verbose:
        print("========================================")
        print("M size =", M.nrows(), "x", M.ncols())
        print("used variables =", used_names)
        print("number of used variables =", len(used_names))

    # ---- 新变量：c0,...,c_{m-1} 和辅助变量 z ----
    m = M.nrows()
    c_names = [f'c{i}' for i in range(m)]
    z_name = 'z_nonzero_aux'

    all_names = used_names + c_names + [z_name]

    # 用 lex 更适合后面消元/解
    R = PolynomialRing(K, all_names, order='lex')

    # 旧变量映到新环
    sub_to_R = {v: R(str(v)) for v in used_vars}

    # 迁移矩阵到新环
    M_new = matrix(
        R,
        M.nrows(),
        M.ncols(),
        [
            R(M[i, j].subs(sub_to_R)) if M[i, j] != 0 else R.zero()
            for i in range(M.nrows())
            for j in range(M.ncols())
        ]
    )

    orig_vars_R = [R(name) for name in used_names]
    c_vars = [R(name) for name in c_names]
    z = R(z_name)

    # ---- 处理 fixed_vars：统一转成新环里的变量 ----
    fixed_subs = {}
    for k, v in fixed_vars.items():
        k_str = str(k)
        if k_str not in all_names:
            raise ValueError(f"fixed_vars 中的 {k} 不在系统变量里。可选变量: {all_names}")
        fixed_subs[R(k_str)] = R(v)

    # 已固定变量名
    fixed_names = {str(k) for k in fixed_vars.keys()}

    # ---- 构造列方程：c * M = 0 ----
    eqs = []
    for j in range(M_new.ncols()):
        eq = sum(c_vars[i] * M_new[i, j] for i in range(M_new.nrows()))
        if eq != 0:
            eqs.append(eq)

    # ---- 先代入 fixed_vars ----
    if fixed_subs:
        eqs = [eq.subs(fixed_subs) for eq in eqs]

    # ---- 加归一化条件：c_k = 1 ----
    if normalize_c_index < 0 or normalize_c_index >= len(c_vars):
        raise ValueError("normalize_c_index 超出范围")
    eqs.append(c_vars[normalize_c_index] - 1)

    # ---- fixed 后，nonzero 中已固定的变量不用再乘进去 ----
    effective_nonzero_vars = tuple(name for name in nonzero_vars if str(name) not in fixed_names)

    # ---- 加非零条件：z * prod(nonzero_vars) - 1 = 0 ----
    nz_prod = R.one()
    for name in effective_nonzero_vars:
        if str(name) not in all_names:
            raise ValueError(f"nonzero_vars 中的 {name} 不在系统变量里。可选变量: {all_names}")
        nz_prod *= R(str(name))
    eqs.append(z * nz_prod - 1)

    # fixed_subs 也要作用到后加的方程（虽然通常不会影响 c_k=1 和 z*prod-1）
    if fixed_subs:
        eqs = [eq.subs(fixed_subs) for eq in eqs]

    # 去掉恒等于 0 的方程
    eqs = [eq for eq in eqs if eq != 0]

    if verbose:
        print("number of equations =", len(eqs))
        print("nonzero constraints on =", list(effective_nonzero_vars))
        print("normalization =", c_vars[normalize_c_index], "= 1")
        print("fixed vars =", {str(k): v for k, v in fixed_vars.items()})
        print("total variables in ring =", len(all_names))
        print("ring =", R)
        print("========================================")

    I = Ideal(eqs)

    # ---- 先尝试看维数 ----
    try:
        dimI = I.dimension()
    except Exception as e:
        dimI = None
        if verbose:
            print("dimension() failed:", e)

    if verbose:
        print("Ideal dimension =", dimI)
        print("Computing Gröbner basis ...")

    # ---- 直接算 GB ----
    try:
        G = I.groebner_basis()
        if verbose:
            print("GB computed, size =", len(G))
    except Exception as e:
        if verbose:
            print("groebner_basis() failed:", e)
        return {
            'ring': R,
            'equations': eqs,
            'ideal': I,
            'groebner_basis': None,
            'dimension': dimI,
            'error': e,
        }

    result = {
        'ring': R,
        'equations': eqs,
        'ideal': I,
        'groebner_basis': G,
        'dimension': dimI,
    }

    # =========================================================
    # 0维：直接尝试找根
    # =========================================================
    if dimI == 0:
        if verbose:
            print("Ideal appears 0-dimensional. Trying to solve ...")

        try:
            sols = I.variety()
            if verbose:
                print("number of solutions =", len(sols))

            clean_sols = []
            for sol in sols:
                clean = {str(k): v for k, v in sol.items() if str(k) != z_name}

                # 把 fixed_vars 也补回输出，便于看完整解
                for k, v in fixed_vars.items():
                    clean[str(k)] = v

                clean_sols.append(clean)

            result['solutions'] = clean_sols

            if len(clean_sols) > 0:
                result['solution'] = clean_sols[0]
                if verbose:
                    print("Found a solution:")
                    print(clean_sols[0])
            return result

        except Exception as e:
            if verbose:
                print("variety() failed:", e)
                print("Will return GB only.")
            result['solve_error'] = e
            return result

    # =========================================================
    # 非0维：随机切片找一个点
    # =========================================================
    if verbose:
        print("Ideal is not 0-dimensional (or dimension unknown).")
        print("Trying random affine slices to get one point ...")

    # 已固定变量不要再参与切片
    orig_vars_free = [v for v in orig_vars_R if str(v) not in fixed_names]
    vars_for_slice = [v for v in orig_vars_free + c_vars if v != c_vars[normalize_c_index]]
    # z 不切，留给非零约束自己处理

    def randval():
        if hasattr(K, "is_finite") and K.is_finite():
            L = list(K)
            return L[ZZ.random_element(len(L))]
        return ZZ.random_element(-5, 6)

    for tr in range(max_slice_trials):
        subs = {}
        free_keep = []

        # 默认留 1 个未固定原变量
        if len(orig_vars_free) > 0:
            keep_var = orig_vars_free[ZZ.random_element(len(orig_vars_free))]
            free_keep = [keep_var]
        else:
            free_keep = []

        for v in vars_for_slice:
            if v not in free_keep:
                subs[v] = randval()

        sliced_eqs = [f.subs(subs) for f in eqs]
        sliced_eqs = [f for f in sliced_eqs if f != 0]
        J = Ideal(sliced_eqs)

        try:
            dJ = J.dimension()
        except Exception:
            dJ = None

        if verbose:
            print(f"[slice {tr}] keep =", [str(v) for v in free_keep], "dimension =", dJ)

        try:
            if dJ == 0:
                sols = J.variety()
                if len(sols) > 0:
                    sol0 = sols[0]
                    full_sol = {}

                    for name in all_names:
                        vv = R(name)
                        if vv in sol0:
                            full_sol[name] = sol0[vv]
                        elif vv in subs:
                            full_sol[name] = subs[vv]

                    # 补回 fixed_vars
                    for k, v in fixed_vars.items():
                        full_sol[str(k)] = v

                    if z_name in full_sol:
                        del full_sol[z_name]

                    result['sliced_solution'] = full_sol
                    if verbose:
                        print("Found sliced solution:")
                        print(full_sol)
                    return result
        except Exception as e:
            if verbose:
                print(f"[slice {tr}] solve failed:", e)

    if verbose:
        print("No sliced solution found.")
    return result

In [4]:
# from sage.all import *
# import random

# class PoseidonSage:
#     def __init__(self, t=5, rate=4, capacity=1, s=1, p=101, alpha=3, n_rounds=30, seed=None):
#         """
#         Poseidon toy model in Sage
#         """
#         assert rate + capacity == t, "rate + capacity must equal t"

#         self.t = t
#         self.rate = rate
#         self.capacity = capacity
#         self.s = s
#         self.p = p
#         self.alpha = alpha
#         self.n_rounds = n_rounds
#         self.seed = seed
#         self.F = GF(p)

#         if seed is not None:
#             random.seed(seed)

#         # round constants and MDS matrix
#         self.round_constants = self._generate_round_constants()
#         self.mds_matrix = self._generate_cauchy_mds()

#     # ------------------ Round constants ------------------
#     def _generate_round_constants(self):
#         RC = []
#         for _ in range(self.n_rounds):
#             RC.append(vector(self.F, [self.F(random.randint(0, self.p-1)) for _ in range(self.t)]))
#         return RC

#     # ------------------ Cauchy MDS matrix ------------------
#     def _generate_cauchy_mds(self):
#         elems = random.sample(range(self.p), 2*self.t)
#         x = elems[:self.t]
#         y = elems[self.t:]
#         M = matrix(self.F, self.t, self.t)
#         for i in range(self.t):
#             for j in range(self.t):
#                 diff = x[i] - y[j]
#                 assert diff != 0, "Zero denominator in Cauchy construction"
#                 M[i,j] = self.F(1)/self.F(diff)
#         return M

#     # ------------------ S-box ------------------
#     def sbox(self, x):
#         return x ** self.alpha

#     def external_sbox_layer(self, state):
#         return vector(self.F, [self.sbox(x) for x in state])

#     def internal_sbox_layer(self, state):
#         state = vector(state)
#         for i in range(self.s):
#             state[i] = self.sbox(state[i])
#         return state

#     def inv_sbox(self, x):
#         alpha_inv = inverse_mod(self.alpha, self.p - 1)
#         return x ** alpha_inv

#     def inv_internal_sbox_layer(self, state):
#         state = vector(state)
#         for i in range(self.s):
#             state[i] = self.inv_sbox(state[i])
#         return state
    
#     def inv_mds_layer(self, state):
#         return self.mds_matrix.inverse() * state

#     def sub_round_constants(self, state, round_idx):
#         return state - self.round_constants[round_idx]
#     # def sub_round_constants(self, state, round_idx):
#     #     R = state[0].parent()
#     #     rc = vector(R, [R(c) for c in self.round_constants[round_idx]])
#     #     return state - rc

#     def inv_internal_round(self, state, round_idx):
#         state = self.inv_mds_layer(state)
#         state = self.inv_internal_sbox_layer(state)
#         state = self.sub_round_constants(state, round_idx)
#         return state
#     # ------------------ Linear layer ------------------
#     def mds_layer(self, state):
#         return self.mds_matrix * state

#     # ------------------ Round functions ------------------
#     def add_round_constants(self, state, round_idx):
#         return state + self.round_constants[round_idx]

#     def external_round(self, state, round_idx):
#         state = self.add_round_constants(state, round_idx)
#         state = self.external_sbox_layer(state)
#         state = self.mds_layer(state)
#         return state

#     def internal_round(self, state, round_idx):
#         state = self.add_round_constants(state, round_idx)
#         state = self.internal_sbox_layer(state)
#         state = self.mds_layer(state)
#         return state

#     # ------------------ Kernel / Subspace search ------------------
#     def kernel_mod_p(self, M):
#         """
#         Compute kernel of M over GF(p)
#         Return as a matrix whose columns are basis vectors
#         """
#         return M.right_kernel().matrix().transpose()

#     def compute_mds_invariant_subspaces_sage(self):
#         p = self.p
#         s = self.s
#         Fp = GF(p)

#         # MDS matrix mod p
#         M = Matrix(ZZ, self.mds_matrix).apply_map(lambda x: x % p).change_ring(Fp)

#         B = M[:s, s:]
#         D = M[s:, s:]

#         D_inv = D.inverse()

#         # 初始不变子空间 W：n × k 矩阵（列空间）
#         W = self.kernel_mod_p(B).change_ring(Fp)
#         W_list = [W]

#         while W.ncols() > 0:
#             # W2 = D^{-1} W
#             W2 = D_inv * W

#             # Z = [ W | -W2 ]
#             Z = W.augment(-W2)

#             # 求核
#             ns = self.kernel_mod_p(Z).change_ring(Fp)
#             if ns.ncols() == 0:
#                 break

#             # W 的列向量空间
#             n = W.nrows()
#             k_old = W.ncols()

#             # 把 ns 的前 k_old 行当作系数
#             new_vectors = []
#             for i in range(ns.ncols()):
#                 coeffs = vector(Fp, [ns[j, i] for j in range(W.ncols())])
#                 v = W * coeffs          # v 是 n×1 矩阵
#                 new_vectors.append(v)  # 转成 vector

#             if not new_vectors:
#                 break

#             # 重新组装成 n × k_new 的矩阵
#             W = matrix(Fp, new_vectors).transpose()
#             W_list.append(W)

#         return W_list

#     def show_invariant_subspace_dynamics(self, W_list):
#         p = self.p
#         s = self.s
#         M = Matrix(ZZ, self.mds_matrix).apply_map(lambda x: x % p)
#         n = M.nrows()

#         for idx, W in enumerate(W_list, start=1):
#             print(f"\n{'='*16} W_{idx} {'='*16}")
#             print(f"Dimension: {W.ncols()}")
#             print("Basis (columns):")
#             print(W)

#             # padding s行零在上面
#             Z = zero_matrix(ZZ, s, W.ncols())
#             V = Z.stack(W).apply_map(lambda x: x % p)

#             print("\nInitial vectors after padding zeros:")
#             print(V)

#             # 连续左乘 M，次数等于 idx
#             cur = V
#             for step in range(1, idx + 1):
#                 cur = (M * cur).apply_map(lambda x: x % p)
#                 print(f"\nAfter multiplying M^{step}:")
#                 print(cur)


#         # ------------------ CICO equations over GF(p) ------------------
    
#     def make_symbolic_X(self, k):
#         """
#         返回 (R, X)
#         R: PolynomialRing
#         X: k维多项式向量
#         """
#         Fp = self.F
#         R = PolynomialRing(Fp, [f"x{i}" for i in range(k)])
#         X = vector(R, R.gens())
#         return R, X
    
#     def make_symbolic_X_with_t(self, k, num_t):
#         Fp = self.F

#         x_vars = [f"x{i}" for i in range(k)]
#         t_vars = [f"t{i}" for i in range(num_t)]

#         R = PolynomialRing(Fp, x_vars + t_vars)

#         gens = R.gens()

#         X = vector(R, gens[:k])
#         T = vector(R, gens[k:k+num_t])

#         return R, X, T

#     def initial_state_from_W(self, W, X):
#         """
#         state = [0,...,0 || W * X]
#         返回多项式向量（长度 t）
#         """
#         R = X[0].parent()
#         s = self.s

#         lin_part = list(W.change_ring(R) * X)   # 这是 list，不是 vector
#         state_list = [R.zero()] * s + lin_part

#         return vector(R, state_list)
    
#     def rp_round_polynomial(self, state, round_index):
#         """
#         一轮 internal round（r_p）
#         state: Polynomial vector
#         """
#         R = state[0].parent()
#         Fp = self.F
#         s  = self.s

#         # 加轮常量
#         rc = vector(R, [R(c) for c in self.round_constants[round_index]])
#         state = state + rc

#         # internal S-box
#         state = vector(R, state)
#         for i in range(s):
#             state[i] = state[i] ** self.alpha

#         # MDS
#         M = self.mds_matrix.change_ring(R)
#         state = M * state

#         return state
#     def rf_round_polynomial(self, state, round_index):
#         """
#         一轮 external round（r_f）
#         """
#         R = state[0].parent()

#         # 加轮常量
#         rc = vector(R, [R(c) for c in self.round_constants[round_index]])
#         state = state + rc

#         # external S-box
#         state = vector(R, [x ** self.alpha for x in state])

#         # MDS
#         M = self.mds_matrix.change_ring(R)
#         state = M * state

#         return state
#     # def generate_cico_equations_modp(self, W, constants, rounds, round_type="internal"):
#     def generate_cico_equations_modp(self, W, constants, rounds, exter1, inter, exter2):
#         """
#         round_type: "internal" or "external"
#         """
#         k = W.ncols()
#         R, X = self.make_symbolic_X(k)

#         # 初始状态
#         state = self.initial_state_from_W(W, X)

#         assert exter1 + inter + exter2 == rounds
#         round_idx = 0

#         # 轮函数迭代#############注意这里的轮常数序列是错的
#         for _ in range(exter1):
#             state = self.rf_round_polynomial(state, round_idx)
#             round_idx += 1
#         for _ in range(inter):
#             state = self.rp_round_polynomial(state, round_idx)
#             round_idx += 1
#         for _ in range(exter2):
#             state = self.rf_round_polynomial(state, round_idx)
#             round_idx += 1

#         # for r in range(rounds):
#         #     if round_type == "internal":
#         #         state = self.rp_round_polynomial(state, r)
#         #     else:
#         #         state = self.rf_round_polynomial(state, r)

#         # 构造 CICO 方程
#         const_vec = vector(R, [R(c) for c in constants])
#         equations = list(state - const_vec)

#         return equations, R, X

#     def generate_cico_equations_withnosubspace(self, constants, rounds, exter1, inter, exter2):
#         """
#         round_type: "internal" or "external"
#         """
#         k = self.t
#         R, X = self.make_symbolic_X(k)
#         W = identity_matrix(self.F, self.t)
#         # lift 到多项式环
#         state = W.change_ring(R) * X

#         assert exter1 + inter + exter2 == rounds
#         round_idx = 0

#         # 轮函数迭代#############注意这里的轮常数序列是错的
#         for _ in range(exter1):
#             state = self.rf_round_polynomial(state, round_idx)
#             round_idx += 1
#         for _ in range(inter):
#             state = self.rp_round_polynomial(state, round_idx)
#             round_idx += 1
#         for _ in range(exter2):
#             state = self.rf_round_polynomial(state, round_idx)
#             round_idx += 1

#         # for r in range(rounds):
#         #     if round_type == "internal":
#         #         state = self.rp_round_polynomial(state, r)
#         #     else:
#         #         state = self.rf_round_polynomial(state, r)

#         # 构造 CICO 方程
#         const_vec = vector(R, [R(c) for c in constants])
#         equations = list(state - const_vec)

#         return equations, R, X

#     def simple_form(self, target, rounds):
#         state = target
#         for round_idx in range(rounds):
#             state = self.inv_internal_round(state, round_idx)
#         return state
    
#     def retain_dim_shear_nosubspace(self, constants):######在S盒之前=t引入参数
#         k = 3
#         num_t = 16#######加一个高次参数
#         # R, X, t= self.make_symbolic_X_with_t(k)
#         R, X, T = self.make_symbolic_X_with_t(k, num_t)

#         # state = vector(R, [T[19]*X[0]+T[0], 
#         #                    T[20]*X[1]+T[1],
#         #                    T[21]*X[2]+T[2],
#         #                    T[22]*X[3]+T[3],
#         #                    R.zero()])
#         state = vector(R, [T[0], 
#                             T[1]*X[0]+T[2],
#                             T[3]*X[1]+T[4],
#                             T[5]*X[2]+T[6],
#                             R.zero()])
#         round_idx = 0

#         # 轮函数迭代#############注意这里的轮常数序列是错的
#         eqs = []
#         reduced_ideal = []

#         state = self.rp_round_polynomial(state, round_idx)
#         round_idx += 1
#         f = R(state[0])
#         eq = f - T[7]
#         eqs.append(eq)
#         reduced_ideal.append(eq)
#         state = vector(R, [T[7]] + list(state[1:]))

#         state = self.rp_round_polynomial(state, round_idx)
#         round_idx += 1
#         state = self.rp_round_polynomial(state, round_idx)
#         round_idx += 1
#         f = R(state[0])
#         eq = f - (T[8]*X[0]+T[9]*X[1]+T[10]*X[2]+T[11])
#         eqs.append(eq)
#         reduced_ideal.append(eq)
#         state = vector(R, [T[8]*X[0]+T[9]*X[1]+T[10]*X[2]+T[11]] + list(state[1:]))

#         # state = self.rp_round_polynomial(state, round_idx)
#         # round_idx += 1
#         # f = R(state[0])
#         # eq = f - (T[15]*X[0]+T[16]*X[1]+T[17]*X[2]+T[14])
#         # eqs.append(eq)
#         # reduced_ideal.append(eq)
#         # state = vector(R, [T[15]*X[0]+T[16]*X[1]+T[17]*X[2]+T[14]] + list(state[1:]))

#         state = self.rp_round_polynomial(state, round_idx)
#         round_idx += 1
#         # state = self.rp_round_polynomial(state, round_idx)
#         # round_idx += 1

#         # 构造 CICO 方程
#         # const_vec = vector(R, [R(c) for c in constants])
#         # eqs += list(state - const_vec)
#         target_state = vector(R, [
#             T[12]*X[0] + T[13]*X[1] + T[14]*X[2] + T[15],
#             R(constants[1]),
#             R(constants[2]),
#             R(constants[3]),
#             R(constants[4]),
#         ])
#         eqs += list(state - target_state)

#         return eqs, reduced_ideal, R, X, T
    
# def square_submatrix(M):
#     n = M.ncols()
#     return M[:n, :]

In [5]:
# poseidon = PoseidonSage(seed=0)
# W_list = poseidon.compute_mds_invariant_subspaces_sage()
# poseidon.show_invariant_subspace_dynamics(W_list)


In [6]:
def x_degree(f, x_vars):
    """
    只统计多项式 f 在 x_vars 上的最大次数，不计 t 变量次数
    """
    if f == 0:
        return -1

    dmax = 0
    for exp in f.exponents():
        d = sum(exp[i] for i in range(len(x_vars)))
        if d > dmax:
            dmax = d
    return dmax

def coeff_only_t(f, mon, x_vars, S):
    """
    从 f 中提取 x-单项式 mon 的系数，系数中允许含 t 变量。

    例如：
        f = (t0+1)*x0^2 + (t0^2+t1)*x0*x1 + 3*t1*x1^2
    若 mon = x0^2
    返回:
        t0 + 1
    """

    R = f.parent()
    gens = R.gens()

    x_vars = list(x_vars)
    x_idx = [gens.index(x) for x in x_vars]
    t_idx = [i for i in range(len(gens)) if i not in x_idx]

    # t 变量子环
    T = S.gens()
    # t_names = [str(gens[i]) for i in t_idx]
    # S = PolynomialRing(R.base_ring(), t_names)
    # T = S.gens()

    # mon 在 x 变量上的目标指数
    target_exp = tuple(mon.degree(x) for x in x_vars)

    res = S.zero()

    for exp, coeff in f.dict().items():
        x_exp = tuple(exp[i] for i in x_idx)

        # 必须和列单项式完全匹配
        if x_exp == target_exp:
            term = S(coeff)
            for j, idx in enumerate(t_idx):
                if exp[idx] != 0:
                    term *= T[j] ** exp[idx]
            res += term

    return res
import time 

def build_macaulay_matrix(eqs, reduced_eqs, n, mult_deg, col_deg,
                          f3_index=-1,
                          lift_f3=True,
                          homogeneous=True):
    R = eqs[0].parent()
    X = list(R.gens()[:n])

    gens = R.gens()
    t_vars = list(gens[n:])
    S = PolynomialRing(R.base_ring(), [str(v) for v in t_vars])

    f3 = eqs[f3_index]
    print("enter build_macaulay_matrix")
    # 列
    if homogeneous:
        col_mons = monomials_of_degree(R, X, col_deg)
        mult_mons = monomials_of_degree(R, X, mult_deg)
    else:
        col_mons = monomials_leq_degree(R, X, col_deg)
        mult_mons = monomials_leq_degree(R, X, mult_deg)
        
    print("got monomials", len(mult_mons), len(col_mons))

    # <f1,f2> 产生的行
    base_rows = []
    for m in mult_mons:
        for f in reduced_eqs:
            print("multiplying one row...")
            base_rows.append(m * f)
    print("base_rows done", len(base_rows))
    matrices = []

    # f3 不升次
    if not lift_f3:
        rows = list(base_rows)
        rows.append(f3)

        M = matrix(S, len(rows), len(col_mons))
        # for i, f in enumerate(rows):
            # for j, mon in enumerate(col_mons):
            #     M[i, j] = coeff_only_t(f, mon, X, S)
        for i, f in enumerate(rows):
            cmap = {}
            for mon_exp, coeff in f.dict().items():
                x_exp = tuple(mon_exp[:len(X)])
                t_exp = tuple(mon_exp[len(X):])

                term = S(coeff)
                for k, e in enumerate(t_exp):
                    if e != 0:
                        term *= S.gen(k)**e

                cmap[x_exp] = cmap.get(x_exp, S.zero()) + term

            for j, mon in enumerate(col_mons):
                exp = tuple(mon.degree(v) for v in X)
                M[i, j] = cmap.get(exp, S.zero())
        return [M], col_mons

    # f3 升次
    deg_f3_x = max(sum(mon.degree(x) for x in X) for mon in f3.monomials()) if f3 != 0 else -1
    lift_deg = col_deg - deg_f3_x

    if lift_deg < 0:
        raise ValueError(f"col_deg={col_deg} 小于 f3 的 x-次数 {deg_f3_x}")

    if lift_deg == 0:
        lift_mons = [R.one()]
    else:
        lift_mons = monomials_of_degree(R, X, lift_deg)

    for m in lift_mons:
        rows = list(base_rows)
        rows.append(m * f3)

        M = matrix(S, len(rows), len(col_mons))
        # t0 = time.time()
        # for i, f in enumerate(rows):
        #     # print("building row", i, "num_terms =", len(f.dict()))
        #     # t1 = time.time()
        #     for j, mon in enumerate(col_mons):
        #         M[i, j] = coeff_only_t(f, mon, X)
        #     # print("row", i, "done, time =", time.time() - t1)  
        # # print("total build time =", time.time() - t0)
        for i, f in enumerate(rows):
            cmap = {}
            for mon_exp, coeff in f.dict().items():
                x_exp = tuple(mon_exp[:len(X)])
                t_exp = tuple(mon_exp[len(X):])

                term = S(coeff)
                for k, e in enumerate(t_exp):
                    if e != 0:
                        term *= S.gen(k)**e

                cmap[x_exp] = cmap.get(x_exp, S.zero()) + term

            for j, mon in enumerate(col_mons):
                exp = tuple(mon.degree(v) for v in X)
                M[i, j] = cmap.get(exp, S.zero())
        matrices.append(M)

    return matrices, col_mons

def build_macaulay_matrix_fast(
    eqs,
    reduced_ideal,
    n,
    mult_deg,
    col_deg,
    f3_index=3,
    lift_f3=False,
    homogeneous=False
):

    R = eqs[0].parent()
    gens = list(R.gens())

    X = gens[:n]
    T = gens[n:]

    # ---------- columns ----------
    if homogeneous:
        col_mons = monomials_of_degree(R, X, col_deg)
        mult_mons = monomials_of_degree(R, X, mult_deg)
    else:
        col_mons = monomials_leq_degree(R, X, col_deg)
        mult_mons = monomials_leq_degree(R, X, mult_deg)

    col_exp = [tuple(m.degree(v) for v in X) for m in col_mons]
    col_index = {e:i for i,e in enumerate(col_exp)}

    mult_exp = [tuple(m.degree(v) for v in X) for m in mult_mons]

    # ---------- coefficient extraction ----------
    def extract_x_coeffs(f):

        d = {}

        for mon_exp, coeff in f.dict().items():

            x_exp = tuple(mon_exp[:n])
            t_exp = mon_exp[n:]

            # rebuild t polynomial
            term = coeff
            for i,e in enumerate(t_exp):
                if e:
                    term *= T[i]**e

            d[x_exp] = d.get(x_exp, R.zero()) + term

        return d

    reduced_maps = [extract_x_coeffs(f) for f in reduced_ideal]
    last_map = extract_x_coeffs(eqs[f3_index])

    rows = []

    # ---------- lifted rows ----------
    for beta in mult_exp:
        for cmap in reduced_maps:

            row = [R.zero()] * len(col_mons)

            for alpha, tpoly in cmap.items():

                new_exp = tuple(a+b for a,b in zip(alpha,beta))

                j = col_index.get(new_exp)
                if j is not None:
                    row[j] += tpoly

            rows.append(row)

    # ---------- last equation ----------
    row = [R.zero()] * len(col_mons)

    for alpha, tpoly in last_map.items():

        j = col_index.get(alpha)
        if j is not None:
            row[j] += tpoly

    rows.append(row)

    M = matrix(R, rows)

    return [M], col_mons

def build_macaulay_matrix_mixed_fast(
    eqs,
    reduced_ideal,
    n,
    mult_deg,
    col_deg,
    f3_index=2,
    lift_f3=False,
    homogeneous=False
):
    """
    支持 reduced_ideal 中不同生成元使用不同乘子次数的 Macaulay 矩阵构造。

    参数：
        eqs            : 原方程列表
        reduced_ideal  : 用来生成前缀行的小理想生成元列表，如 [f1, f2]
        n              : x 变量个数（前 n 个变量视为 x，其余视为 t）
        mult_deg       : 
            - 若为 int，则所有 reduced_ideal 生成元统一用这个次数
            - 若为 list/tuple，则第 i 个生成元用 mult_deg[i]
              例如 [2,0] 表示 f1 乘 <=2 次单项式，f2 只乘 1
        col_deg        : 列单项式最大次数（或恰次数，取决于 homogeneous）
        f3_index       : eqs 中“目标多项式”下标
        lift_f3        : 是否也对 f3 做升次
        homogeneous    :
            - True  : 用恰次数 monomials_of_degree
            - False : 用小于等于次数 monomials_leq_degree

    返回：
        [M], col_mons
    """

    R = eqs[0].parent()
    gens = list(R.gens())

    X = gens[:n]
    T = gens[n:]

    r = len(reduced_ideal)

    # ---- 统一 mult_deg 成列表 ----
    if isinstance(mult_deg, (list, tuple)):
        if len(mult_deg) != r:
            raise ValueError(f"mult_deg 长度应等于 len(reduced_ideal)={r}，当前是 {len(mult_deg)}")
        mult_deg_list = list(mult_deg)
    else:
        mult_deg_list = [mult_deg] * r

    # ---- columns ----
    if homogeneous:
        col_mons = monomials_of_degree(R, X, col_deg)
    else:
        col_mons = monomials_leq_degree(R, X, col_deg)

    col_exp = [tuple(m.degree(v) for v in X) for m in col_mons]
    col_index = {e: i for i, e in enumerate(col_exp)}

    # ---- coefficient extraction ----
    def extract_x_coeffs(f):
        d = {}
        for mon_exp, coeff in f.dict().items():
            x_exp = tuple(mon_exp[:n])
            t_exp = mon_exp[n:]

            term = coeff
            for i, e in enumerate(t_exp):
                if e:
                    term *= T[i] ** e

            d[x_exp] = d.get(x_exp, R.zero()) + term
        return d

    reduced_maps = [extract_x_coeffs(f) for f in reduced_ideal]
    f3_map = extract_x_coeffs(eqs[f3_index])

    rows = []

    # ---- 前缀行：每个生成元各自按自己的 mult_deg 升次 ----
    for cmap, d in zip(reduced_maps, mult_deg_list):
        if homogeneous:
            if d < 0:
                raise ValueError("homogeneous=True 时 mult_deg 不能为负")
            mult_mons = monomials_of_degree(R, X, d)
        else:
            if d < 0:
                raise ValueError("homogeneous=False 时 mult_deg 不能为负")
            mult_mons = monomials_leq_degree(R, X, d)

        mult_exp = [tuple(m.degree(v) for v in X) for m in mult_mons]

        for beta in mult_exp:
            row = [R.zero()] * len(col_mons)

            for alpha, tpoly in cmap.items():
                new_exp = tuple(a + b for a, b in zip(alpha, beta))
                j = col_index.get(new_exp)
                if j is not None:
                    row[j] += tpoly

            rows.append(row)

    # ---- f3 行 ----
    if not lift_f3:
        row = [R.zero()] * len(col_mons)
        for alpha, tpoly in f3_map.items():
            j = col_index.get(alpha)
            if j is not None:
                row[j] += tpoly
        rows.append(row)

    else:
        # 自动根据 f3 的 x-次数决定还要乘多少次
        deg_f3_x = max((sum(exp[:n]) for exp in eqs[f3_index].dict().keys()), default=-1)
        lift_deg = col_deg - deg_f3_x

        if lift_deg < 0:
            raise ValueError(f"col_deg={col_deg} 小于 f3 的 x-次数 {deg_f3_x}")

        if homogeneous:
            lift_mons = monomials_of_degree(R, X, lift_deg)
        else:
            lift_mons = monomials_leq_degree(R, X, lift_deg)

        lift_exp = [tuple(m.degree(v) for v in X) for m in lift_mons]

        for beta in lift_exp:
            row = [R.zero()] * len(col_mons)
            for alpha, tpoly in f3_map.items():
                new_exp = tuple(a + b for a, b in zip(alpha, beta))
                j = col_index.get(new_exp)
                if j is not None:
                    row[j] += tpoly
            rows.append(row)

    M = matrix(R, rows)
    return [M], col_mons


def make_square_and_det(M, keep_last=True, verbose=False):
    """
    M: 原矩阵
    keep_last: 是否保留最后一行（目标方程）
    返回: 方阵，行列式
    """
    rows, cols = M.nrows(), M.ncols()
    
    if rows < cols:
        print("矩阵行数不足，无法变成方阵")
        return None, None
    else: 
    # 先确定要保留的行
        if keep_last:
            # 保留最后一行
            keep_idx = [rows-1]  # 最后一行
            # 然后按顺序选够 cols-1 行
            for i in range(rows-1):
                if len(keep_idx) >= cols:
                    break
                keep_idx.insert(0, i)  # 从前面开始插入
        else:
            # 直接选前 cols 行
            keep_idx = list(range(cols))
        
        if verbose:
            print(f"保留的行索引: {keep_idx}")

        # 生成方阵
        M_square = M[keep_idx, :]
        
        # 计算行列式
        det_val = M_square.det()
        
        return M_square, det_val

def test_random_ranks(M, trials=10):
    S = M.base_ring()
    F = S.base_ring()
    ts = S.gens()

    for _ in range(trials):
        vals = [F.random_element() for _ in ts]
        M0 = matrix(F, M.substitute({ts[i]: vals[i] for i in range(len(ts))}))
        r1 = M0.rank()
        r2 = M0[:-1].rank()
        print("vals =", vals, " rank =", r1, " rank_without_last =", r2, " same?", r1 == r2)

def solve_t_for_last_row_span(M):
    """
    设前 r 行线性组合等于最后一行：
        c0*row0 + ... + c_{r-1}*row_{r-1} = last_row
    返回一个带消元顺序的多项式环，以及对应方程。
    """
    S = M.base_ring()          # 例如 GF(p)[t0,t1,t2]
    F = S.base_ring()
    t_names = [str(x) for x in S.gens()]

    r = M.nrows() - 1
    c_names = [f'c{i}' for i in range(r)]

    # 关键：消元顺序，先消 c，再留 t
    # c 在前，t 在后；lex 通常够用
    Rbig = PolynomialRing(F, c_names + t_names, order='lex')
    vars_big = Rbig.gens()

    nc = len(c_names)
    nt = len(t_names)

    c_big = vars_big[:nc]
    t_big = vars_big[nc:]

    # 把 S 中的 t 变量映到 Rbig 中的 t_big
    sub_t = {S.gens()[i]: t_big[i] for i in range(nt)}

    eqs = []
    for j in range(M.ncols()):
        lhs = Rbig.zero()
        for i in range(r):
            lhs += c_big[i] * Rbig(M[i, j].subs(sub_t))
        rhs = Rbig(M[r, j].subs(sub_t))
        eqs.append(lhs - rhs)

    return Rbig, eqs, c_big, t_big

def independent_row_basis(M):
    """
    对矩阵 M 的前 n-1 行，在分式域上取一组线性无关“行空间基”。
    这里返回的不是原始若干行，而是 echelon form 里的非零行，
    也就是允许线性组合后的基。

    返回:
        B : 前缀行空间的一组基（matrix over fraction field）
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    A = Mf[:-1, :]
    E = A.echelon_form()

    rows = []
    for i in range(E.nrows()):
        if E.row(i) != 0:
            rows.append(E.row(i))

    if len(rows) == 0:
        return matrix(K, 0, M.ncols())

    return matrix(K, rows)

def compress_rows_keep_last(M):
    """
    把前缀行压成一组线性无关基，再把最后一行接上。
    """
    S = M.base_ring()
    K = S.fraction_field()

    B = independent_row_basis(M)
    last = matrix(K, [matrix(K, M)[-1]])

    return B.stack(last)

def support_columns_of_matrix(M):
    """
    返回矩阵 M 中所有出现过非零元的列下标。
    """
    cols = []
    for j in range(M.ncols()):
        nonzero = False
        for i in range(M.nrows()):
            if M[i, j] != 0:
                nonzero = True
                break
        if nonzero:
            cols.append(j)
    return cols

def pivot_columns_of_prefix(M):
    """
    返回前面行（不含最后一行）在分式域上的 pivot 列
    """
    S = M.base_ring()
    K = S.fraction_field()
    A = matrix(K, M[:-1, :])
    piv = A.pivots()
    return list(piv)
def important_columns(M, extra=5):
    """
    取 pivot 列，再额外补一些最后一行非零列
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    piv = pivot_columns_of_prefix(M)
    last = Mf[-1]

    extra_cols = []
    for j in range(M.ncols()):
        if j not in piv and last[j] != 0:
            extra_cols.append(j)
        if len(extra_cols) >= extra:
            break

    cols = sorted(set(piv + extra_cols))
    return cols
def compress_matrix(M):
    """
    先压前缀行到线性无关基，再保留必要列。

    必要列 = 前缀基真正用到的列 ∪ 最后一行真正用到的列
    """
    M1 = compress_rows_keep_last(M)
    cols = safe_columns_for_last_row_test(M)
    return M1[:, cols], cols

def independent_row_indices_over_fraction_field(M):
    """
    对矩阵 M 的前 n-1 行，在分式域上找一组线性无关的行下标
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    A = Mf[:-1, :]
    idx = []
    current = matrix(K, 0, A.ncols())

    for i in range(A.nrows()):
        candidate = current.stack(matrix(K, [A.row(i)]))
        if candidate.rank() > current.rank():
            idx.append(i)
            current = candidate

    return idx

def t_conditions_from_last_row_span(M, pivot_cols_override=None, verbose=True):
    """
    目标：
        找 t 的条件，使最后一行属于“前面各行张成的行空间”。

    这里采用的是“显式解系数 c 再代回”的方法，不是 Groebner 消元版。

    做法：
    1. 对前缀行取一组线性无关基 B（允许线性组合）
    2. 选 B 的一组 pivot 列
    3. 在这些列上解出系数 c(t)
    4. 代回全部列，得到残差 residual_j(t)
    5. 取这些残差的分子，作为只关于 t 的候选条件

    返回:
        {
            "B": 前缀行空间的一组基
            "pivot_cols": 用来解 c 的列
            "c_sol": 解出的系数 c(t)
            "residuals": 所有列上的残差
            "numerators": 残差分子去重后的列表
        }

    注意：
    - 这是在某个 pivot minor 非零的开集上成立的条件。
    - 若该 minor 为 0，则这组 c(t) 的公式可能失效，需要换别的列块。
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    # 前缀行与最后一行
    A = Mf[:-1, :]
    b = vector(K, Mf[-1])

    # 新逻辑：前缀行空间的一组基（允许线性组合）
    B = independent_row_basis(M)

    if verbose:
        print("原前缀行数 =", A.nrows())
        print("前缀行空间维数 =", B.nrows())
        print("列数 =", B.ncols())

    r = B.nrows()
    n = B.ncols()

    if r == 0:
        # 前缀全是零行
        residuals = [ -b[j] for j in range(n) ]
        numerators = []
        for rj in residuals:
            if rj != 0:
                try:
                    num = rj.numerator()
                except AttributeError:
                    num = rj
                if num != 0:
                    numerators.append(num)

        numerators_unique = []
        seen = set()
        for g in numerators:
            s = str(g)
            if s not in seen:
                seen.add(s)
                numerators_unique.append(g)

        return {
            "B": B,
            "pivot_cols": [],
            "c_sol": vector(K, []),
            "residuals": residuals,
            "numerators": numerators_unique,
        }

    # 选 B 的 pivot 列：要从列里挑 r 列，使对应子块可逆
    if pivot_cols_override is None:
        piv = list(B.pivots())
        if len(piv) < r:
            raise ValueError("B 的 pivot 列数小于行秩，无法构造 c(t) 的解")
    else:
        piv = list(pivot_cols_override)
        if len(piv) != r:
            raise ValueError(f"手动指定的 pivot 列数必须等于前缀行空间维数 r={r}")
        if len(set(piv)) != r:
            raise ValueError("手动指定的 pivot 列里有重复")
        if min(piv) < 0 or max(piv) >= n:
            raise ValueError("手动指定的 pivot 列超出范围")

    if verbose:
        print("pivot 列 =", piv)

    # C 是 r x r 子块，要求可逆
    C = B.matrix_from_columns(piv)
    detC = C.det()

    if verbose:
        print("det(pivot block) =", detC)

    if detC == 0:
        raise ValueError("当前指定的 pivot 列对应子块不可逆，请换一组列")

    # 在 pivot 列上解：
    #   sum_i c_i * B[i, piv[k]] = b[piv[k]]
    # 也就是 (C^T) c = b_piv
    rhs = vector(K, [b[j] for j in piv])
    c_sol = C.transpose().solve_right(rhs)

    if verbose:
        print("解出的 c(t) =")
        for i, ci in enumerate(c_sol):
            print(f"c[{i}] = {ci}")

    # 代回所有列，求残差
    residuals = []
    for j in range(n):
        val = sum(c_sol[i] * B[i, j] for i in range(r)) - b[j]
        residuals.append(val)

    # 取分子
    numerators = []
    for rj in residuals:
        if rj != 0:
            try:
                num = rj.numerator()
            except AttributeError:
                num = rj
            if num != 0:
                numerators.append(num)

    # 去重
    numerators_unique = []
    seen = set()
    for g in numerators:
        s = str(g)
        if s not in seen:
            seen.add(s)
            numerators_unique.append(g)

    return {
        "B": B,
        "pivot_cols": piv,
        "pivot_det": detC,
        "c_sol": c_sol,
        "residuals": residuals,
        "numerators": numerators_unique,
    }

def print_t_conditions(info):
    nums = info["numerators"]
    print("只关于 t 的候选条件（来自残差分子）:")
    if len(nums) == 0:
        print("没有残差条件：在当前选的 pivot minor 非零时，最后一行已经被 span。")
    else:
        for i, g in enumerate(nums):
            print(f"[{i}] {g}")

def kernel_hits_last_row(M, verbose=True):
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    Kleft = Mf.transpose().right_kernel()
    basis = Kleft.basis()

    if verbose:
        print("kernel dimension =", Kleft.dimension())
        for idx, v in enumerate(basis):
            print(f"basis[{idx}] =", v)
            print("last entry =", v[-1])
            print("----")

    hit = any(v[-1] != 0 for v in basis)
    print("是否存在最后一项非零的核向量:", hit)
    return Kleft, hit

def safe_columns_for_last_row_test(M):
    """
    安全压列：

    1. 先对前缀行取一组线性无关基 B（允许线性组合）
    2. 保留 B 中所有真正出现过的列
    3. 再并上最后一行中所有非零列

    这样不会因为只保留 pivot 列而丢掉前缀行空间的信息。
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    B = independent_row_basis(M)
    last = Mf[-1]

    supp_B = support_columns_of_matrix(B)
    supp_last = [j for j in range(M.ncols()) if last[j] != 0]

    cols = sorted(set(supp_B + supp_last))
    return cols

def compress_rows_keep_last_only(M):
    """
    只对前缀行做压缩：
    - 前缀行变成一组线性无关基（允许线性组合）
    - 最后一行原样保留

    返回:
        Msmall : 压缩后的矩阵
        info   : 一些辅助信息
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    B = independent_row_basis(M)
    last = matrix(K, [Mf[-1]])

    Msmall = B.stack(last)

    info = {
        "prefix_rank": B.nrows(),
        "ncols": M.ncols(),
        "basis_support_cols": support_columns_of_matrix(B),
        "last_row_support_cols": [j for j in range(M.ncols()) if Mf[-1, j] != 0],
    }

    return Msmall, info

def variables_in_polys(polys):
    """
    从多项式列表里提取真正出现过的变量。
    返回按字符串排序后的变量列表。
    """
    vars_set = set()
    for f in polys:
        try:
            vars_set.update(f.variables())
        except AttributeError:
            pass
    return sorted(vars_set, key=str)
def analyze_t_ideal_from_info(info, verbose=True):
    """
    对 info["numerators"] 生成的 t-理想做分析。

    返回:
        {
            "polys": ...,
            "vars": ...,
            "ideal": ...,
            "gb": ...,
            "dimension": ...,
            "is_zero_dimensional": ...,
        }
    """
    polys = [f for f in info["numerators"] if f != 0]

    if len(polys) == 0:
        print("没有任何 t-约束，多半表示在当前开集上自动成立。")
        return {
            "polys": [],
            "vars": [],
            "ideal": None,
            "gb": None,
            "dimension": None,
            "is_zero_dimensional": None,
        }

    R = polys[0].parent()
    tvars = variables_in_polys(polys)
    J = R.ideal(polys)

    if verbose:
        print("=== t-ideal basic info ===")
        print("number of generators =", len(polys))
        print("variables appearing =", tvars)
        print()

        print("generators:")
        for i, f in enumerate(polys):
            print(f"[{i}] {f}")
        print()

    # Groebner basis
    try:
        gb = J.groebner_basis()
    except Exception as e:
        print("Groebner basis 计算失败：", e)
        gb = None

    if verbose and gb is not None:
        print("Groebner basis:")
        for i, g in enumerate(gb):
            print(f"GB[{i}] {g}")
        print()

    # 维数判断
    dimJ = None
    is_zero_dim = None

    try:
        dimJ = J.dimension()
        is_zero_dim = (dimJ == 0)
    except Exception as e:
        if verbose:
            print("J.dimension() 失败：", e)
        # 退化判断：如果 Groebner basis 里每个变量都有某个纯变量首项，往往是 0 维
        if gb is not None:
            try:
                is_zero_dim = J.is_zero_dimensional()
            except Exception:
                is_zero_dim = None

    if verbose:
        print("dimension =", dimJ)
        print("is_zero_dimensional =", is_zero_dim)
        print()

    return {
        "polys": polys,
        "vars": tvars,
        "ideal": J,
        "gb": gb,
        "dimension": dimJ,
        "is_zero_dimensional": is_zero_dim,
    }

def analyze_t_ideal_small_ring(info, verbose=True):
    """
    适配当前 info 结构:
        info.keys() = ['B', 'pivot_cols', 'pivot_det', 'c_sol', 'residuals', 'numerators']

    做的事：
      1. 从 numerators 读取方程
      2. 从这些方程的 parent / 或 B 的 base ring 推出原环
      3. 自动识别名字以 t 开头的变量
      4. 建立只含 t 的小环
      5. 把 numerators 搬到小环并做 Groebner 分析
    """

    if "numerators" not in info:
        raise KeyError(f"info 里没有 'numerators'。现有 keys = {list(info.keys())}")

    numerators = info["numerators"]
    if len(numerators) == 0:
        raise ValueError("info['numerators'] 是空的，无法建立小理想")

    if verbose:
        print("=== info keys ===")
        print(list(info.keys()))
        print("number of numerators =", len(numerators))

    # 1) 推原环 R
    R = None

    # 最稳：先从 numerator parent 取
    try:
        R = numerators[0].parent()
    except Exception:
        R = None

    # 若上面失败，再尝试从 B 的 base ring 取
    if R is None and "B" in info:
        try:
            R = info["B"].base_ring()
        except Exception:
            R = None

    if R is None:
        raise ValueError("无法从 numerators 或 B 推断原多项式环")

    if verbose:
        print("\nDetected parent ring R:")
        print(R)

    # 2) 自动找 t 变量
    gens = list(R.gens())
    t_vars = [g for g in gens if str(g).startswith("t")]

    if len(t_vars) == 0:
        raise ValueError(f"在原环生成元里没有找到名字以 't' 开头的变量。gens = {gens}")

    if verbose:
        print("\nDetected t_vars:")
        print(t_vars)

    # 3) 建小环
    F = R.base_ring()
    Rsmall = PolynomialRing(F, [str(v) for v in t_vars], order='lex')
    us = Rsmall.gens()
    sub_map = {t_vars[i]: us[i] for i in range(len(t_vars))}

    if verbose:
        print("\nRsmall =")
        print(Rsmall)

    # 4) 把 numerators 搬到小环
    polys_small = []
    failed = []

    for i, f in enumerate(numerators):
        if f == 0:
            continue

        # 先检查是否真的只含 t 变量
        try:
            vars_f = f.variables()
        except Exception:
            vars_f = ()

        bad_vars = [v for v in vars_f if v not in t_vars]
        if len(bad_vars) != 0:
            failed.append((i, f, f"contains non-t vars: {bad_vars}"))
            continue

        try:
            g = f.subs(sub_map)
        except Exception:
            g = f

        try:
            g = Rsmall(g)
        except Exception as e:
            failed.append((i, f, f"coercion failed: {e}"))
            continue

        if g != 0:
            polys_small.append(g)

    if verbose:
        print("\nlen(polys_small) =", len(polys_small))
        for i, g in enumerate(polys_small[:10]):
            print(f"[{i}] {g}")
            print("parent =", g.parent())

        if len(failed) > 0:
            print("\nSkipped numerators:")
            for item in failed[:10]:
                idx, f, reason = item
                print(f"[{idx}] reason = {reason}")
                print("    ", f)

    if len(polys_small) == 0:
        return {
            "ring": Rsmall,
            "polys": [],
            "ideal": Rsmall.ideal([]),
            "groebner_basis": [],
            "dimension": None,
            "t_vars": t_vars,
            "failed": failed,
            "error": "no valid t-only numerators could be moved into Rsmall",
        }

    # 去重，避免重复方程拖慢
    polys_small = list(dict.fromkeys(polys_small))

    if verbose:
        print("\nafter dedup, len(polys_small) =", len(polys_small))

    J = Rsmall.ideal(polys_small)

    if verbose:
        print("\nIdeal J created successfully.")
        print(J)

    # 5) Groebner
    try:
        gb = J.groebner_basis()
    except Exception as e:
        if verbose:
            print("\nGroebner failed:")
            print(e)
        return {
            "ring": Rsmall,
            "polys": polys_small,
            "ideal": J,
            "groebner_basis": None,
            "dimension": None,
            "t_vars": t_vars,
            "failed": failed,
            "error": str(e),
        }

    # 6) 维数
    try:
        dimJ = J.dimension()
    except Exception as e:
        dimJ = None
        if verbose:
            print("\nDimension computation failed:")
            print(e)

    if verbose:
        print("\nGroebner basis:")
        for g in gb:
            print(g)
        print("dimension =", dimJ)

    return {
        "ring": Rsmall,
        "polys": polys_small,
        "ideal": J,
        "groebner_basis": gb,
        "dimension": dimJ,
        "t_vars": t_vars,
        "failed": failed,
    }

def solve_zero_dim_t_ideal(analysis, verbose=True):
    """
    若 analysis 对应的理想是 0 维，尝试直接求解。
    """
    J = analysis["ideal"]
    if J is None:
        print("理想为空。")
        return None

    if analysis["is_zero_dimensional"] is not True:
        print("这个理想目前看起来不是 0 维，或者还无法确认 0 维，不能直接按有限解集处理。")
        return None

    R = analysis["polys"][0].parent()

    # 方法1：尝试 variety
    try:
        sols = J.variety()
        print("variety() 找到的解：")
        for k, sol in enumerate(sols):
            print(f"solution {k}: {sol}")
        return sols
    except Exception as e:
        if verbose:
            print("J.variety() 失败：", e)

    # 方法2：如果基域不是代数闭域，尝试到分式域/QQbar
    try:
        from sage.all import QQbar
        S = PolynomialRing(QQbar, R.variable_names(), order='lex')
        gens2 = [S(f) for f in J.gens()]
        J2 = S.ideal(gens2)
        sols = J2.variety()
        print("在 QQbar 上 variety() 找到的解：")
        for k, sol in enumerate(sols):
            print(f"solution {k}: {sol}")
        return sols
    except Exception as e:
        if verbose:
            print("转到 QQbar 后 variety() 仍失败：", e)

    print("没能直接解出有限解集。")
    return None

def analyze_and_try_solve_t_conditions(info, verbose=True):
    analysis = analyze_t_ideal_from_info(info, verbose=verbose)

    if analysis["ideal"] is None:
        return analysis, None

    sols = None
    if analysis["is_zero_dimensional"] is True:
        sols = solve_zero_dim_t_ideal(analysis, verbose=verbose)
    else:
        print("暂时看不是 0 维，或者无法确认是 0 维。")
        print("这不一定表示无解，只是说明解集可能是正维的，或者 Groebner 分析还不够。")

    return analysis, sols

def simplify_polys_for_ideal(polys):
    """
    对多项式做一点简单清理：
    - 去掉 0
    - 尽量 primitive part / squarefree part
    - 去重
    """
    out = []
    seen = set()

    for f in polys:
        if f == 0:
            continue

        g = f
        try:
            # 如果有分数内容，先 primitive_part
            g = g.primitive_part()
        except Exception:
            pass

        try:
            # 平方自由部分
            g = g.squarefree_part()
        except Exception:
            pass

        s = str(g)
        if s not in seen and g != 0:
            seen.add(s)
            out.append(g)

    return out

def evaluate_expr_at_point(expr, pt):
    """
    在点 pt 处对 expr 求值。
    pt 形如 {"t0": a, "t1": b, "t2": c}
    """
    try:
        P = expr.parent()
        gens = P.gens()
    except Exception:
        return expr

    if len(gens) == 0:
        return expr

    vals = []
    used_any = False
    for g in gens:
        name = str(g)
        if name in pt:
            vals.append(pt[name])
            used_any = True
        else:
            vals.append(g)

    if not used_any:
        return expr

    try:
        return expr(*vals)
    except TypeError:
        return expr


def denominator_part(expr):
    """
    返回 expr 的分母；如果 expr 不是分式，就返回 1
    """
    try:
        return expr.denominator()
    except AttributeError:
        return 1


def collect_avoid_polynomial_from_info(info):
    """
    把当前 chart 下所有必须避开的分母收集起来，乘成一个 avoid_poly。
    只要 avoid_poly(pt) != 0，说明当前这套 c(t)/residual(t) 表达式在 pt 处合法。
    """
    polys = []

    # c_sol 的分母
    for ci in info["c_sol"]:
        d = denominator_part(ci)
        if d != 1:
            polys.append(d)

    # residuals 的分母
    for rj in info["residuals"]:
        d = denominator_part(rj)
        if d != 1:
            polys.append(d)

    if len(polys) == 0:
        return 1

    R = polys[0].parent()
    avoid = R.one()

    # 去重后相乘
    seen = set()
    for d in polys:
        s = str(d)
        if s not in seen:
            seen.add(s)
            avoid *= d

    return avoid


def specialize_matrix_at_point(M, pt):
    """
    把矩阵 M 在点 pt 处代值。
    若某项分母为 0，则报错。
    """
    rows = []

    for i in range(M.nrows()):
        row = []
        for j in range(M.ncols()):
            f = M[i, j]

            try:
                num = f.numerator()
                den = f.denominator()
            except AttributeError:
                num = f
                den = 1

            numv = evaluate_expr_at_point(num, pt)
            denv = evaluate_expr_at_point(den, pt)

            if denv == 0:
                raise ZeroDivisionError(
                    f"entry ({i},{j}) has denominator 0 at point {pt}"
                )

            row.append(numv / denv)

        rows.append(row)

    return matrix(rows)


def denominator_zero_positions(M, pt):
    """
    检查矩阵 M 在点 pt 处哪些位置分母为 0。
    返回 [(i, j, numv, denv), ...]
    """
    bad = []

    for i in range(M.nrows()):
        for j in range(M.ncols()):
            f = M[i, j]

            try:
                num = f.numerator()
                den = f.denominator()
            except AttributeError:
                num = f
                den = 1

            numv = evaluate_expr_at_point(num, pt)
            denv = evaluate_expr_at_point(den, pt)

            if denv == 0:
                bad.append((i, j, numv, denv))

    return bad


def last_row_in_span_after_specialize(M, pt, verbose=True):
    """
    直接用 rank 检查：
    代值后最后一行是否在前面各行张成空间里
    """
    M0 = specialize_matrix_at_point(M, pt)
    r_all = M0.rank()
    r_prev = M0[:-1, :].rank()
    ok = (r_all == r_prev)

    if verbose:
        print("rank(M0) =", r_all)
        print("rank(prefix) =", r_prev)
        print("last row in span ? ->", ok)

    return M0, ok


def find_point_by_slicing_strict(small, avoid_poly, Mcheck=None, trials=150, verbose=True):
    """
    在有限域上找点 pt，使得：
      1. small["polys"] = 0
      2. avoid_poly(pt) != 0
      3. 若给了 Mcheck，则额外要求：
         - Mcheck 在 pt 处没有分母爆炸
         - 且代值后最后一行真的在前缀行空间里
    """
    R = small["ring"]
    F = R.base_ring()
    polys = small["polys"]
    vars_ = R.gens()

    if len(vars_) != 3:
        raise ValueError("目前这个函数只处理 3 个变量。")

    t0, t1, t2 = vars_

    tested_slices = 0
    for a in F:
        tested_slices += 1
        if tested_slices > trials:
            break

        polys2_raw = [f(t0=a) for f in polys]

        R2 = PolynomialRing(F, [str(t1), str(t2)], order='lex')
        u1, u2 = R2.gens()

        polys2 = []
        for f in polys2_raw:
            try:
                ff = f.subs({t1: u1, t2: u2})
                polys2.append(R2(ff))
            except Exception:
                try:
                    polys2.append(R2(f))
                except Exception:
                    pass

        if len(polys2) == 0:
            continue

        J = R2.ideal(polys2)

        try:
            dimJ = J.dimension()
        except Exception:
            continue

        if dimJ != 0:
            continue

        try:
            sols = J.variety()
        except Exception:
            continue

        if not sols:
            continue

        for sol in sols:
            b = sol[u1] if u1 in sol else sol[str(u1)]
            c = sol[u2] if u2 in sol else sol[str(u2)]

            pt = {
                str(t0): a,
                str(t1): b,
                str(t2): c
            }

            # 必须避开所有分母
            if evaluate_expr_at_point(avoid_poly, pt) == 0:
                continue

            # 若给了原矩阵，再做最终真验证
            if Mcheck is not None:
                bad = denominator_zero_positions(Mcheck, pt)
                if len(bad) != 0:
                    continue

                _, ok = last_row_in_span_after_specialize(Mcheck, pt, verbose=False)
                if not ok:
                    continue

            if verbose:
                print("找到严格合法的点：", pt)
            return pt

    if verbose:
        print("没找到满足 numerators=0 且避开全部分母并通过最终验证的点。")
    return None

def find_point_by_slicing_no_chart(small, trials=120, verbose=True):
    """
    在有限域上找一个点：
    只要求满足 small["polys"] = 0
    不再检查任何 chart denominator。

    做法：
    固定 t0 = a 后，在二维里求 0 维切片；
    找到解就直接返回。
    """
    R = small["ring"]
    F = R.base_ring()
    polys = small["polys"]
    t0, t1, t2 = R.gens()

    vals = []
    k = 0
    for a in F:
        vals.append(a)
        k += 1
        if k >= trials:
            break

    for a in vals:
        polys2_raw = [f(t0=a) for f in polys]

        R2 = PolynomialRing(F, [str(t1), str(t2)], order='lex')
        u1, u2 = R2.gens()

        polys2 = []
        for f in polys2_raw:
            try:
                ff = f.subs({t1: u1, t2: u2})
                polys2.append(R2(ff))
            except Exception:
                try:
                    polys2.append(R2(f))
                except Exception:
                    pass

        if len(polys2) == 0:
            continue

        J = R2.ideal(polys2)

        try:
            dimJ = J.dimension()
        except Exception:
            continue

        if dimJ != 0:
            continue

        try:
            sols = J.variety()
        except Exception:
            continue

        if not sols:
            continue

        for sol in sols:
            b = sol[u1] if u1 in sol else sol[str(u1)]
            c = sol[u2] if u2 in sol else sol[str(u2)]

            pt = {
                str(t0): a,
                str(t1): b,
                str(t2): c
            }

            if verbose:
                print("找到满足 numerators=0 的候选点：", pt)
            return pt

    if verbose:
        print("在前", trials, "个切片里没找到候选点。")
    return None

def find_point_by_slicing_strict_debug(small, avoid_poly, Mcheck=None, trials=150, verbose=True):
    R = small["ring"]
    F = R.base_ring()
    polys = small["polys"]
    vars_ = R.gens()

    if len(vars_) != 3:
        raise ValueError("目前这个函数只处理 3 个变量。")

    t0, t1, t2 = vars_

    count_slice_total = 0
    count_zero_dim = 0
    count_candidates = 0
    count_pass_avoid = 0
    count_pass_den = 0
    count_pass_rank = 0

    for a in F:
        count_slice_total += 1
        if count_slice_total > trials:
            break

        polys2_raw = [f(t0=a) for f in polys]

        R2 = PolynomialRing(F, [str(t1), str(t2)], order='lex')
        u1, u2 = R2.gens()

        polys2 = []
        for f in polys2_raw:
            try:
                ff = f.subs({t1: u1, t2: u2})
                polys2.append(R2(ff))
            except Exception:
                try:
                    polys2.append(R2(f))
                except Exception:
                    pass

        if len(polys2) == 0:
            continue

        J = R2.ideal(polys2)

        try:
            dimJ = J.dimension()
        except Exception:
            continue

        if dimJ != 0:
            continue

        count_zero_dim += 1

        try:
            sols = J.variety()
        except Exception:
            continue

        if verbose:
            print(f"slice t0={a}: zero-dimensional, #sols={len(sols)}")

        for sol in sols:
            count_candidates += 1

            b = sol[u1] if u1 in sol else sol[str(u1)]
            c = sol[u2] if u2 in sol else sol[str(u2)]

            pt = {str(t0): a, str(t1): b, str(t2): c}

            av = evaluate_expr_at_point(avoid_poly, pt)
            if av == 0:
                if verbose:
                    print("  rejected by avoid_poly:", pt)
                continue

            count_pass_avoid += 1

            if Mcheck is not None:
                bad = denominator_zero_positions(Mcheck, pt)
                if len(bad) != 0:
                    if verbose:
                        print("  rejected by matrix denominator:", pt, "bad count =", len(bad))
                    continue

                count_pass_den += 1

                _, ok = last_row_in_span_after_specialize(Mcheck, pt, verbose=False)
                if not ok:
                    if verbose:
                        print("  rejected by final rank test:", pt)
                    continue

            count_pass_rank += 1
            print("找到严格合法的点：", pt)
            return pt

    print("\n===== debug summary =====")
    print("slices tested =", count_slice_total)
    print("zero-dimensional slices =", count_zero_dim)
    print("candidate points from varieties =", count_candidates)
    print("pass avoid_poly =", count_pass_avoid)
    print("pass matrix denominator =", count_pass_den)
    print("pass final rank =", count_pass_rank)

    return None

def test_candidate_on_original_matrix(M, pt, verbose=True):
    """
    不管 pt 是否落在 chart 分母上，
    直接代回原始矩阵 M，检查最后一行是否在前缀行空间里。
    """
    M0 = specialize_matrix_at_point(M, pt)

    r_all = M0.rank()
    r_prev = M0[:-1, :].rank()
    ok = (r_all == r_prev)

    if verbose:
        print("candidate pt =", pt)
        print("rank(M0) =", r_all)
        print("rank(prefix) =", r_prev)
        print("last row in span ? ->", ok)

    return M0, ok
def find_true_point_by_slicing_and_rank_check(small, Mcheck, trials=120, verbose=True):
    """
    先找满足 numerators=0 的候选点，
    再直接代回原始矩阵 Mcheck 检查 rank。
    不检查 chart denominator。
    """
    R = small["ring"]
    F = R.base_ring()
    polys = small["polys"]
    t0, t1, t2 = R.gens()

    vals = []
    k = 0
    for a in F:
        vals.append(a)
        k += 1
        if k >= trials:
            break

    for a in vals:
        polys2_raw = [f(t0=a) for f in polys]

        R2 = PolynomialRing(F, [str(t1), str(t2)], order='lex')
        u1, u2 = R2.gens()

        polys2 = []
        for f in polys2_raw:
            try:
                ff = f.subs({t1: u1, t2: u2})
                polys2.append(R2(ff))
            except Exception:
                try:
                    polys2.append(R2(f))
                except Exception:
                    pass

        if len(polys2) == 0:
            continue

        J = R2.ideal(polys2)

        try:
            dimJ = J.dimension()
        except Exception:
            continue

        if dimJ != 0:
            continue

        try:
            sols = J.variety()
        except Exception:
            continue

        if not sols:
            continue

        for sol in sols:
            b = sol[u1] if u1 in sol else sol[str(u1)]
            c = sol[u2] if u2 in sol else sol[str(u2)]

            pt = {
                str(t0): a,
                str(t1): b,
                str(t2): c
            }

            if verbose:
                print("testing candidate:", pt)

            try:
                M0 = specialize_matrix_at_point(Mcheck, pt)
            except Exception as e:
                if verbose:
                    print("  specialize failed:", e)
                continue

            ok = (M0.rank() == M0[:-1, :].rank())

            if verbose:
                print("  rank(M0) =", M0.rank())
                print("  rank(prefix) =", M0[:-1, :].rank())
                print("  ok =", ok)

            if ok:
                print("找到原始矩阵上的真解：", pt)
                return pt

    if verbose:
        print("没找到满足 numerators=0 且在原始矩阵上真正降 rank 的点。")
    return None

In [7]:
def _normalize_point_for_ring(pt, ring_or_gens):
    """
    把点 pt 统一成:
        { "t0": val0, "t1": val1, ... }
    支持:
        - dict: {"t0":..., "t1":...}
        - list/tuple: [..]
    """
    gens = ring_or_gens.gens() if hasattr(ring_or_gens, "gens") else tuple(ring_or_gens)

    if isinstance(pt, dict):
        out = {}
        for g in gens:
            name = str(g)
            if name in pt:
                out[name] = pt[name]
        return out

    if isinstance(pt, (list, tuple)):
        if len(pt) != len(gens):
            raise ValueError(f"需要 {len(gens)} 个参数值，但给了 {len(pt)} 个")
        return {str(gens[i]): pt[i] for i in range(len(gens))}

    raise TypeError("pt 必须是 dict / list / tuple")


def _values_from_point_for_ring(pt, ring_or_gens):
    """
    按 ring 的变量顺序返回代值列表。
    """
    gens = ring_or_gens.gens() if hasattr(ring_or_gens, "gens") else tuple(ring_or_gens)
    pt_dict = _normalize_point_for_ring(pt, gens)
    vals = []
    for g in gens:
        name = str(g)
        if name not in pt_dict:
            raise ValueError(f"点里缺少变量 {name} 的取值")
        vals.append(pt_dict[name])
    return vals


def _first_field_values(F, limit=None):
    """
    取有限域前 limit 个元素；limit=None 时取全域。
    """
    vals = []
    k = 0
    for a in F:
        vals.append(a)
        k += 1
        if limit is not None and k >= limit:
            break
    return vals


def _specialize_polys_fix_first_var(polys, vars_):
    """
    返回一个函数 specialize(a)，把 vars_[0]=a 代入后，
    把多项式列表重建到剩余变量的小环里。

    输出:
        rem_vars, specialize
    其中 specialize(a) 返回:
        - rem_vars 为空时: 常数/标量列表
        - rem_vars 非空时: 新环上的多项式列表
    """
    vars_ = list(vars_)
    v0 = vars_[0]
    rem_vars = vars_[1:]
    F = polys[0].parent().base_ring() if len(polys) > 0 else v0.parent().base_ring()

    if len(rem_vars) == 0:
        def specialize(a):
            out = []
            for f in polys:
                try:
                    out.append(f.subs({v0: a}))
                except Exception:
                    try:
                        out.append(f(v0=a))
                    except Exception:
                        out.append(f)
            return out
        return rem_vars, specialize

    R2 = PolynomialRing(F, [str(v) for v in rem_vars], order='lex')
    u = R2.gens()
    sub_map_rem = {rem_vars[i]: u[i] for i in range(len(rem_vars))}

    def specialize(a):
        out = []
        for f in polys:
            try:
                g = f.subs({v0: a})
            except Exception:
                try:
                    g = f(**{str(v0): a})
                except Exception:
                    g = f

            try:
                gg = g.subs(sub_map_rem)
                out.append(R2(gg))
            except Exception:
                try:
                    out.append(R2(g))
                except Exception:
                    pass
        return out

    return rem_vars, specialize


def _enumerate_points_by_recursive_slicing(polys, vars_, trials=150, verbose=False, prefix_pt=None):
    """
    通用递归切片：
      - 任意个 t 变量都能处理
      - 最后剩 0/1/2 个变量时直接解
      - 返回候选点 dict 的生成器

    注意：
      变量多时会变慢，trials 表示“每一层最多枚举多少个域元素”。
    """
    vars_ = list(vars_)
    prefix_pt = {} if prefix_pt is None else dict(prefix_pt)

    if len(vars_) == 0:
        ok = True
        for f in polys:
            try:
                if f != 0:
                    ok = False
                    break
            except Exception:
                if f != 0:
                    ok = False
                    break
        if ok:
            yield prefix_pt
        return

    # 只有一个变量：直接暴力验根
    if len(vars_) == 1:
        v = vars_[0]
        F = polys[0].parent().base_ring() if len(polys) > 0 else v.parent().base_ring()

        for a in _first_field_values(F, None):   # 1 维时直接全域扫
            good = True
            for f in polys:
                try:
                    val = f.subs({v: a})
                except Exception:
                    try:
                        val = f(v=a)
                    except Exception:
                        val = f
                if val != 0:
                    good = False
                    break
            if good:
                pt = dict(prefix_pt)
                pt[str(v)] = a
                yield pt
        return

    # 只剩两个变量：沿第一个变量切片，然后 variety()
    if len(vars_) == 2:
        v0, v1 = vars_
        F = polys[0].parent().base_ring() if len(polys) > 0 else v0.parent().base_ring()

        vals0 = _first_field_values(F, trials)
        for a in vals0:
            polys_raw = []
            for f in polys:
                try:
                    polys_raw.append(f.subs({v0: a}))
                except Exception:
                    try:
                        polys_raw.append(f(v0=a))
                    except Exception:
                        pass

            R2 = PolynomialRing(F, [str(v1)], order='lex')
            u1 = R2.gen()

            polys2 = []
            for f in polys_raw:
                try:
                    ff = f.subs({v1: u1})
                    polys2.append(R2(ff))
                except Exception:
                    try:
                        polys2.append(R2(f))
                    except Exception:
                        pass

            if len(polys2) == 0:
                continue

            J = R2.ideal(polys2)
            try:
                dimJ = J.dimension()
            except Exception:
                continue

            if dimJ != 0:
                # 维数不是 0，就退化成 1 维暴力
                for b in _first_field_values(F, None):
                    good = True
                    for f in polys2:
                        try:
                            if f(u1=b) != 0:
                                good = False
                                break
                        except Exception:
                            try:
                                if f.subs({u1: b}) != 0:
                                    good = False
                                    break
                            except Exception:
                                good = False
                                break
                    if good:
                        pt = dict(prefix_pt)
                        pt[str(v0)] = a
                        pt[str(v1)] = b
                        yield pt
                continue

            try:
                sols = J.variety()
            except Exception:
                continue

            for sol in sols:
                b = sol[u1] if u1 in sol else sol[str(u1)]
                pt = dict(prefix_pt)
                pt[str(v0)] = a
                pt[str(v1)] = b
                yield pt
        return

    # 变量 >= 3：继续递归切第一维
    F = polys[0].parent().base_ring()
    rem_vars, specialize = _specialize_polys_fix_first_var(polys, vars_)
    v0 = vars_[0]

    for a in _first_field_values(F, trials):
        polys2 = specialize(a)
        if len(polys2) == 0:
            continue

        pt2 = dict(prefix_pt)
        pt2[str(v0)] = a

        for sol in _enumerate_points_by_recursive_slicing(
            polys2,
            rem_vars,
            trials=trials,
            verbose=verbose,
            prefix_pt=pt2
        ):
            yield sol


def evaluate_expr_at_point(expr, pt):
    """
    在点 pt 处对 expr 求值。
    pt 可为:
        - {"t0": a, "t1": b, ...}
        - [a, b, ...] / (a, b, ...)
    """
    try:
        P = expr.parent()
        gens = P.gens()
    except Exception:
        return expr

    if len(gens) == 0:
        return expr

    pt_dict = _normalize_point_for_ring(pt, gens)

    vals = []
    used_any = False
    for g in gens:
        name = str(g)
        if name in pt_dict:
            vals.append(pt_dict[name])
            used_any = True
        else:
            vals.append(g)

    if not used_any:
        return expr

    try:
        return expr(*vals)
    except TypeError:
        try:
            return expr.subs({gens[i]: vals[i] for i in range(len(gens))})
        except Exception:
            return expr


def last_row_in_span_after_specialize(M, vals, verbose=True):
    """
    vals 可为:
        - 按顺序的 list/tuple
        - dict, 例如 {"t0":..., "t1":..., ...}

    返回代值后最后一行是否在前面行张成空间里
    """
    S = M.base_ring()
    F = S.base_ring()
    ts = S.gens()

    vals_list = _values_from_point_for_ring(vals, ts)
    sub_dict = {ts[i]: vals_list[i] for i in range(len(ts))}
    M0 = matrix(F, M.substitute(sub_dict))

    r_all = M0.rank()
    r_prev = M0[:-1].rank()

    if verbose:
        print("代入值 =", vals_list)
        print("rank(M0) =", r_all)
        print("rank(M0[:-1]) =", r_prev)
        print("最后一行是否在前面行空间里：", r_all == r_prev)

    return M0, (r_all == r_prev)


def find_point_by_slicing_strict(small, avoid_poly, Mcheck=None, trials=150, verbose=True):
    """
    在有限域上找点 pt，使得：
      1. small["polys"] = 0
      2. avoid_poly(pt) != 0
      3. 若给了 Mcheck，则额外要求：
         - Mcheck 在 pt 处没有分母爆炸
         - 且代值后最后一行真的在前缀行空间里

    现在支持任意个 t 变量。
    """
    R = small["ring"]
    polys = small["polys"]
    vars_ = list(R.gens())

    for pt in _enumerate_points_by_recursive_slicing(polys, vars_, trials=trials, verbose=verbose):
        if evaluate_expr_at_point(avoid_poly, pt) == 0:
            continue

        if Mcheck is not None:
            bad = denominator_zero_positions(Mcheck, pt)
            if len(bad) != 0:
                continue

            _, ok = last_row_in_span_after_specialize(Mcheck, pt, verbose=False)
            if not ok:
                continue

        if verbose:
            print("找到严格合法的点：", pt)
        return pt

    if verbose:
        print("没找到满足 numerators=0 且避开全部分母并通过最终验证的点。")
    return None


def find_point_by_slicing_no_chart(small, trials=120, verbose=True):
    """
    在有限域上找一个点：
    只要求满足 small["polys"] = 0
    不检查 chart denominator。
    现在支持任意个 t 变量。
    """
    R = small["ring"]
    polys = small["polys"]
    vars_ = list(R.gens())

    for pt in _enumerate_points_by_recursive_slicing(polys, vars_, trials=trials, verbose=verbose):
        if verbose:
            print("找到满足 numerators=0 的候选点：", pt)
        return pt

    if verbose:
        print("在递归切片搜索里没找到候选点。")
    return None


def find_point_by_slicing_strict_debug(small, avoid_poly, Mcheck=None, trials=150, verbose=True):
    """
    debug 版：支持任意个 t 变量。
    """
    R = small["ring"]
    polys = small["polys"]
    vars_ = list(R.gens())

    count_candidates = 0
    count_pass_avoid = 0
    count_pass_den = 0
    count_pass_rank = 0

    for pt in _enumerate_points_by_recursive_slicing(polys, vars_, trials=trials, verbose=verbose):
        count_candidates += 1

        av = evaluate_expr_at_point(avoid_poly, pt)
        if av == 0:
            if verbose:
                print("rejected by avoid_poly:", pt)
            continue

        count_pass_avoid += 1

        if Mcheck is not None:
            bad = denominator_zero_positions(Mcheck, pt)
            if len(bad) != 0:
                if verbose:
                    print("rejected by matrix denominator:", pt, "bad count =", len(bad))
                continue

            count_pass_den += 1

            _, ok = last_row_in_span_after_specialize(Mcheck, pt, verbose=False)
            if not ok:
                if verbose:
                    print("rejected by final rank test:", pt)
                continue

        count_pass_rank += 1
        print("找到严格合法的点：", pt)
        return pt

    print("\n===== debug summary =====")
    print("candidate points from recursive slicing =", count_candidates)
    print("pass avoid_poly =", count_pass_avoid)
    print("pass matrix denominator =", count_pass_den)
    print("pass final rank =", count_pass_rank)

    return None


def test_candidate_on_original_matrix(M, pt, verbose=True):
    """
    不管 pt 是否落在 chart 分母上，
    直接代回原始矩阵 M，检查最后一行是否在前缀行空间里。
    pt 可为 dict / list / tuple
    """
    M0 = specialize_matrix_at_point(M, pt)

    r_all = M0.rank()
    r_prev = M0[:-1, :].rank()
    ok = (r_all == r_prev)

    if verbose:
        print("candidate pt =", _normalize_point_for_ring(pt, M.base_ring()))
        print("rank(M0) =", r_all)
        print("rank(prefix) =", r_prev)
        print("last row in span ? ->", ok)

    return M0, ok


def find_true_point_by_slicing_and_rank_check(small, Mcheck, trials=120, verbose=True):
    """
    先找满足 numerators=0 的候选点，
    再直接代回原始矩阵 Mcheck 检查 rank。
    不检查 chart denominator。
    现在支持任意个 t 变量。
    """
    R = small["ring"]
    polys = small["polys"]
    vars_ = list(R.gens())

    for pt in _enumerate_points_by_recursive_slicing(polys, vars_, trials=trials, verbose=verbose):
        if verbose:
            print("testing candidate:", pt)

        try:
            M0 = specialize_matrix_at_point(Mcheck, pt)
        except Exception as e:
            if verbose:
                print("  specialize failed:", e)
            continue

        ok = (M0.rank() == M0[:-1, :].rank())

        if verbose:
            print("  rank(M0) =", M0.rank())
            print("  rank(prefix) =", M0[:-1, :].rank())
            print("  ok =", ok)

        if ok:
            print("找到原始矩阵上的真解：", pt)
            return pt

    if verbose:
        print("没找到满足 numerators=0 且在原始矩阵上真正降 rank 的点。")
    return None

def analyze_t_ideal_with_avoid(small, avoid_poly, verbose=True, compute_variety=False):
    """
    在扩环 F[t..., s] 中分析:
        small["polys"] = 0
        s * avoid_poly - 1 = 0

    这等价于只保留 avoid_poly != 0 的解。
    自动处理 avoid_poly 与 small["ring"] parent 不一致的问题。
    """

    R = small["ring"]
    polys = [f for f in small["polys"] if f != 0]
    t_vars = list(R.gens())
    F = R.base_ring()

    # 扩一个新变量 s
    Rext = PolynomialRing(F, [str(v) for v in t_vars] + ["s"], order='lex')
    gens_ext = Rext.gens()
    u_t = gens_ext[:-1]
    s = gens_ext[-1]

    # 先搬 small["polys"]
    sub_map_small = {t_vars[i]: u_t[i] for i in range(len(t_vars))}

    polys_ext = []
    for f in polys:
        g = f.subs(sub_map_small)
        polys_ext.append(Rext(g))

    # 再单独搬 avoid_poly：必须按 avoid_poly 自己的 parent 来替换
    Pavoid = avoid_poly.parent()
    avoid_gens = list(Pavoid.gens())

    name_to_ut = {str(t_vars[i]): u_t[i] for i in range(len(t_vars))}
    avoid_sub_map = {}

    for v in avoid_gens:
        name = str(v)
        if name in name_to_ut:
            avoid_sub_map[v] = name_to_ut[name]

    if verbose:
        print("avoid_poly parent =", Pavoid)
        print("small ring =", R)
        print("Rext =", Rext)
        print("avoid_sub_map =", avoid_sub_map)

    avoid_tmp = avoid_poly.subs(avoid_sub_map)
    avoid_ext = Rext(avoid_tmp)

    polys_ext.append(s * avoid_ext - 1)

    J = Rext.ideal(polys_ext)

    if verbose:
        print("number of generators =", len(polys_ext))
        print("last generator =")
        print(polys_ext[-1])

    try:
        is_one = J.is_one()
    except Exception:
        is_one = False

    if verbose:
        print("J.is_one() =", is_one)

    gb = None
    dimJ = None
    sols = None

    if not is_one:
        try:
            gb = J.groebner_basis()
            if verbose:
                print("\nGroebner basis computed.")
                if len(gb) <= 20:
                    for g in gb:
                        print(g)
                else:
                    print("GB size =", len(gb))
        except Exception as e:
            if verbose:
                print("\nGroebner basis failed:")
                print(e)

        try:
            dimJ = J.dimension()
            if verbose:
                print("dimension =", dimJ)
        except Exception as e:
            if verbose:
                print("dimension() failed:", e)

        if compute_variety and dimJ == 0:
            try:
                sols = J.variety()
                if verbose:
                    print("number of solutions =", len(sols))
                    if len(sols) <= 10:
                        for sol in sols:
                            print(sol)
            except Exception as e:
                if verbose:
                    print("variety() failed:", e)

    return {
        "ring": Rext,
        "ideal": J,
        "groebner_basis": gb,
        "dimension": dimJ,
        "is_one": is_one,
        "solutions": sols,
        "generators": polys_ext,
        "avoid_ext": avoid_ext,
    }

In [8]:
import random

def _random_field_element(F):
    try:
        return F.random_element()
    except Exception:
        vals = list(F)
        return random.choice(vals)

def _substitute_by_name(expr, target_ring):
    P = expr.parent()
    src_gens = list(P.gens())
    tgt_gens = list(target_ring.gens())
    name_to_tgt = {str(g): g for g in tgt_gens}

    sub_map = {}
    for v in src_gens:
        sv = str(v)
        if sv in name_to_tgt:
            sub_map[v] = name_to_tgt[sv]

    try:
        out = expr.subs(sub_map)
    except Exception:
        out = expr

    return target_ring(out)

def _specialize_expr_by_name(expr, assignments):
    P = expr.parent()
    src_gens = list(P.gens())

    sub_map = {}
    for v in src_gens:
        sv = str(v)
        if sv in assignments:
            sub_map[v] = assignments[sv]

    try:
        return expr.subs(sub_map)
    except Exception:
        return expr

def _build_slice_ring_and_system(small, avoid_poly, fixed_assignments, verbose=False):
    R = small["ring"]
    polys = [f for f in small["polys"] if f != 0]
    all_t = list(R.gens())
    F = R.base_ring()

    remaining_t = [t for t in all_t if str(t) not in fixed_assignments]

    polys_spec_raw = []
    for f in polys:
        g = _specialize_expr_by_name(f, fixed_assignments)
        if g != 0:
            polys_spec_raw.append(g)

    avoid_spec_raw = _specialize_expr_by_name(avoid_poly, fixed_assignments)

    Rslice = PolynomialRing(F, [str(v) for v in remaining_t] + ["s"], order='lex')
    gens_slice = list(Rslice.gens())
    s = gens_slice[-1]

    polys_slice = []
    for g in polys_spec_raw:
        gg = _substitute_by_name(g, Rslice)
        if gg != 0:
            polys_slice.append(gg)

    avoid_slice = _substitute_by_name(avoid_spec_raw, Rslice)
    polys_slice.append(s * avoid_slice - 1)

    if verbose:
        print("fixed_assignments =", fixed_assignments)
        print("remaining_t =", remaining_t)
        print("Rslice =", Rslice)
        print("num generators =", len(polys_slice))
        print("last generator =", polys_slice[-1])

    return {
        "ring": Rslice,
        "generators": polys_slice,
        "remaining_t": remaining_t,
        "fixed_assignments": fixed_assignments,
        "avoid_slice": avoid_slice,
    }

def _analyze_slice_system(slice_data, compute_gb=True, compute_variety_if_zero_dim=True, verbose=False):
    Rslice = slice_data["ring"]
    gens = slice_data["generators"]
    J = Rslice.ideal(gens)

    result = {
        "ideal": J,
        "classification": None,
        "dimension": None,
        "is_one": None,
        "gb": None,
        "solutions": None,
        "error": None,
    }

    try:
        is_one = J.is_one()
    except Exception as e:
        is_one = None
        result["error"] = f"is_one failed: {e}"

    result["is_one"] = is_one

    if is_one is True:
        result["classification"] = "one_ideal"
        return result

    gb = None
    if compute_gb:
        try:
            gb = J.groebner_basis()
            result["gb"] = gb
        except Exception as e:
            result["classification"] = "gb_failed"
            result["error"] = f"groebner_basis failed: {e}"
            return result

    try:
        dimJ = J.dimension()
        result["dimension"] = dimJ
    except Exception as e:
        result["classification"] = "gb_failed"
        old = result["error"]
        result["error"] = (old + " ; " if old else "") + f"dimension failed: {e}"
        return result

    if dimJ == 0:
        if compute_variety_if_zero_dim:
            try:
                sols = J.variety()
                result["solutions"] = sols
                if len(sols) == 0:
                    result["classification"] = "zero_dim_no_solution"
                else:
                    result["classification"] = "zero_dim_has_solution"
            except Exception as e:
                result["classification"] = "variety_failed"
                old = result["error"]
                result["error"] = (old + " ; " if old else "") + f"variety failed: {e}"
        else:
            result["classification"] = "zero_dim"
    else:
        result["classification"] = "positive_dim"

    return result

def _merge_full_point(all_t, fixed_assignments, sol):
    """
    把切片解 sol 与 fixed_assignments 合并成原始 ring 上完整的 t 点。
    自动去掉 s。
    返回:
        {"t0":..., "t1":..., ...}
    """
    full = dict(fixed_assignments)

    for k, v in sol.items():
        sk = str(k)
        if sk == "s":
            continue
        full[sk] = v

    # 按 all_t 顺序整理
    ordered = {}
    for t in all_t:
        st = str(t)
        if st in full:
            ordered[st] = full[st]
    return ordered

def random_slice_diagnose(
    small,
    avoid_poly,
    num_fix=4,
    num_trials=20,
    seed=None,
    compute_gb=True,
    compute_variety_if_zero_dim=True,
    verbose_each=False,
    stop_on_first_good=False,
):
    """
    随机固定 num_fix 个 t，分析低维切片。
    现在会自动保存找到的完整参数点:
        summary["good_points"]
        summary["first_good_point"]
    """
    if seed is not None:
        random.seed(seed)

    R = small["ring"]
    all_t = list(R.gens())
    F = R.base_ring()

    if num_fix < 0 or num_fix > len(all_t):
        raise ValueError(f"num_fix 必须在 0 到 {len(all_t)} 之间")

    summary = {
        "num_trials": num_trials,
        "num_fix": num_fix,
        "all_t": all_t,
        "counts": {
            "one_ideal": 0,
            "zero_dim_no_solution": 0,
            "zero_dim_has_solution": 0,
            "positive_dim": 0,
            "gb_failed": 0,
            "variety_failed": 0,
            "zero_dim": 0,
            "other": 0,
        },
        "trials": [],
        "good_points": [],
        "first_good_point": None,
    }

    for trial_id in range(num_trials):
        chosen = random.sample(all_t, num_fix)
        fixed_assignments = {str(t): _random_field_element(F) for t in chosen}

        try:
            slice_data = _build_slice_ring_and_system(
                small,
                avoid_poly,
                fixed_assignments,
                verbose=verbose_each
            )
            analysis = _analyze_slice_system(
                slice_data,
                compute_gb=compute_gb,
                compute_variety_if_zero_dim=compute_variety_if_zero_dim,
                verbose=verbose_each
            )
        except Exception as e:
            slice_data = None
            analysis = {
                "classification": "other",
                "dimension": None,
                "is_one": None,
                "gb": None,
                "solutions": None,
                "error": str(e),
            }

        full_points = []
        if analysis.get("classification") == "zero_dim_has_solution" and analysis.get("solutions") is not None:
            for sol in analysis["solutions"]:
                full_pt = _merge_full_point(all_t, fixed_assignments, sol)
                full_points.append(full_pt)
                summary["good_points"].append(full_pt)
                if summary["first_good_point"] is None:
                    summary["first_good_point"] = full_pt

        record = {
            "trial_id": trial_id,
            "fixed_assignments": fixed_assignments,
            "remaining_t": None if slice_data is None else slice_data["remaining_t"],
            "classification": analysis["classification"],
            "dimension": analysis.get("dimension", None),
            "is_one": analysis.get("is_one", None),
            "num_solutions": None if analysis.get("solutions", None) is None else len(analysis["solutions"]),
            "solutions": analysis.get("solutions", None),
            "full_points": full_points,
            "error": analysis.get("error", None),
        }

        cls = record["classification"]
        if cls in summary["counts"]:
            summary["counts"][cls] += 1
        else:
            summary["counts"]["other"] += 1

        summary["trials"].append(record)

        msg = f"[trial {trial_id}] class={cls}, dim={record['dimension']}, fixed={fixed_assignments}"
        if len(full_points) > 0:
            msg += f"  <-- found {len(full_points)} full point(s)"
        print(msg)

        if stop_on_first_good and cls == "zero_dim_has_solution":
            print("发现 zero_dim_has_solution，提前停止。")
            break

    print("\n===== summary =====")
    for k, v in summary["counts"].items():
        if v:
            print(f"{k}: {v}")

    print("total good full points saved =", len(summary["good_points"]))
    if summary["first_good_point"] is not None:
        print("first_good_point =", summary["first_good_point"])

    return summary

def print_good_trials(summary, max_show=10):
    good_classes = {"zero_dim_has_solution", "zero_dim_no_solution", "one_ideal"}
    shown = 0
    for rec in summary["trials"]:
        if rec["classification"] in good_classes:
            print("\n------------------------------")
            print("trial_id =", rec["trial_id"])
            print("classification =", rec["classification"])
            print("dimension =", rec["dimension"])
            print("fixed_assignments =", rec["fixed_assignments"])
            print("remaining_t =", rec["remaining_t"])
            print("num_solutions =", rec["num_solutions"])
            if rec["solutions"] is not None and rec["num_solutions"] is not None and rec["num_solutions"] <= 5:
                print("solutions =", rec["solutions"])
            if rec["full_points"] is not None and len(rec["full_points"]) > 0:
                print("full_points =", rec["full_points"])
            if rec["error"] is not None:
                print("error =", rec["error"])
            shown += 1
            if shown >= max_show:
                break

def restrict_f3_to_line_via_x2(f1, f2, f3, x0, x1, x2, verbose=True):
    """
    输入:
        f1, f2, f3 : Sage 多项式
        x0, x1, x2 : 你想消去/保留的三个变量
                     这里默认把 x2 当作交线参数 z
    输出:
        一个字典，包含:
          - det      : 解 x0,x1 时的分母
          - x0_on_L  : 交线上 x0 的表达式
          - x1_on_L  : 交线上 x1 的表达式
          - g        : f3 限制到 V(f1,f2) 后关于 x2 的多项式
          - coeffs   : {0:c0, 1:c1, 2:c2, 3:c3}
    """

    R = f1.parent()
    F = R.fraction_field()

    # 提升到分式域，避免除法问题
    f1F = F(f1)
    f2F = F(f2)
    f3F = F(f3)

    # 提取关于 x0,x1,x2 的一次系数与常数项
    def affine_parts(f):
        c0 = f.subs({x0:0, x1:0, x2:0})
        a0 = f.coefficient(x0)
        a1 = f.coefficient(x1)
        a2 = f.coefficient(x2)
        # 检查确实是仿射线性
        rem = f - (a0*x0 + a1*x1 + a2*x2 + c0)
        if rem != 0:
            print("Warning: 这个多项式不是关于 x0,x1,x2 的仿射一次式:")
            print(rem)
        return a0, a1, a2, c0

    # a00, a01, a02, b0 = affine_parts(f1F)
    # a10, a11, a12, b1 = affine_parts(f2F)
    a00, a01, a02, b0 = affine_parts(f1)
    a10, a11, a12, b1 = affine_parts(f2)

    # 解方程:
    # a00*x0 + a01*x1 + a02*x2 + b0 = 0
    # a10*x0 + a11*x1 + a12*x2 + b1 = 0
    #
    # 把 x2 视为自由变量，解 x0,x1
    det = a00*a11 - a01*a10

    if det == 0:
        raise ValueError("用于解 x0,x1 的 2x2 子式恒等为 0；需要换一个自由变量或换一个子式。")

    # rhs0 = -(a02*x2 + b0)
    # rhs1 = -(a12*x2 + b1)

    # x0_on_L = ( rhs0*a11 - a01*rhs1 ) / det
    # x1_on_L = ( a00*rhs1 - rhs0*a10 ) / det

    # # 代回 f3
    # g = f3F.subs({
    #     x0: x0_on_L,
    #     x1: x1_on_L
    # }).expand()
    rhs0 = F(-(a02*x2 + b0))
    rhs1 = F(-(a12*x2 + b1))
    det  = F(det)

    x0_on_L = (rhs0*a11 - a01*rhs1) / det
    x1_on_L = (a00*rhs1 - rhs0*a10) / det

    g = F(f3).subs({
        x0: x0_on_L,
        x1: x1_on_L
    })
    # 这时 g 应该只依赖 x2（以及 t 参数）
    # 抽取 x2^0, x2^1, x2^2, x2^3 的系数
    # coeffs = {}
    # for k in range(4):
    #     coeffs[k] = g.coefficient(x2, k)
    num = g.numerator()
    den = g.denominator()
    coeffs = {}
    for k in range(4):
        coeffs[k] = num.coefficient(x2^k)

    if verbose:
        print("="*70)
        print("det =")
        print(det)
        print("="*70)
        print("x0 on V(f1,f2) =")
        print(x0_on_L)
        print("="*70)
        print("x1 on V(f1,f2) =")
        print(x1_on_L)
        print("="*70)
        print("g(x2) = f3|_{V(f1,f2)} =")
        print(g)
        print("="*70)
        print("coefficients of 1, x2, x2^2, x2^3:")
        for k in range(4):
            print(f"coeff x2^{k} =")
            print(coeffs[k])
            print("-"*50)

    return {
        "det": det,
        "x0_on_L": x0_on_L,
        "x1_on_L": x1_on_L,
        "g": g,
        "num": num,
        "den": den,
        "coeffs": coeffs,
    }

from itertools import product

def search_point_via_left_kernel_slice(M, fixed_pt, free_vars, value_pool, verbose=True):
    """
    M: 原矩阵，比如 Mcrit 或 Ms[0]
    fixed_pt: dict，例如 {"t3":1, "t4":7, ...}，固定大部分 t
    free_vars: 剩下自由变量，例如 [t0, t1, t2]
    value_pool: 搜索取值，例如 range(20) 或 finite field 元素列表

    返回:
        {
            "kernel_dim": ...,
            "good_basis_vectors": [...],   # 最后一个坐标不恒为0的左核基向量
            "point": {...} or None         # 找到的完整 t 点
        }
    """

    # 1. 先固定大部分 t
    M_sub = M.substitute(fixed_pt)

    # 2. 在分式域上算左核
    S = M_sub.base_ring()
    Kfrac = S.fraction_field()
    Mf = matrix(Kfrac, M_sub)

    K = Mf.left_kernel()
    basis = K.basis()

    if verbose:
        print("left kernel dim =", K.dimension())
        print("basis size =", len(basis))

    # 3. 只保留“最后一个坐标不恒等于0”的左核向量
    good_basis = []
    for v in basis:
        if v[-1] != 0:
            good_basis.append(v)

    if verbose:
        print("basis vectors with nonzero last entry =", len(good_basis))
        for i, v in enumerate(good_basis[:5]):
            print(f"\n[good basis {i}]")
            print(v)
            print("last entry =", v[-1])

    if len(good_basis) == 0:
        return {
            "kernel_dim": K.dimension(),
            "good_basis_vectors": [],
            "point": None
        }

    # 4. 搜自由变量取值
    for vals in product(value_pool, repeat=len(free_vars)):
        pt = dict(fixed_pt)
        for x, a in zip(free_vars, vals):
            pt[str(x)] = a

        # 检查是否有某个左核向量在该点合法且最后坐标非0
        for v in good_basis:
            ok = True
            eval_v = []

            for entry in v:
                den = denominator_part(entry)
                denv = evaluate_expr_at_point(den, pt)
                if denv == 0:
                    ok = False
                    break
                eval_v.append(evaluate_expr_at_point(entry, pt))

            if not ok:
                continue

            if eval_v[-1] == 0:
                continue

            # 5. 最终直接检查 rank
            try:
                M0, rank_ok = last_row_in_span_after_specialize(M, pt, verbose=False)
            except Exception:
                continue

            if rank_ok:
                if verbose:
                    print("\n找到点：", pt)
                    print("specialized kernel vector =", eval_v)
                    print("last entry =", eval_v[-1])
                    print("rank check = True")
                return {
                    "kernel_dim": K.dimension(),
                    "good_basis_vectors": good_basis,
                    "point": pt
                }

    if verbose:
        print("没找到满足条件的点。")

    return {
        "kernel_dim": K.dimension(),
        "good_basis_vectors": good_basis,
        "point": None
    }

In [34]:
from sage.all import *
import random

class PoseidonSage:
    def __init__(self, t=5, rate=4, capacity=1, s=1, p=2130706433, alpha=3, n_rounds=30, seed=None):
        """####4294967311
        Poseidon toy model in Sage
        """
        assert rate + capacity == t, "rate + capacity must equal t"

        self.t = t
        self.rate = rate
        self.capacity = capacity
        self.s = s
        self.p = p
        self.alpha = alpha
        self.n_rounds = n_rounds
        self.seed = seed
        self.F = GF(p)

        if seed is not None:
            random.seed(seed)

        # round constants and MDS matrix
        self.round_constants = self._generate_round_constants()
        self.mds_matrix = self._generate_cauchy_mds()

    # ------------------ Round constants ------------------
    def _generate_round_constants(self):
        RC = []
        for _ in range(self.n_rounds):
            RC.append(vector(self.F, [self.F(random.randint(0, self.p-1)) for _ in range(self.t)]))
        return RC

    # ------------------ Cauchy MDS matrix ------------------
    def _generate_cauchy_mds(self):
        elems = random.sample(range(self.p), 2*self.t)
        x = elems[:self.t]
        y = elems[self.t:]
        M = matrix(self.F, self.t, self.t)
        for i in range(self.t):
            for j in range(self.t):
                diff = x[i] - y[j]
                assert diff != 0, "Zero denominator in Cauchy construction"
                M[i,j] = self.F(1)/self.F(diff)
        return M

    # ------------------ S-box ------------------
    def sbox(self, x):
        return x ** self.alpha

    def external_sbox_layer(self, state):
        return vector(self.F, [self.sbox(x) for x in state])

    def internal_sbox_layer(self, state):
        state = vector(state)
        for i in range(self.s):
            state[i] = self.sbox(state[i])
        return state

    def inv_sbox(self, x):
        alpha_inv = inverse_mod(self.alpha, self.p - 1)
        return x ** alpha_inv

    def inv_internal_sbox_layer(self, state):
        state = vector(state)
        for i in range(self.s):
            state[i] = self.inv_sbox(state[i])
        return state
    
    def inv_mds_layer(self, state):
        return self.mds_matrix.inverse() * state

    def sub_round_constants(self, state, round_idx):
        return state - self.round_constants[round_idx]
    # def sub_round_constants(self, state, round_idx):
    #     R = state[0].parent()
    #     rc = vector(R, [R(c) for c in self.round_constants[round_idx]])
    #     return state - rc

    def inv_internal_round(self, state, round_idx):
        state = self.inv_mds_layer(state)
        state = self.inv_internal_sbox_layer(state)
        state = self.sub_round_constants(state, round_idx)
        return state
    # ------------------ Linear layer ------------------
    def mds_layer(self, state):
        return self.mds_matrix * state

    # ------------------ Round functions ------------------
    def add_round_constants(self, state, round_idx):
        return state + self.round_constants[round_idx]

    def external_round(self, state, round_idx):
        state = self.add_round_constants(state, round_idx)
        state = self.external_sbox_layer(state)
        state = self.mds_layer(state)
        return state

    def internal_round(self, state, round_idx):
        state = self.add_round_constants(state, round_idx)
        state = self.internal_sbox_layer(state)
        state = self.mds_layer(state)
        return state

    # ------------------ Kernel / Subspace search ------------------
    def kernel_mod_p(self, M):
        """
        Compute kernel of M over GF(p)
        Return as a matrix whose columns are basis vectors
        """
        return M.right_kernel().matrix().transpose()

    def compute_mds_invariant_subspaces_sage(self):
        p = self.p
        s = self.s
        c = 1 #############这里要改
        Fp = GF(p)

        # MDS matrix mod p
        M = Matrix(ZZ, self.mds_matrix).apply_map(lambda x: x % p).change_ring(Fp)

        B = M[:s, s:]##############这里有容量c，所以计算子空间的时候必须摒弃最后一位
        D = M[s:, s:]
        tail_dim = self.t - s
        free_dim = tail_dim - c

        U = matrix(Fp, tail_dim, free_dim)
        for i in range(free_dim):
            U[i, i] = 1
        def intersect_column_spaces(A, B):
            # A, B 的列空间取交；返回交空间的一组列基
            if A.ncols() == 0 or B.ncols() == 0:
                return matrix(Fp, A.nrows(), 0)

            Z = block_matrix(Fp, 1, 2, [A, -B])
            K = Z.right_kernel().basis()
            if len(K) == 0:
                return matrix(Fp, A.nrows(), 0)

            cols = []
            a_cols = A.ncols()
            for z in K:
                u = vector(Fp, z[:a_cols])
                cols.append(A * u)

            # 这里必须按“列向量”方式构造
            Wint = matrix(Fp, cols).transpose()

            # 去线性相关，返回列基
            return Wint.column_space().basis_matrix().transpose()

        D_inv = D.inverse()

        # 初始不变子空间 W：n × k 矩阵（列空间）
        W = self.kernel_mod_p(B).change_ring(Fp)
        W = intersect_column_spaces(W, U)
        W_list = [W]

        while W.ncols() > 0:
            # W2 = D^{-1} W
            W2 = D_inv * W

            # Z = [ W | -W2 ]
            Z = W.augment(-W2)

            # 求核
            ns = self.kernel_mod_p(Z).change_ring(Fp)
            if ns.ncols() == 0:
                break

            # W 的列向量空间
            n = W.nrows()
            k_old = W.ncols()

            # 把 ns 的前 k_old 行当作系数
            new_vectors = []
            for i in range(ns.ncols()):
                coeffs = vector(Fp, [ns[j, i] for j in range(W.ncols())])
                v = W * coeffs          # v 是 n×1 矩阵
                new_vectors.append(v)  # 转成 vector

            if not new_vectors:
                break

            # 重新组装成 n × k_new 的矩阵
            W = matrix(Fp, new_vectors).transpose()
            W = intersect_column_spaces(W, U)
            W_list.append(W)

        return W_list

    def show_invariant_subspace_dynamics(self, W_list):
        p = self.p
        s = self.s
        M = Matrix(ZZ, self.mds_matrix).apply_map(lambda x: x % p)
        n = M.nrows()

        for idx, W in enumerate(W_list, start=1):
            print(f"\n{'='*16} W_{idx} {'='*16}")
            print(f"Dimension: {W.ncols()}")
            print("Basis (columns):")
            print(W)

            # padding s行零在上面
            Z = zero_matrix(ZZ, s, W.ncols())
            V = Z.stack(W).apply_map(lambda x: x % p)

            print("\nInitial vectors after padding zeros:")
            print(V)

            # 连续左乘 M，次数等于 idx
            cur = V
            for step in range(1, idx + 1):
                cur = (M * cur).apply_map(lambda x: x % p)
                print(f"\nAfter multiplying M^{step}:")
                print(cur)


        # ------------------ CICO equations over GF(p) ------------------
    
    def make_symbolic_X(self, k):
        """
        返回 (R, X)
        R: PolynomialRing
        X: k维多项式向量
        """
        Fp = self.F
        R = PolynomialRing(Fp, [f"x{i}" for i in range(k)])
        X = vector(R, R.gens())
        return R, X
    
    def make_symbolic_X_with_t(self, k, num_t):
        Fp = self.F

        x_vars = [f"x{i}" for i in range(k)]
        t_vars = [f"t{i}" for i in range(num_t)]

        R = PolynomialRing(Fp, x_vars + t_vars)

        gens = R.gens()

        X = vector(R, gens[:k])
        T = vector(R, gens[k:k+num_t])

        return R, X, T

    def initial_state_from_W(self, W, X):
        """
        state = [0,...,0 || W * X]
        返回多项式向量（长度 t）
        """
        R = X[0].parent()
        s = self.s

        lin_part = list(W.change_ring(R) * X)   # 这是 list，不是 vector
        state_list = [R.zero()] * s + lin_part

        return vector(R, state_list)
    
    def rp_round_polynomial(self, state, round_index):
        """
        一轮 internal round（r_p）
        state: Polynomial vector
        """
        R = state[0].parent()
        Fp = self.F
        s  = self.s

        # 加轮常量
        rc = vector(R, [R(c) for c in self.round_constants[round_index]])
        state = state + rc

        # internal S-box
        state = vector(R, state)
        for i in range(s):
            state[i] = state[i] ** self.alpha

        # MDS
        M = self.mds_matrix.change_ring(R)
        state = M * state

        return state
    def rf_round_polynomial(self, state, round_index):
        """
        一轮 external round（r_f）
        """
        R = state[0].parent()

        # 加轮常量
        rc = vector(R, [R(c) for c in self.round_constants[round_index]])
        state = state + rc

        # external S-box
        state = vector(R, [x ** self.alpha for x in state])

        # MDS
        M = self.mds_matrix.change_ring(R)
        state = M * state

        return state
    # def generate_cico_equations_modp(self, W, constants, rounds, round_type="internal"):
    def generate_cico_equations_modp(self, W, constants, rounds, exter1, inter, exter2):
        """
        round_type: "internal" or "external"
        """
        k = W.ncols()
        R, X = self.make_symbolic_X(k)

        # 初始状态
        state = self.initial_state_from_W(W, X)

        assert exter1 + inter + exter2 == rounds
        round_idx = 0

        # 轮函数迭代#############注意这里的轮常数序列是错的
        for _ in range(exter1):
            state = self.rf_round_polynomial(state, round_idx)
            round_idx += 1
        for _ in range(inter):
            state = self.rp_round_polynomial(state, round_idx)
            round_idx += 1
        for _ in range(exter2):
            state = self.rf_round_polynomial(state, round_idx)
            round_idx += 1

        # for r in range(rounds):
        #     if round_type == "internal":
        #         state = self.rp_round_polynomial(state, r)
        #     else:
        #         state = self.rf_round_polynomial(state, r)

        # 构造 CICO 方程
        const_vec = vector(R, [R(c) for c in constants])
        equations = list(state - const_vec)

        return equations, R, X

    def generate_cico_equations_withnosubspace(self, constants, rounds, exter1, inter, exter2):
        """
        round_type: "internal" or "external"
        """
        k = self.t
        R, X = self.make_symbolic_X(k)
        W = identity_matrix(self.F, self.t)
        # lift 到多项式环
        state = W.change_ring(R) * X

        assert exter1 + inter + exter2 == rounds
        round_idx = 0

        # 轮函数迭代#############注意这里的轮常数序列是错的
        for _ in range(exter1):
            state = self.rf_round_polynomial(state, round_idx)
            round_idx += 1
        for _ in range(inter):
            state = self.rp_round_polynomial(state, round_idx)
            round_idx += 1
        for _ in range(exter2):
            state = self.rf_round_polynomial(state, round_idx)
            round_idx += 1

        # for r in range(rounds):
        #     if round_type == "internal":
        #         state = self.rp_round_polynomial(state, r)
        #     else:
        #         state = self.rf_round_polynomial(state, r)

        # 构造 CICO 方程
        const_vec = vector(R, [R(c) for c in constants])
        equations = list(state - const_vec)

        return equations, R, X

    def simple_form(self, target, rounds):
        state = target
        for round_idx in range(rounds):
            state = self.inv_internal_round(state, round_idx)
        return state
    
    def retain_dim_shear_nosubspace(self, constants, l):######在S盒之前=t引入参数################l是线性约束的个数，l最小是2个才用到W_list，对应W_list[0]
        k = self.t - l ############这是因为有容量，所以要多-1
        num_t = 20######加一个高次参数
        # R, X, t= self.make_symbolic_X_with_t(k)
        R, X, T = self.make_symbolic_X_with_t(k, num_t)

        if l >= 2:
            W = self.compute_mds_invariant_subspaces_sage()[l-2]
            W_R = matrix(R, W.nrows(), W.ncols(), [R(a) for a in W.list()])

            subspace_coords = vector(R, [
                T[1]*X[0] + T[2],
                T[3]*X[1] + T[4]
                # T[5]*X[2] + T[6]
            ])

            tail = W_R * subspace_coords
            state = vector(R, [T[0]] + list(tail))
        elif l == 1:
            # state = vector(R, [T[19]*X[0]+T[0], 
            #                    T[20]*X[1]+T[1],
            #                    T[21]*X[2]+T[2],
            #                    T[22]*X[3]+T[3],
            #                    R.zero()])
            state = vector(R, [T[0], 
                                T[1]*X[0]+T[2],
                                T[3]*X[1]+T[4],
                                T[5]*X[2]+T[6],
                                T[7]*X[3]+T[8]
                                ])
        elif l == 0:
            state = vector(R, [
                                T[1]*X[0]+T[2],
                                T[3]*X[1]+T[4],
                                T[5]*X[2]+T[6],
                                T[7]*X[3]+T[8],
                                R.zero()
                                ])
            
        round_idx = 0

        # 轮函数迭代#############注意这里的轮常数序列是错的
        eqs = []
        reduced_ideal = []

        state = self.rp_round_polynomial(state, round_idx)
        round_idx += 1
        f = R(state[0])
        eq = f - T[9]
        # eqs.append(eq)
        reduced_ideal.append(eq)
        state = vector(R, [T[9]] + list(state[1:]))

        # state = self.rp_round_polynomial(state, round_idx)
        # round_idx += 1
        # state = self.rp_round_polynomial(state, round_idx)
        # round_idx += 1
        # f = R(state[0])
        # eq = f - (T[9])#######
        # # eqs.append(eq)
        # reduced_ideal.append(eq)
        # state = vector(R, [T[9]] + list(state[1:]))############引入t5前面几位就不能随便代了，但其实也不影响

        state = self.rp_round_polynomial(state, round_idx)
        round_idx += 1
        
        state = self.rp_round_polynomial(state, round_idx)
        round_idx += 1
        f = R(state[0])
        eq = f - (T[10]*X[0]+T[11]*X[1]+T[12]*X[2]+T[13]*X[3]+T[14])
        # eqs.append(eq)
        reduced_ideal.append(eq)
        state = vector(R, [T[10]*X[0]+T[11]*X[1]+T[12]*X[2]+T[13]*X[3]+T[14]] + list(state[1:]))

        state = self.rp_round_polynomial(state, round_idx)
        round_idx += 1
        f = R(state[0])
        eq = f - (T[15]*X[0]+T[16]*X[1]+T[17]*X[2]+T[18]*X[3]+T[19])
        eqs.append(eq)
        # reduced_ideal.append(eq)
        state = vector(R, [T[15]*X[0]+T[16]*X[1]+T[17]*X[2]+T[18]*X[3]+T[19]] + list(state[1:]))

        # state = self.rp_round_polynomial(state, round_idx)
        # round_idx += 1
        # f = R(state[0])
        # eq = f - (T[20]*X[0]+T[21]*X[1]+T[22]*X[2]+T[23]*X[3]+T[24])
        # eqs.append(eq)
        # # reduced_ideal.append(eq)
        # state = vector(R, [T[20]*X[0]+T[21]*X[1]+T[22]*X[2]+T[23]*X[3]+T[24]] + list(state[1:]))
        # state = self.rp_round_polynomial(state, round_idx)
        # round_idx += 1

        # 构造 CICO 方程
        # const_vec = vector(R, [R(c) for c in constants])
        # eqs += list(state - const_vec)
        # target_state = vector(R, [
        #     T[9]*X[0]+T[10]*X[1]+T[12],
        #     R(constants[0]),
        #     R(constants[1]),
        #     R(constants[2]),
        #     R(constants[3])
        # ])
        # eqs += list(state - target_state)

        return eqs, reduced_ideal, R, X, T
    
def square_submatrix(M):
    n = M.ncols()
    return M[:n, :]

poseidon = PoseidonSage(seed=0)
W_list = poseidon.compute_mds_invariant_subspaces_sage()
poseidon.show_invariant_subspace_dynamics(W_list)



================ W_1 ================
Dimension: 2
Basis (columns):
[         1          0]
[         0          1]
[2029583814 2023448220]
[         0          0]

Initial vectors after padding zeros:
[         0          0]
[         1          0]
[         0          1]
[2029583814 2023448220]
[         0          0]

After multiplying M^1:
[         0          0]
[ 499014658 1128441114]
[1057816401  294888412]
[1532082932 2052201101]
[ 917775114 1540369709]


/tmp/ipykernel_957548/52864149.py:22: DeprecationWarning: Seeding based on hashing is deprecated
since Python 3.9 and will be removed in a subsequent version. The only 
supported seed types are: None, int, float, str, bytes, and bytearray.
  random.seed(seed)


In [10]:
if

SyntaxError: invalid syntax (4110802630.py, line 1)

In [35]:
# W = W_list[1]
constants = [9, 12, 52, 91, 43]
# constants = [9, 123, 5411, 524, 9871]
# specific_p = 4294967311
specific_p = 2130706433 #######KoalaBear prime

eqs, reduced_ideal, R, X, T = poseidon.retain_dim_shear_nosubspace(constants, l=1)

# subs = {
#     T[0]:12, T[1]:922, T[2]:56, T[3]:430, 
#     # T[4]:2, T[5]:3, 
#     T[6]:4, T[7]:5, T[8]:6, 
#     T[9]:7, T[10]:8, #T[11]:9, 
#     T[12]:10, T[13]:11,
#     #T[14]:12, 
#     T[15]:13, T[16]:14, 
#     #T[17]:15, T[18]:16
# }

# eqs_sub = [f.substitute(subs) for f in eqs]
# red_sub = [f.substitute(subs) for f in reduced_ideal]

Ms, cols = build_macaulay_matrix_mixed_fast(
    eqs,
    reduced_ideal,
    n=4,#######################################
    mult_deg=[2,0],
    col_deg=3,
    f3_index=0,
    lift_f3=False,
    homogeneous=False
)

for row in Ms[0]:
    print(row)
# Mraw = Ms[0]
print(Ms[0].nrows())
print(Ms[0].ncols())

def find_used_T_in_matrix(M):
    S = M.base_ring()
    T_vars = list(S.gens())

    used = set()

    for entry in M.list():
        if entry == 0:
            continue
        mons = entry.monomials()
        for m in mons:
            for v in m.variables():
                if v in T_vars:
                    used.add(v)

    return sorted(used, key=lambda x: str(x))


used_T = find_used_T_in_matrix(Ms[0])
print("used T:", used_T)
print("indices:", [int(str(t)[1:]) for t in used_T])

# Mcrit, info_compress = compress_rows_keep_last_only(Ms[0])

# print("compressed size =", Mcrit.nrows(), Mcrit.ncols())
# print("compress info =", info_compress)
# info = t_conditions_from_last_row_span(Mcrit, verbose=True)
# print("pivot cols =", info["pivot_cols"])
specific_p=2130706433

(736667229*t0^3 - 876091635*t0^2 - 1008865445*t0 + 970160856*t2 + 850438748*t4 - 300600916*t6 - 377033780*t8 - t9 - 153296228, 970160856*t1, 850438748*t3, -300600916*t5, -377033780*t7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 736667229*t0^3 - 876091635*t0^2 - 1008865445*t0 + 970160856*t2 + 850438748*t4 - 300600916*t6 - 377033780*t8 - t9 - 153296228, 0, 0, 0, 970160856*t1, 850438748*t3, -300600916*t5, -377033780*t7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 736667229*t0^3 - 876091635*t0^2 - 1008865445*t0 + 970160856*t2 + 850438748*t4 - 300600916*t6 - 377033780*t8 - t9 - 153296228, 0, 0, 0, 970160856*t1, 0, 0, 850438748*t3, -300600916*t5, -377033780*t7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 736667229*t0^3 - 876091635*t0^2 - 1008865445*t0 + 970160856*t2 + 850438748*t4 - 300600916*t6 - 377033780*t8 - t9 - 153296228, 0, 0, 0, 970160856*t1, 0, 0, 850438748*

In [17]:
print("reduced")
for i in range(len(reduced_ideal)):
    print(reduced_ideal[i])
print("eqs")
for i in range(len(eqs)):
    print(eqs[i])

reduced
736667229*t0^3 - 876091635*t0^2 + 970160856*x0*t1 + 850438748*x1*t3 - 300600916*x2*t5 - 377033780*x3*t7 - 1008865445*t0 + 970160856*t2 + 850438748*t4 - 300600916*t6 - 377033780*t8 - t9 - 153296228
-377041086*t0^9 + 301188073*t0^6*t9^3 + 987763601*t0^3*t9^6 + 72304251*t9^9 + 437989411*t0^8 - 57972927*x0*t0^6*t1 + 1046393119*x1*t0^6*t3 + 450923869*x2*t0^6*t5 + 989276753*x3*t0^6*t7 + 788417231*t0^6*t9^2 + 726304691*t0^5*t9^3 + 769158930*x0*t0^3*t1*t9^3 - 379027104*x1*t0^3*t3*t9^3 - 326662217*x2*t0^3*t5*t9^3 + 746278128*x3*t0^3*t7*t9^3 + 217045853*t0^3*t9^5 - 390816013*t0^2*t9^6 - 667871244*x0*t1*t9^6 - 18002779*x1*t3*t9^6 - 1040227843*x2*t5*t9^6 + 1035955405*x3*t7*t9^6 + 850355072*t9^8 + 505358981*t0^7 + 484723915*x0*t0^5*t1 + 753738220*x0^2*t0^3*t1^2 - 57972927*t0^6*t2 + 291914231*x1*t0^5*t3 - 811950805*x0*x1*t0^3*t1*t3 - 168149992*x1^2*t0^3*t3^2 + 1046393119*t0^6*t4 + 471829799*x2*t0^5*t5 - 210356304*x0*x2*t0^3*t1*t5 - 840487957*x1*x2*t0^3*t3*t5 + 272992454*x2^2*t0^3*t5^2 + 4509

In [18]:
import random

def random_assign_t0_t7(T, p, require_nonzero=(1,3,5,7), seed=None):
    """
    随机生成 t0..t7 的取值（在 GF(p) 中）
    
    T: 变量列表 [t0, t1, ..., t15]
    p: 有限域大小
    require_nonzero: 哪些 index 必须非零（默认 t1,t3,t5）
    seed: 可选随机种子
    """
    if seed is not None:
        random.seed(seed)
    
    F = GF(p)
    subs = {}
    
    for i in range(10):#####################################这里每次要改，第一批固定的T
        if i in require_nonzero:
            # 保证非零
            val = F.random_element()
            while val == 0:
                val = F.random_element()
        else:
            val = F.random_element()
        
        subs[T[i]] = val
    
    return subs
def substitute_matrix(M, subs):
    """
    把 subs 代入矩阵 M
    """
    return M.apply_map(lambda x: x.subs(subs))
def get_last_16_cols(M):
    ncols = M.ncols()
    return M.matrix_from_columns(range(ncols - 30, ncols))#################这里要改的数字是tail的列数

p = specific_p
# p = 101

subs_t0_t7 = random_assign_t0_t7(T, p)

print("===== t0..t7 =====")
for i in range(10):#######################################第一批固定的T
    print(f"t{i} =", subs_t0_t7[T[i]])

M0_fixed = substitute_matrix(Ms[0], subs_t0_t7)

M0_tail = get_last_16_cols(M0_fixed)
print("===== M0_tail =====")
print(M0_tail)

===== t0..t7 =====
t0 = 482429642
t1 = 1264188353
t2 = 172669890
t3 = 1608911998
t4 = 909765259
t5 = 991176468
t6 = 1293612732
t7 = 779174805
t8 = 1483728965
t9 = 1070554810
===== M0_tail =====
[                                                   0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0           

In [20]:
F = GF(specific_p)
# F = GF(101)

A_raw = M0_tail[:16, :].transpose()########################这里的数字是prefix行数
b = vector(M0_tail[16])######################这是最后一行，也就是prefix_l

A = Matrix(F, A_raw.nrows(), A_raw.ncols(), [F(x) for x in A_raw.list()])

K = A.transpose().right_kernel()

print("rank(A) =", A.rank())
print("dim ker(A^T) =", K.dimension())

basis = K.basis()
eqs = [sum(F(v[i]) * b[i] for i in range(len(b))) for v in basis]

print("number of equations =", len(eqs))
for i, eq in enumerate(eqs, 1):
    print(f"eq{i} =", eq)



# =========================================
# 输入: eqs  (你已经得到的那 6 个方程)
# 作用:
#   1) 投到真正的 4 变量环里
#   2) 输出“真实维数”
#   3) 若正维，随机切片找一个有限域点
# =========================================

def analyze_and_find_point(eqs, p=specific_p, max_trials=200, seed=None, verbose=True):
    import random
    rnd = random.Random(seed)
    # if seed is not None:
    #     random.seed(seed)

    F = GF(p)

    # 只保留真正出现的变量；你这里应当就是 t8,t9,t10,t11
    used_vars = []
    for f in eqs:
        for v in f.variables():
            if v not in used_vars:
                used_vars.append(v)

    used_vars = sorted(used_vars, key=str)
    if verbose:
        print("used vars =", used_vars)

    # 重建到“小环”里
    names = [str(v) for v in used_vars]
    Rsmall = PolynomialRing(F, names=names, order='degrevlex')
    gens = Rsmall.gens()
    subst_to_small = {used_vars[i]: gens[i] for i in range(len(used_vars))}

    # eqs_small = [Rsmall(f.subs(subst_to_small)) for f in eqs]
    used_vars = sorted(set().union(*[set(f.variables()) for f in eqs]), key=str)
    Rsmall = PolynomialRing(GF(p), [str(v) for v in used_vars], order='degrevlex')
    small_gens = Rsmall.gens()

    name_map = {str(v): g for v, g in zip(used_vars, small_gens)}

    eqs_small = []
    for f in eqs:
        d = f.dict()   # 指数 -> 系数
        g = Rsmall.zero()
        parent_gens = f.parent().gens()
        parent_names = [str(v) for v in parent_gens]

        for exp, coeff in d.items():
            term = Rsmall(coeff)
            for i, e in enumerate(exp):
                if e != 0:
                    vname = parent_names[i]
                    if vname in name_map:
                        term *= name_map[vname] ** e
                    else:
                        # 这里其实不该发生，因为你已经确认只剩 t8,t9,t10,t11
                        if e != 0:
                            raise ValueError(f"unexpected variable {vname} appears in f")
            g += term

        eqs_small.append(g)

    I = ideal(eqs_small)

    # 真正维数
    d = I.dimension()
    print("true dimension =", d)

    # 先看是否无解
    G = I.groebner_basis()
    if any(g == 1 for g in G):
        print("No solution: 1 in I")
        return {
            "ring": Rsmall,
            "eqs": eqs_small,
            "ideal": I,
            "dimension": d,
            "point": None,
        }

    # 0维时直接枚举
    if d == 0:
        try:
            sols = I.variety()
            if sols:
                print("found point =", sols[0])
                return {
                    "ring": Rsmall,
                    "eqs": eqs_small,
                    "ideal": I,
                    "dimension": d,
                    "point": sols[0],
                }
            else:
                print("No F_p-rational point found by variety()")
                return {
                    "ring": Rsmall,
                    "eqs": eqs_small,
                    "ideal": I,
                    "dimension": d,
                    "point": None,
                }
        except Exception as e:
            print("variety() failed on zero-dimensional ideal:", e)
            return {
                "ring": Rsmall,
                "eqs": eqs_small,
                "ideal": I,
                "dimension": d,
                "point": None,
            }

    # 正维时：随机切片
    # 用随机线性方程把维数 d 的簇切到 0 维，再尝试 variety()
    vars_small = list(gens)

    if verbose:
        print("Positive-dimensional ideal; trying random slices...")

    for trial in range(1, max_trials + 1):
        # 随机固定 d 个线性条件，最简单就是随机赋值给前 d 个变量
        # 也可以随机挑变量，这里更稳一点：随机挑 d 个变量固定
        chosen = random.sample(vars_small, d)
        vals = {v: F(rnd.randrange(p)) for v in chosen}
        # vals = {v: F.random_element() for v in chosen}
        slice_eqs = eqs_small + [v - vals[v] for v in chosen]
        if verbose:
            print("chosen =", chosen)
            print("vals =", vals)
        Islice = ideal(slice_eqs)

        try:
            ds = Islice.dimension()
        except Exception:
            ds = None

        if verbose and trial % 10 == 1:
            print(f"[trial {trial}] slice vars = {chosen}, dim =", ds)

        try:
            Gs = Islice.groebner_basis()
            if any(g == 1 for g in Gs):
                continue
        except Exception:
            continue

        try:
            sols = Islice.variety()
            if sols:
                pt = sols[0]
                print("found sliced point =", pt)

                # 补全没被固定但恰好也被解出的变量
                return {
                    "ring": Rsmall,
                    "eqs": eqs_small,
                    "ideal": I,
                    "dimension": d,
                    "point": pt,
                    "slice_vars": chosen,
                    "slice_values": vals,
                }
        except Exception:
            continue

    print("No point found within max_trials random slices.")
    return {
        "ring": Rsmall,
        "eqs": eqs_small,
        "ideal": I,
        "dimension": d,
        "point": None,
    }

res_mid = analyze_and_find_point(eqs, p=specific_p, max_trials=300, seed=None, verbose=True)

sol_8_11 = res_mid["point"]

print("\n===== t8..t11 =====")
if sol_8_11 is not None:
    for k, v in sol_8_11.items():
        print(f"{k} =", v)
else:
    print("No solution found")


rank(A) = 15
dim ker(A^T) = 15
number of equations = 15
eq1 = -861599117*t12^3 + 42664962*t10^2*t13 - 775594049*t12^2*t13 + 206062289*t10*t13^2 + 62728195*t12*t13^2 - 681046426*t13^3 + 79295254*t10^2*t14 + 986099714*t10*t13*t14 - 1000107070*t13^2*t14 - 1055586276*t10^2 + 560264373*t10*t13 + 81861869*t13^2
eq2 = 1022498459*t12^3 + 85329924*t10*t11*t13 - 114826638*t12^2*t13 - 701803248*t10*t13^2 + 206062289*t11*t13^2 - 43276491*t12*t13^2 + 47951861*t13^3 + 158590508*t10*t11*t14 + 362654304*t10*t13*t14 + 986099714*t11*t13*t14 + 1035196189*t13^2*t14 + 19533881*t10*t11 - 574463854*t10*t13 + 560264373*t11*t13 - 591893733*t13^2
eq3 = 751149336*t12^3 + 85329924*t10*t12*t13 + 938339755*t12^2*t13 - 1064416933*t10*t13^2 - 227121115*t12*t13^2 + 828366982*t13^3 + 158590508*t10*t12*t14 - 517381167*t10*t13*t14 + 986099714*t12*t13*t14 + 678338748*t13^2*t14 + 19533881*t10*t12 - 720709979*t10*t13 + 560264373*t12*t13 - 440975269*t13^2
eq4 = 1044583366*t12^3 + 42664962*t11^2*t13 - 521593955*t12^2*t13 - 70

In [21]:
def solve_remaining_linear_from_M0(M0, fixed_subs, remaining_T, prefix_l, force_c_index=None):
    """
    M0          : 原始 Ms[0]
    fixed_subs  : 已经固定好的参数字典，例如 t0..t11
    remaining_T : 剩余4个一次变量的列表，例如 [T[12], T[13], T[14], T[15]]

    返回:
      {
        "success": bool,
        "subs_remaining": {var: value, ...},
        "c": vector or None,
        "full_subs": 完整代入字典,
        "M_eval": 完整代入后的矩阵
      }
    """

    # 先代入已经固定好的 t0..t11
    M_part = M0.apply_map(lambda x: x.subs(fixed_subs))

    F = M_part.base_ring().base_ring() if hasattr(M_part.base_ring(), "base_ring") else None
    # 更稳：直接从 fixed_subs 里取一个值的 parent
    if fixed_subs:
        F = next(iter(fixed_subs.values())).parent()

    nrows = M_part.nrows()
    ncols = M_part.ncols()
    # assert nrows == 12, "这里默认是 12x20 的矩阵"
    # assert len(remaining_T) == 4, "这里默认还剩4个线性变量"
    # prefix_l = 1########################################这里是prefix的长度

    prefix = M_part.matrix_from_rows(range(prefix_l))   # 11 x 20
    last = vector(M_part.row(prefix_l))                 # 20-dim row vector

    # 未知量顺序:
    #   u = (c0,...,c10, r0,r1,r2,r3)
    # 其中 rj 对应 remaining_T[j]
    m = prefix_l + len(remaining_T)

    # 线性系统 L * u = rhs
    # 一共有 20 个列条件
    L = Matrix(F, ncols, m)
    rhs = vector(F, ncols)

    for col in range(ncols):
        # c0..c10 的系数：就是 prefix 的该列
        for i in range(prefix_l):
            L[col, i] = prefix[i, col]

        # last[col] 写成:
        #   const + a0*r0 + ... + a3*r3
        # 则方程 sum_i c_i*prefix[i,col] = const + sum_j a_j*r_j
        # 改写成
        #   sum_i c_i*... - sum_j a_j*r_j = const
        expr = last[col]

        # 常数项：把 remaining_T 都置 0
        const_term = expr.subs({v: F(0) for v in remaining_T})
        print("col =", col)
        print("expr =", expr)
        print("expr vars =", expr.variables())
        print("remaining_T =", remaining_T)

        const_term = expr.subs({v: F(0) for v in remaining_T})
        print("const_term =", const_term)
        print("const_term vars =", const_term.variables())
        rhs[col] = F(const_term)

        # 取每个 remaining var 的一次系数
        # 因为你说它们只在线性出现
        for j, v in enumerate(remaining_T):
            coeff_v = expr.coefficient(v)
            L[col, prefix_l + j] = -F(coeff_v)

    if force_c_index is not None:
        extra_row = [F(0)] * m
        extra_row[force_c_index] = F(1)
        L = L.stack(Matrix(F, 1, m, extra_row))
        rhs = vector(F, list(rhs) + [F(1)])

    # 解线性系统
    # 可能是恰有解，也可能有多个解；我们只要一个
    try:
        sol = L.solve_right(rhs)
    except Exception:
        # 如果不是唯一解，改用增广矩阵看是否相容，再取一个特解

        Aug = L.augment(matrix(F, L.nrows(), 1, list(rhs)))
        if L.rank() != Aug.rank():
            return {
                "success": False,
                "subs_remaining": None,
                "c": None,
                "full_subs": None,
                "M_eval": None,
            }
        # 有解但不唯一：取一个特解
        # Sage 某些版本里 solve_right 对欠定系统不直接给解，这里手工取
        # 用右核参数化，先做 echelon_form
        sol = L.solve_right(rhs)  # 如果这里仍失败，再换更手工的方法

    c = vector(F, sol[:prefix_l])
    rem_vals = sol[prefix_l:]

    subs_remaining = {remaining_T[j]: F(rem_vals[j]) for j in range(len(remaining_T))}

    full_subs = dict(fixed_subs)
    full_subs.update(subs_remaining)

    M_eval = M0.apply_map(lambda x: x.subs(full_subs))

    return {
        "success": True,
        "subs_remaining": subs_remaining,
        "c": c,
        "full_subs": full_subs,
        "M_eval": M_eval,
    }

def convert_sol_8_11_to_original_T(sol_8_11, T):
    subs_8_11 = {}
    for k, v in sol_8_11.items():
        name = str(k)
        if name.startswith('t'):
            idx = Integer(name[1:])
            if 10 <= idx <= 14:#########################################这里每次需要修改
                subs_8_11[T[idx]] = F(v)
    return subs_8_11

def point_to_T_subs_by_name(pt, T, F=None, only_existing=True):
    """
    把 analyze_and_find_point / variety() 返回的点 pt
    变成当前这套 T 上的替换字典 {T[i]: value}。

    pt 可以是:
      - {'t6': ..., 't7': ...}
      - {t6: ..., t7: ...}
    """
    if pt is None:
        return {}

    name_to_T = {str(v): v for v in T}
    subs = {}

    for k, v in pt.items():
        kname = str(k)
        if kname in name_to_T:
            subs[name_to_T[kname]] = F(v) if F is not None else v
        elif not only_existing:
            raise KeyError(f"{kname} 不在当前 T 里")

    return subs


def merge_round_subs(T, *parts):
    """
    按顺序合并多份替换字典。
    后面的会覆盖前面的同名变量。
    """
    name_to_T = {str(v): v for v in T}
    out = {}

    for part in parts:
        if not part:
            continue
        for k, v in part.items():
            kname = str(k)
            if kname in name_to_T:
                out[name_to_T[kname]] = v

    return out

def point_to_T_subs_by_name(pt, T, F=None, only_existing=True):
    """
    把 analyze_and_find_point / variety() 返回的点 pt
    变成当前这套 T 上的替换字典 {T[i]: value}。

    pt 可以是:
      - {'t6': ..., 't7': ...}
      - {t6: ..., t7: ...}
    """
    if pt is None:
        return {}

    name_to_T = {str(v): v for v in T}
    subs = {}

    for k, v in pt.items():
        kname = str(k)
        if kname in name_to_T:
            subs[name_to_T[kname]] = F(v) if F is not None else v
        elif not only_existing:
            raise KeyError(f"{kname} 不在当前 T 里")

    return subs


def merge_round_subs(T, *parts):
    """
    按顺序合并多份替换字典。
    后面的会覆盖前面的同名变量。
    """
    name_to_T = {str(v): v for v in T}
    out = {}

    for part in parts:
        if not part:
            continue
        for k, v in part.items():
            kname = str(k)
            if kname in name_to_T:
                out[name_to_T[kname]] = v

    return out


In [28]:

prefix_l = 30##############################################这里每次修改
F = T[0].parent().base_ring()

res_mid = analyze_and_find_point(eqs, p=specific_p, max_trials=300, seed=None, verbose=True)##################这里有个p需要改
sol_8_11 = res_mid["point"]

subs_8_11 = convert_sol_8_11_to_original_T(sol_8_11, T)

subs_fixed = {}
subs_fixed.update(subs_t0_t7)
subs_fixed.update(subs_8_11)

print("===== subs_fixed so far =====")
for i in range(15):############################################这里是第一批和第二批T的数量总和
    if T[i] in subs_fixed:
        print(f"t{i} =", subs_fixed[T[i]])

res_last = solve_remaining_linear_from_M0(
    Ms[0],
    fixed_subs=subs_fixed,
    remaining_T=[T[15], T[16], T[17], T[18], T[19]],#########################################剩下要解的参数
    prefix_l = 16,################################这里每次也要改
    force_c_index=None
)

print("\n===== t12..t15 =====")
if res_last["success"]:
    subs_12_15 = res_last["subs_remaining"]
    for i in range(15, 20):###############################################同上
        print(f"t{i} =", subs_12_15[T[i]])
else:
    print("Failed to solve t12..t15")

print("c =", res_last["c"])
print("target c entry =", res_last["c"][15])######################################
# print("rank(L) =", res_last["L"].rank())
# print("num unknowns =", res_last["L"].ncols())

used vars = [t10, t11, t12, t13, t14]
true dimension = 2
Positive-dimensional ideal; trying random slices...
chosen = [t11, t14]
vals = {t11: 586002162, t14: 1166762331}
[trial 1] slice vars = [t11, t14], dim = 0
found sliced point = {t14: 1166762331, t13: 795339404, t12: 1039982828, t11: 586002162, t10: 410783974}
===== subs_fixed so far =====
t0 = 482429642
t1 = 1264188353
t2 = 172669890
t3 = 1608911998
t4 = 909765259
t5 = 991176468
t6 = 1293612732
t7 = 779174805
t8 = 1483728965
t9 = 1070554810
t10 = 410783974
t11 = 586002162
t12 = 1039982828
t13 = 795339404
t14 = 1166762331
col = 0
expr = -t19 - 890854597
expr vars = (t19,)
remaining_T = [t15, t16, t17, t18, t19]
const_term = -890854597
const_term vars = ()
col = 1
expr = -t15 + 955663625
expr vars = (t15,)
remaining_T = [t15, t16, t17, t18, t19]
const_term = 955663625
const_term vars = ()
col = 2
expr = -t16 + 787368345
expr vars = (t16,)
remaining_T = [t15, t16, t17, t18, t19]
const_term = 787368345
const_term vars = ()
col = 3
ex

In [29]:
if res_last["success"]:
    subs_fixed.update(subs_12_15)

    print("\n===== FULL t =====")
    for i in range(20):#@###############################################这里每次也要改，是全部的T
        print(f"t{i} =", subs_fixed[T[i]])

M_test = Ms[0].apply_map(lambda x: x.subs(subs_fixed))

prefix = M_test[:16, :]################################这里改prefix

print("\n===== RANK CHECK =====")
print("rank(prefix) =", prefix.rank())
print("rank(full)   =", M_test.rank())

# 更严格检查
last = vector(M_test[1])
recon = sum(res_last["c"][i] * vector(prefix[i]) for i in range(16))#####################这里每次改成prefix
print("span check =", last == recon)


===== FULL t =====
t0 = 482429642
t1 = 1264188353
t2 = 172669890
t3 = 1608911998
t4 = 909765259
t5 = 991176468
t6 = 1293612732
t7 = 779174805
t8 = 1483728965
t9 = 1070554810
t10 = 410783974
t11 = 586002162
t12 = 1039982828
t13 = 795339404
t14 = 1166762331
t15 = 1206996979
t16 = 1131330311
t17 = 1951226827
t18 = 1371824262
t19 = 0

===== RANK CHECK =====
rank(prefix) = 16
rank(full)   = 16
span check = False


In [30]:
# ===== rebuild original equations only for ideal check =====

constants = [9, 12, 52, 91, 43]
orig_eqs_check, orig_reduced_check, R_check, X_check, T_check = poseidon.retain_dim_shear_nosubspace(constants, l=1)############

print(orig_reduced_check)
print(orig_eqs_check)
# 把当前 subs_fixed 里的 t0,t1,... 映射到这套新生成的 T_check 上
subs_T_for_check = {}
for i in range(len(T_check)):
    old_name = f"t{i}"
    for k, v in subs_fixed.items():
        if str(k) == old_name:
            subs_T_for_check[T_check[i]] = v
            break

print("===== T substituted for check =====")
for i in range(len(T_check)):
    if T_check[i] in subs_T_for_check:
        print(f"t{i} =", subs_T_for_check[T_check[i]])

g1 = orig_reduced_check[0].subs(subs_T_for_check)
g2 = orig_reduced_check[1].subs(subs_T_for_check)
# g3 = orig_reduced_check[2].subs(subs_T_for_check)
g4 = orig_eqs_check[0].subs(subs_T_for_check)

print("\n===== after substituting T =====")
print("g1 =", g1)
print("g2 =", g2)
# print("g3 =", g3)
print("g4 =", g4)
print("vars(g1) =", g1.variables())
print("vars(g2) =", g2.variables())
# print("vars(g3) =", g3.variables())
print("vars(g3) =", g4.variables())

# ===== dimension check =====
F = GF(specific_p)
Rx = PolynomialRing(F, ['x0', 'x1', 'x2', 'x3'], order='degrevlex')#####################这里每次也要改
x0, x1, x2, x3 = Rx.gens()############################每次要改

def to_Rx(expr):
    P = expr.parent()
    images = []
    for v in P.gens():
        nm = str(v)
        if nm == 'x0':
            images.append(x0)
        elif nm == 'x1':
            images.append(x1)
        elif nm == 'x2':
            images.append(x2)################################################每次要改
        elif nm == 'x3':
            images.append(x3)##########################################每次要改
        else:
            images.append(Rx.zero())
    return P.hom(images, Rx)(expr)

f1 = to_Rx(g1)
f2 = to_Rx(g2)
# f3 = to_Rx(g3)
f4 = to_Rx(g4)

I1 = Rx.ideal([f1, f2])
I12 = Rx.ideal([f1, f2, f4])

print("\n===== dimension check =====")
print("f1 =", f1)
print("f2 =", f2)
# print("f3 =", f3)
print("f4 =", f4)
print("dim(<f1>) =", I1.dimension())
print("dim(<f1,f2>) =", I12.dimension())
# print("f2 == 0 ?", f2 == 0)
################################这里如果出现理想不符合的情况，那大概是因为前面的线性组合系数在,f2上取了0，所以其实f3没有被<f1,f2,f3>纳入

[736667229*t0^3 - 876091635*t0^2 + 970160856*x0*t1 + 850438748*x1*t3 - 300600916*x2*t5 - 377033780*x3*t7 - 1008865445*t0 + 970160856*t2 + 850438748*t4 - 300600916*t6 - 377033780*t8 - t9 - 153296228, -377041086*t0^9 + 301188073*t0^6*t9^3 + 987763601*t0^3*t9^6 + 72304251*t9^9 + 437989411*t0^8 - 57972927*x0*t0^6*t1 + 1046393119*x1*t0^6*t3 + 450923869*x2*t0^6*t5 + 989276753*x3*t0^6*t7 + 788417231*t0^6*t9^2 + 726304691*t0^5*t9^3 + 769158930*x0*t0^3*t1*t9^3 - 379027104*x1*t0^3*t3*t9^3 - 326662217*x2*t0^3*t5*t9^3 + 746278128*x3*t0^3*t7*t9^3 + 217045853*t0^3*t9^5 - 390816013*t0^2*t9^6 - 667871244*x0*t1*t9^6 - 18002779*x1*t3*t9^6 - 1040227843*x2*t5*t9^6 + 1035955405*x3*t7*t9^6 + 850355072*t9^8 + 505358981*t0^7 + 484723915*x0*t0^5*t1 + 753738220*x0^2*t0^3*t1^2 - 57972927*t0^6*t2 + 291914231*x1*t0^5*t3 - 811950805*x0*x1*t0^3*t1*t3 - 168149992*x1^2*t0^3*t3^2 + 1046393119*t0^6*t4 + 471829799*x2*t0^5*t5 - 210356304*x0*x2*t0^3*t1*t5 - 840487957*x1*x2*t0^3*t3*t5 + 272992454*x2^2*t0^3*t5^2 + 450923869*

In [31]:
import random

def build_extended_ring_from_matrix_and_subs(M, subs):
    """
    根据矩阵 M 的原变量 + subs 右侧表达式里出现的新参数，构造扩展环:
        GF(p)[原变量..., 新参数...]
    返回:
        R0        : M 原来的基环
        Rext      : 扩展后的多项式环
        to_ext    : 原环 R0 -> Rext 的名字同态
        name_to_ext_gen : {变量名: Rext中的生成元}
    """
    R0 = M.base_ring()
    F = R0.base_ring()

    orig_names = [str(v) for v in R0.gens()]

    extra_names = []
    for val in subs.values():
        # 只从“表达式本身实际出现的变量”提取
        if hasattr(val, "variables"):
            for g in val.variables():
                nm = str(g)
                if nm not in orig_names and nm not in extra_names:
                    extra_names.append(nm)

    for val in subs.values():
        if hasattr(val, "parent") and hasattr(val.parent(), "gens"):
            P = val.parent()
            print("val =", val)
            print("parent =", P)
            print("gens =", [str(g) for g in P.gens()])
            print("-----")

    all_names = orig_names + extra_names
    Rext = PolynomialRing(F, all_names, order='degrevlex')
    ext_gens = Rext.gens()
    name_to_ext_gen = {all_names[i]: ext_gens[i] for i in range(len(all_names))}

    # 原环 -> 扩展环
    to_ext = R0.hom([name_to_ext_gen[str(v)] for v in R0.gens()], Rext)

    return R0, Rext, to_ext, name_to_ext_gen


def substitute_matrix_safe_with_params(M, subs):
    """
    安全代入：
      - subs 的键可以来自别的 ring，但按名字匹配
      - subs 的值可以含新参数 c0,c1,...
    返回:
      M_sub   : 代入后的矩阵（在扩展环 Rext 上）
      Rext    : 扩展环
      info    : 调试信息
    """
    R0, Rext, to_ext, name_to_ext_gen = build_extended_ring_from_matrix_and_subs(M, subs)

    # 先把矩阵提升到扩展环
    M_ext = M.apply_map(lambda x: to_ext(x))

    # 再把替换字典转成 Rext 上的字典
    subs_ext = {}
    for k, v in subs.items():
        kname = str(k)
        if kname not in name_to_ext_gen:
            continue
        kk = name_to_ext_gen[kname]

        # 把 v 也搬到 Rext
        # 把 v 搬到 Rext
        # 纯域元素 / 常数：直接 coercion
        if not hasattr(v, "variables") or len(v.variables()) == 0:
            vv = Rext(v)
        else:
            Pv = v.parent()
            v_vars = [str(g) for g in v.variables()]
            images = []
            dom_gens = list(Pv.gens())

            for g in dom_gens:
                nm = str(g)
                if nm in name_to_ext_gen:
                    images.append(name_to_ext_gen[nm])
                else:
                    # 对不在表达式里实际出现的生成元，随便给个 0 即可
                    images.append(Rext.zero())

            phi_v = Pv.hom(images, Rext)
            vv = phi_v(v)

        subs_ext[kk] = vv

    M_sub = M_ext.apply_map(lambda x: x.subs(subs_ext))

    info = {
        "R0": R0,
        "Rext": Rext,
        "subs_ext": subs_ext,
        "name_to_ext_gen": name_to_ext_gen,
    }
    return M_sub, Rext, info

import random

def random_assign_t_prefix(T, p, upto, require_nonzero=(), seed=None):
    """
    给 t0..t_{upto-1} 随机赋值
    """
    rnd = random.Random(seed)
    F = GF(p)
    subs = {}

    for i in range(upto):
        if i in require_nonzero:
            val = F(rnd.randrange(p))
            while val == 0:
                val = F(rnd.randrange(p))
        else:
            val = F(rnd.randrange(p))
        subs[T[i]] = val

    return subs


def get_tail_cols(M, tail_cols):
    ncols = M.ncols()
    return M.matrix_from_columns(range(ncols - tail_cols, ncols))

In [ ]:
def solve_remaining_linear_from_M0_parametric(M0, fixed_subs, remaining_T, prefix_l, c_prefix="c"):
    """
    参数化求解：
      给定 fixed_subs 后，解
         sum_i c_i * prefix_row_i = last_row
      中出现的 remaining_T。
    如果解不唯一，则引入新参数 c0,c1,... 表示自由度。

    返回:
      {
        "success": bool,
        "free_dim": int,
        "param_ring": C,                  # F[c0,...,c_{d-1}]
        "subs_remaining": {T[j]: expr},   # 含参数表达式
        "c_expr": vector(...),            # 前缀 span 系数的含参数表达式
        "full_subs": dict,                # fixed_subs + subs_remaining
        "L": L,
        "rhs": rhs,
      }
    """

    # 先代入已固定参数
    R0 = M0.base_ring()

    # 按名字把 fixed_subs 的键换成 R0 里的生成元
    name_to_gen = {str(v): v for v in R0.gens()}

    fixed_subs_R0 = {}
    for k, v in fixed_subs.items():
        kname = str(k)
        if kname in name_to_gen:
            fixed_subs_R0[name_to_gen[kname]] = R0.base_ring()(v)

    M_part = M0.apply_map(lambda x: x.subs(fixed_subs_R0))

    # 基域
    if fixed_subs:
        F = next(iter(fixed_subs.values())).parent()
    else:
        # 退路：从矩阵基环里取
        F = M_part.base_ring().base_ring()

    ncols = M_part.ncols()

    prefix = M_part.matrix_from_rows(range(prefix_l))
    last = vector(M_part.row(prefix_l))

    # 未知量 u = (c_0,...,c_{prefix_l-1}, r_0,...,r_{k-1})
    m = prefix_l + len(remaining_T)

    L = Matrix(F, ncols, m)
    rhs = vector(F, ncols)

    for col in range(ncols):
        # prefix 系数
        for i in range(prefix_l):
            L[col, i] = prefix[i, col]

        expr = last[col]

        # 常数项：remaining_T 全置 0
        const_term = expr.subs({v: F(0) for v in remaining_T})
        if len(const_term.variables()) != 0:
            raise ValueError(
                f"Column {col}: after zeroing remaining_T, still nonconstant: {const_term}, vars={const_term.variables()}"
            )
        rhs[col] = F(const_term)

        # remaining_T 的一次系数
        for j, v in enumerate(remaining_T):
            coeff_v = expr.coefficient(v)
            L[col, prefix_l + j] = -F(coeff_v)
        
        linear_part = const_term + sum(expr.coefficient(v) * v for v in remaining_T)
        residual = expr - linear_part
        if residual != 0:
            raise ValueError(
                f"Column {col}: expression is not linear in remaining_T: residual = {residual}"
    )

    # 先检查相容性
    Aug = L.augment(matrix(F, ncols, 1, list(rhs)))
    if L.rank() != Aug.rank():
        return {
            "success": False,
            "free_dim": None,
            "param_ring": None,
            "subs_remaining": None,
            "c_expr": None,
            "full_subs": None,
            "L": L,
            "rhs": rhs,
        }

    # 一个特解
    u_part = L.solve_right(rhs)

    # 右核：通解 = u_part + sum_j lambda_j * ker_basis[j]
    K = L.right_kernel()
    basisK = K.basis()
    free_dim = len(basisK)

    # 参数环 F[c0,...,c_{d-1}]
    if free_dim == 0:
        C = F
        params = []
    else:
        c_names = [f"{c_prefix}{j}" for j in range(free_dim)]
        C = PolynomialRing(F, c_names, order='degrevlex')
        params = list(C.gens())

    def lift_to_C(x):
        return C(F(x))

    # 构造含参通解
    u_expr = [lift_to_C(u_part[i]) for i in range(m)]
    for j, kb in enumerate(basisK):
        for i in range(m):
            u_expr[i] += lift_to_C(kb[i]) * params[j]

    # 前 prefix_l 个是 span 系数
    # 前 prefix_l 个是 span 系数
    if prefix_l == 1:
        c_expr = u_expr[0]
    else:
        c_expr = list(u_expr[:prefix_l])

    # 后面是 remaining_T 的表达式
    rem_expr = u_expr[prefix_l:]
    subs_remaining = {remaining_T[j]: rem_expr[j] for j in range(len(remaining_T))}

    # 完整替换字典
    full_subs = {}
    for k, v in fixed_subs.items():
        full_subs[k] = lift_to_C(v)
    full_subs.update(subs_remaining)

    return {
        "success": True,
        "free_dim": free_dim,
        "param_ring": C,
        "subs_remaining": subs_remaining,
        "c_expr": c_expr,
        "full_subs": full_subs,
        "L": L,
        "rhs": rhs,
    }

In [ ]:
prefix_l = 31###################要改
front_cols = 5###############################要改
F = T[0].parent().base_ring()

# 直接使用前面已经有的结果，不再重新求 t6,t7,t8
subs_fixed = {}
subs_fixed.update(subs_t0_t7)
subs_fixed.update(subs_8_11)

print("===== subs_fixed so far =====")
for i in range(15):########################################要改
    if T[i] in subs_fixed:
        print(f"t{i} =", subs_fixed[T[i]])

remaining_T = [T[15], T[16], T[17], T[18], T[19]]

res_last = solve_remaining_linear_from_M0_parametric(
    Ms[0],
    fixed_subs=subs_fixed,
    remaining_T=remaining_T,
    prefix_l=prefix_l,
    c_prefix="c"
)

print("\n===== last batch T =====")
if res_last["success"]:
    subs_last = res_last["subs_remaining"]

    subs_all = {}
    subs_all.update(subs_fixed)
    subs_all.update(subs_last)

    print("free_dim (full matrix) =", res_last["free_dim"])
    print("param ring =", res_last["param_ring"])

    print("\n===== ALL T =====")
    for i in range(len(T)):
        if T[i] in subs_all:
            print(f"t{i} =", subs_all[T[i]])

    print("\nspan coeff c =", res_last["c_expr"])

    M_param, Rext, info_ext = substitute_matrix_safe_with_params(Ms[0], subs_all)

    M_front = M_param.matrix_from_columns(list(range(front_cols)))
    print("\n===== front block =====")
    print(M_front)

    target = vector(M_front.row(prefix_l))

    c_expr = res_last["c_expr"]
    if prefix_l == 1:
        c_list = [c_expr] if not isinstance(c_expr, (list, tuple)) else list(c_expr)
    else:
        c_list = list(c_expr)

    recon = sum(c_list[i] * vector(M_front.row(i)) for i in range(prefix_l))
    diff = target - recon
    eq_front = [diff[j] for j in range(len(diff)) if diff[j] != 0]

    print("\n===== front-column equations =====")
    print("number of equations =", len(eq_front))
    for i, e in enumerate(eq_front, 1):
        print(f"eq{i} =", e)

    P = res_last["param_ring"]
    if len(eq_front) == 0:
        print("\nfront free dimension =", len(P.gens()))
    else:
        I_front = P.ideal(eq_front)
        print("\nfront free dimension =", I_front.dimension())

else:
    print("Failed to solve last batch T")

===== subs_fixed so far =====
t0 = 58
t1 = 73
t2 = 61
t3 = 6
t4 = 74
t5 = 32
t6 = 58
t7 = 57
t8 = 18
t9 = 7
t10 = 88
t11 = 100
t12 = 45
t13 = 62
t14 = 14

===== last batch T =====
free_dim (full matrix) = 7
param ring = Multivariate Polynomial Ring in c0, c1, c2, c3, c4, c5, c6 over Finite Field of size 101

===== ALL T =====
t0 = 58
t1 = 73
t2 = 61
t3 = 6
t4 = 74
t5 = 32
t6 = 58
t7 = 57
t8 = 18
t9 = 7
t10 = 88
t11 = 100
t12 = 45
t13 = 62
t14 = 14
t15 = 28*c0 + 47*c2 + 32*c5 - 17
t16 = -3*c2 + 13*c5 + 34*c6 - 7
t17 = 27*c2 - 16*c5 - 3*c6 + 7
t18 = -21*c2 - 10*c5 + 36*c6
t19 = -3*c0 - 21*c6

span coeff c = [c0 - 46, c1 - 38, c2 - 50, c3 + 28, c4 - 31, -38, 8*c1 - 1, 29*c1 + 45, -45*c1 - 39, c5 + 31, -29*c2 + 8*c3 - 18*c5 + 38, 45*c2 + 8*c4 + 14*c5 - 6, -42*c2 + 29*c3 - 20*c5 - 47, -2*c2 - 45*c3 + 29*c4 - 25*c5 - 33, 12*c2 - 45*c4 + 49*c5 - 16, c6 + 29, -29*c1 + 17*c2 - 40*c5, 9*c5 + 27, 42*c2 - 29*c3 + 20*c5 + 3, c2 - 29*c4 - 38*c5 + 4, 35*c1, 17*c5, 12*c2 + 35*c3 + 49*c5, -43*c2 + 35*c

In [ ]:
res_last = solve_remaining_linear_from_M0_parametric(
    Ms[0],
    fixed_subs=subs_fixed,
    remaining_T=[T[15], T[16], T[17], T[18], T[19]],##########################每次要改
    prefix_l=31,#####################################################每次要改
    c_prefix="c"
)

print("\n===== remaining T (parametric) =====")
if res_last["success"]:
    print("free_dim =", res_last["free_dim"])
    print("param ring =", res_last["param_ring"])
    for v in [T[15], T[16], T[17], T[18], T[19]]:################################同上
        print(f"{v} =", res_last["subs_remaining"][v])
    print("span coeff c =", res_last["c_expr"])
else:
    print("No solution")


===== remaining T (parametric) =====
free_dim = 7
param ring = Multivariate Polynomial Ring in c0, c1, c2, c3, c4, c5, c6 over Finite Field of size 101
t15 = 28*c0 + 47*c2 + 32*c5 - 17
t16 = -3*c2 + 13*c5 + 34*c6 - 7
t17 = 27*c2 - 16*c5 - 3*c6 + 7
t18 = -21*c2 - 10*c5 + 36*c6
t19 = -3*c0 - 21*c6
span coeff c = [c0 - 46, c1 - 38, c2 - 50, c3 + 28, c4 - 31, -38, 8*c1 - 1, 29*c1 + 45, -45*c1 - 39, c5 + 31, -29*c2 + 8*c3 - 18*c5 + 38, 45*c2 + 8*c4 + 14*c5 - 6, -42*c2 + 29*c3 - 20*c5 - 47, -2*c2 - 45*c3 + 29*c4 - 25*c5 - 33, 12*c2 - 45*c4 + 49*c5 - 16, c6 + 29, -29*c1 + 17*c2 - 40*c5, 9*c5 + 27, 42*c2 - 29*c3 + 20*c5 + 3, c2 - 29*c4 - 38*c5 + 4, 35*c1, 17*c5, 12*c2 + 35*c3 + 49*c5, -43*c2 + 35*c4 + 18*c5, 40, -35, -4, -1, 48, 48, -26]


In [ ]:
constants = [9, 12, 52, 91, 43]
# constants = [9, 123, 5411, 524, 9871]
# specific_p = 4294967311
specific_p = 101

eqs, reduced_ideal, R, X, T = poseidon.retain_dim_shear_nosubspace(constants, l=0)#########################


Ms, cols = build_macaulay_matrix_mixed_fast(
    eqs,
    [reduced_ideal[0], reduced_ideal[1], eqs[0]],###########################第二个输入是原先的reduced——ideal也就是prefix
    n=4,##########################################
    mult_deg=[2, 2, 0],
    col_deg=3,
    f3_index=1,
    lift_f3=False,
    homogeneous=False
)

for row in Ms[0]:
    print(row)
# Mraw = Ms[0]
print(Ms[0].nrows())
print(Ms[0].ncols())

def find_used_T_in_matrix(M):
    S = M.base_ring()
    T_vars = list(S.gens())

    used = set()

    for entry in M.list():
        if entry == 0:
            continue
        mons = entry.monomials()
        for m in mons:
            for v in m.variables():
                if v in T_vars:
                    used.add(v)

    return sorted(used, key=lambda x: str(x))


used_T = find_used_T_in_matrix(Ms[0])
print("used T:", used_T)
print("indices:", [int(str(t)[1:]) for t in used_T])

# Mcrit, info_compress = compress_rows_keep_last_only(Ms[0])

# print("compressed size =", Mcrit.nrows(), Mcrit.ncols())
# print("compress info =", info_compress)
# info = t_conditions_from_last_row_span(Mcrit, verbose=True)
# print("pivot cols =", info["pivot_cols"])

(-t0 + t2, t1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, -t0 + t2, 0, 0, 0, t1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, -t0 + t2, 0, 0, 0, t1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, -t0 + t2, 0, 0, 0, t1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, -t0 + t2, 0, 0, 0, t1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, -t0 + t2, 0, 0, 0, 0, 0, 0, 0, 0, 0, t1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0, -t0 + t2, 0, 0, 0, 0, 0, 0, 0, 0, 0, t1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0, 0, -t0 + t2, 0, 0, 0, 0, 0, 0, 0, 0, 0, t1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0, 0, 0, -t0 + t2, 0, 0, 0, 0, 0, 0, 0, 0, 0, t1, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
# ===== tail compatibility via ker(prefix_tail) against the last row =====

prefix_l = 31##################################################
tail_cols = 30#################################################

try:
    subs_for_tail = subs_all
except NameError:
    subs_for_tail = res_last["full_subs"]

M_sub, Rext, info_ext = substitute_matrix_safe_with_params(Ms[0], subs_for_tail)

ncols = M_sub.ncols()
tail_idx = list(range(ncols - tail_cols, ncols))
M_tail = M_sub.matrix_from_columns(tail_idx)

print("===== tail block =====")
for row in M_tail:
    print(row)
print("shape =", M_tail.nrows(), "x", M_tail.ncols())

S = M_tail.base_ring()
all_gens = list(S.gens())
c_vars = [v for v in all_gens if str(v).startswith("c")]
print("c vars =", c_vars)

prefix_tail = M_tail.matrix_from_rows(range(prefix_l))
last_row = vector(M_tail.row(prefix_l))

print("\n===== prefix / last =====")
print("prefix_tail shape =", prefix_tail.nrows(), "x", prefix_tail.ncols())
print("rank(prefix_tail) =", prefix_tail.rank())

# --------------------------------------------------
# no c: directly rank check
# --------------------------------------------------
if len(c_vars) == 0:
    full_test = M_tail.matrix_from_rows(list(range(prefix_l)) + [prefix_l])
    print("\n===== no c: direct rank check =====")
    print("rank(prefix_tail) =", prefix_tail.rank())
    print("rank(prefix+last) =", full_test.rank())
    print("spanned ?         =", prefix_tail.rank() == full_test.rank())

# --------------------------------------------------
# with c: use ker(prefix_tail) to get compatibility equations in c
# --------------------------------------------------
else:
    print("\n===== with c: kernel compatibility equations =====")

    # 只保留 c-环
    Pc = PolynomialRing(GF(specific_p), [str(v) for v in c_vars], order='degrevlex')
    Pc_gens = Pc.gens()
    name_to_pc = {str(v): Pc_gens[i] for i, v in enumerate(c_vars)}

    def to_Pc(expr):
        Pold = expr.parent()
        images = []
        for v in Pold.gens():
            nm = str(v)
            if nm in name_to_pc:
                images.append(name_to_pc[nm])
            else:
                images.append(Pc.zero())
        return Pold.hom(images, Pc)(expr)

    prefix_pc = prefix_tail.apply_map(to_Pc)
    last_pc = vector(Pc, [to_Pc(x) for x in last_row])

    print("prefix_pc base ring =", prefix_pc.base_ring())

    # right kernel of prefix_tail
    Kmat = prefix_pc.right_kernel_matrix()
    print("kernel basis matrix shape =", Kmat.nrows(), "x", Kmat.ncols())
    print("=== kernel basis vectors ===")
    for i in range(Kmat.nrows()):
        kv = vector(Pc, list(Kmat.row(i)))
        print(f"kv[{i}] =", kv)
        print("prefix_pc * kv^T =", prefix_pc * kv.column())
    # eqs_tail = []
    # for i in range(Kmat.nrows()):
    #     kv = vector(Pc, Kmat.row(i))
    #     e = sum(kv[j] * last_pc[j] for j in range(len(kv)))
    #     if e != 0:
    #         eqs_tail.append(e)
    eqs_tail = []
    print("=== compatibility equations from kernel vectors ===")
    for i in range(Kmat.nrows()):
        kv = vector(Pc, list(Kmat.row(i)))
        e = sum(kv[j] * last_pc[j] for j in range(Kmat.ncols()))
        print(f"kernel vec {i} gives:", e)
        if e != 0:
            eqs_tail.append(e)

    # 去重
    eqs_tail_unique = []
    seen = set()
    for e in eqs_tail:
        s = str(e)
        if s not in seen:
            seen.add(s)
            eqs_tail_unique.append(e)
    eqs_tail = eqs_tail_unique

    print("number of compatibility equations =", len(eqs_tail))
    for i, e in enumerate(eqs_tail, 1):
        print(f"eq{i} =", e)

    print("\n===== GB / quotient-ring dimension check =====")
    I_tail = Pc.ideal(eqs_tail)
    G_tail = I_tail.groebner_basis()
    G_nz = [g for g in G_tail if g != 0]

    if any(g == 1 for g in G_nz):
        print("dim = -1   # empty variety / unit ideal")
    elif len(G_nz) == 0:
        print("dim =", Pc.ngens())
    elif len(G_nz) == 1:
        print("dim =", Pc.ngens() if G_nz[0] == 0 else Pc.ngens() - 1)
    else:
        try:
            print("dim =", I_tail.dimension())
        except AttributeError:
            print("dim = [Sage不给，至少不是零理想，也不是单主理想]")

    print("GB size =", len(G_tail))
    for g in G_tail:
        print(g)

    print("1 in ideal ?", any(g == 1 for g in G_nz))

    print("\n===== ideal / quotient dimension =====")
    print("I_tail =", I_tail)
    try:
        print("dim(I_tail) =", I_tail.dimension())
    except Exception as e:
        print("I_tail.dimension() failed:", e)

    Q = Pc.quotient(I_tail)
    print("Q =", Q)
    try:
        print("dim(Pc / I_tail) =", Q.dimension())
    except Exception as e:
        print("Q.dimension() failed:", e)

    print("GB(I_tail) =", G_tail)
    print("1 in ideal ?", any(g == 1 for g in G_tail))

    print("=== check substituted last row in tail ===")
    last_row = vector(M_tail.row(prefix_l))
    for j, e in enumerate(last_row):
        print(j, e, e.variables())

    print("=== check substituted remaining T ===")
    for v in [T[15], T[16], T[17], T[18], T[19]]:
        print(v, "=", subs_all[v], "vars =", subs_all[v].variables() if hasattr(subs_all[v], "variables") else ())

    print("\n===== last row over Pc =====")
print("last_pc =", last_pc)
for j in range(len(last_pc)):
    print(f"last_pc[{j}] =", last_pc[j])
####################################################################
print("\n===== kernel vectors and dot products =====")
for i in range(Kmat.nrows()):
    kv = vector(Pc, list(Kmat.row(i)))
    print(f"\nkv[{i}] =", kv)

    terms = []
    for j in range(Kmat.ncols()):
        if kv[j] != 0:
            terms.append((j, kv[j], last_pc[j], kv[j] * last_pc[j]))

    print("nonzero terms:")
    for j, a, b, prod in terms:
        print(f"  col {j}: ({a}) * ({b}) = {prod}")

    e = sum(kv[j] * last_pc[j] for j in range(Kmat.ncols()))
    print("dot product =", e)
print("variables in dot product =", e.variables() if hasattr(e, "variables") else ())

val = 58
parent = Finite Field of size 101
gens = ['1']
-----
val = 73
parent = Finite Field of size 101
gens = ['1']
-----
val = 61
parent = Finite Field of size 101
gens = ['1']
-----
val = 6
parent = Finite Field of size 101
gens = ['1']
-----
val = 74
parent = Finite Field of size 101
gens = ['1']
-----
val = 32
parent = Finite Field of size 101
gens = ['1']
-----
val = 58
parent = Finite Field of size 101
gens = ['1']
-----
val = 57
parent = Finite Field of size 101
gens = ['1']
-----
val = 18
parent = Finite Field of size 101
gens = ['1']
-----
val = 7
parent = Finite Field of size 101
gens = ['1']
-----
val = 14
parent = Finite Field of size 101
gens = ['1']
-----
val = 62
parent = Finite Field of size 101
gens = ['1']
-----
val = 45
parent = Finite Field of size 101
gens = ['1']
-----
val = 100
parent = Finite Field of size 101
gens = ['1']
-----
val = 88
parent = Finite Field of size 101
gens = ['1']
-----
val = 28*c0 + 47*c2 + 32*c5 - 17
parent = Multivariate Polynomial Ring 

In [32]:
# ============================================================
# Export current Poseidon parameters / constants / solved t's
# to a LaTeX snippet file for appendix use
# ============================================================

from pathlib import Path

# ---------- config ----------
output_tex = "appendix_parameters_pKoalaBear_t5.tex"   # output file name
make_standalone = False                           # True => standalone LaTeX doc
matrix_env = "bmatrix"                            # "bmatrix" or "pmatrix"
rc_per_row = None                                # None => one round constant vector per row
# ----------------------------

def to_int_str(x):
    """Convert Sage / field elements / integers to plain integer strings."""
    try:
        return str(Integer(x))
    except Exception:
        try:
            return str(int(x))
        except Exception:
            return str(x)

def latex_escape_text(s):
    repl = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\textasciicircum{}',
    }
    out = []
    for ch in str(s):
        out.append(repl.get(ch, ch))
    return ''.join(out)

def latex_vector(vec):
    entries = [to_int_str(v) for v in vec]
    body = r" \\ ".join(entries)
    return r"\begin{%s}%s\end{%s}" % (matrix_env, body, matrix_env)

def latex_row_vector(vec):
    entries = [to_int_str(v) for v in vec]
    body = " & ".join(entries)
    return r"\begin{%s}%s\end{%s}" % (matrix_env, body, matrix_env)

def latex_matrix(M):
    rows = []
    for i in range(M.nrows()):
        row = " & ".join(to_int_str(M[i, j]) for j in range(M.ncols()))
        rows.append(row)
    body = r" \\ ".join(rows)
    return r"\begin{%s}%s\end{%s}" % (matrix_env, body, matrix_env)

def get_alpha_from_poseidon(poseidon_obj):
    for attr in ["alpha", "sbox_exp"]:
        if hasattr(poseidon_obj, attr):
            return getattr(poseidon_obj, attr)
    raise AttributeError("Could not find alpha on poseidon object.")

def get_p_from_poseidon(poseidon_obj):
    if hasattr(poseidon_obj, "p"):
        return getattr(poseidon_obj, "p")
    if hasattr(poseidon_obj, "F"):
        try:
            return poseidon_obj.F.order()
        except Exception:
            pass
    raise AttributeError("Could not find p on poseidon object.")

def get_t_from_poseidon(poseidon_obj):
    if hasattr(poseidon_obj, "t"):
        return getattr(poseidon_obj, "t")
    raise AttributeError("Could not find t on poseidon object.")

def get_round_constants_from_poseidon(poseidon_obj):
    for attr in ["round_constants", "RC"]:
        if hasattr(poseidon_obj, attr):
            return getattr(poseidon_obj, attr)
    raise AttributeError("Could not find round constants on poseidon object.")

def get_mds_from_poseidon(poseidon_obj):
    for attr in ["mds_matrix", "MDS", "mds"]:
        if hasattr(poseidon_obj, attr):
            return getattr(poseidon_obj, attr)
    raise AttributeError("Could not find MDS matrix on poseidon object.")

# ---------- collect objects from current notebook state ----------
alpha = get_alpha_from_poseidon(poseidon)
p_val = get_p_from_poseidon(poseidon)
t_val = get_t_from_poseidon(poseidon)
round_constants = get_round_constants_from_poseidon(poseidon)
mds_matrix = get_mds_from_poseidon(poseidon)

# solved t0..t15:
# expects T and subs_fixed already exist in the notebook
solved_t = []
for i in range(len(T)):
    key = T[i]
    if key in subs_fixed:
        solved_t.append((f"t_{i}", subs_fixed[key]))
    else:
        solved_t.append((f"t_{i}", None))

# ---------- build latex ----------
parts = []

if make_standalone:
    parts.append(r"\documentclass[11pt]{article}")
    parts.append(r"\usepackage{amsmath,amssymb,booktabs,longtable}")
    parts.append(r"\usepackage[margin=1in]{geometry}")
    parts.append(r"\begin{document}")

parts.append(r"\section*{Poseidon Parameters and Concrete Instance}")
parts.append("")
parts.append(r"\paragraph{Global parameters.}")
parts.append(r"\begin{itemize}")
parts.append(r"  \item Field characteristic: $p = %s$." % to_int_str(p_val))
parts.append(r"  \item State width: $t = %s$." % to_int_str(t_val))
parts.append(r"  \item S-box exponent: $\alpha = %s$." % to_int_str(alpha))
parts.append(r"\end{itemize}")
parts.append("")

parts.append(r"\paragraph{Solved parameters $t_0,\dots,t_{15}$.}")
parts.append(r"\begin{align*}")
for idx, (name, val) in enumerate(solved_t):
    if val is None:
        line = rf"{name} &= \text{{not assigned}}"
    else:
        line = rf"{name} &= {to_int_str(val)}"
    if idx != len(solved_t) - 1:
        line += r", \\"
    parts.append(line)
parts.append(r"\end{align*}")
parts.append("")

parts.append(r"\paragraph{Cauchy MDS matrix.}")
parts.append(r"\[")
parts.append(r"M = %s" % latex_matrix(mds_matrix))
parts.append(r"\]")
parts.append("")

parts.append(r"\paragraph{Round constants.}")
parts.append(r"\begin{longtable}{@{}cl@{}}")
parts.append(r"\toprule")
parts.append(r"Round & Constant vector \\")
parts.append(r"\midrule")
parts.append(r"\endhead")

for r_idx, rc in enumerate(round_constants):
    rc_tex = latex_row_vector(rc)
    parts.append(rf"{r_idx} & ${rc_tex}$ \\")

parts.append(r"\bottomrule")
parts.append(r"\end{longtable}")
parts.append("")

if make_standalone:
    parts.append(r"\end{document}")

tex_content = "\n".join(parts)

# ---------- write file ----------
out_path = Path(output_tex)
out_path.write_text(tex_content, encoding="utf-8")

print(f"Wrote LaTeX to: {out_path.resolve()}")
print("\nPreview (first 40 lines):")
for line in tex_content.splitlines()[:40]:
    print(line)

Wrote LaTeX to: /mnt/e/Poseidon_Analysis/appendix_parameters_pKoalaBear_t5.tex

Preview (first 40 lines):
\section*{Poseidon Parameters and Concrete Instance}

\paragraph{Global parameters.}
\begin{itemize}
  \item Field characteristic: $p = 2130706433$.
  \item State width: $t = 5$.
  \item S-box exponent: $\alpha = 3$.
\end{itemize}

\paragraph{Solved parameters $t_0,\dots,t_{15}$.}
\begin{align*}
t_0 &= 482429642, \\
t_1 &= 1264188353, \\
t_2 &= 172669890, \\
t_3 &= 1608911998, \\
t_4 &= 909765259, \\
t_5 &= 991176468, \\
t_6 &= 1293612732, \\
t_7 &= 779174805, \\
t_8 &= 1483728965, \\
t_9 &= 1070554810, \\
t_10 &= 410783974, \\
t_11 &= 586002162, \\
t_12 &= 1039982828, \\
t_13 &= 795339404, \\
t_14 &= 1166762331, \\
t_15 &= 1206996979, \\
t_16 &= 1131330311, \\
t_17 &= 1951226827, \\
t_18 &= 1371824262, \\
t_19 &= 0
\end{align*}

\paragraph{Cauchy MDS matrix.}
\[
M = \begin{bmatrix}736667229 & 970160856 & 850438748 & 1830105517 & 1753672653 \\ 1342567709 & 149224043 & 1090950053 & 